# Residual Sibling Gating | Public v90

This is an alternative public branch for comparison with our v49 (0.730). It reproduces [Ozertuu v90](https://www.kaggle.com/code/ozertuu/0-730-residual-sibling-gating-pb115-hybrid?scriptVersionId=374798467). The notebook is listed among the public 0.730+ solutions. The title and score history are attached to the public notebook, and this copy needs its own scoring run.

The source keeps the original GLM mass queue and uses paired PB115 features with joint typography filtering and the GLM-out guard. The comparison evaluates the complete branch against ours. Its 47 inputs and GPU image are taken from the v90 run. The upstream inference and credits follow.

In [ ]:
'''Check all source mounts before a visible preview can write its submission.'''
from pathlib import Path
import json

class KaggleInputs:
    SOURCES = {'competition_sources': ['name-that-llm'], 'dataset_sources': ['jamesliebel/glm53-iq3xxs-segs-a-mirror', 'jamesliebel/glm53-iq3xxs-segs-b-mirror', 'jamesliebel/glm53-flash-iq3xxs-segs-mirror', 'dhruvtibarewal/nlm-l24-adapters', 'lightningv08/ntl-olmo2-native24-final-inference-assets', 'crtproduction/name-that-llm-open-pool', 'seyitkaangunes/ntl-extra-api-passages', 'abdullah1afify/ntl-generations', 'dophuchung/ntl-proxy-generations', 'zbigniewkirchner/ntl-qwen3-8b-lora-claude-head', 'dhruvtibarewal/nlm-matched-clean', 'dhruvtibarewal/nlm-offline-wheels', 'josephadamski91/ntl-atlas-qwen3-8b-v1', 'lucasboesen/ntl-free-proxy-generations', 'nuridudak/ntl-q14-lora', 'nuridudak/ntl-q14-lora-v4', 'lumibee/ntl-prefixb-full-115', 'kirilltym/ntl-spectral-style-heads-paired-4404', 'nuridudak/ntl-svc-prefit-pool-v4n', 'dhruvtibarewal/nlm-enc-v5', 'nuridudak/ntl-two-stage-encoders', 'nishantdahal/ntl-two-stage-lora', 'nishantdahal/ntl-handwriting-blend', 'dhruvtibarewal/nlm-l24-adapters-v5', 'prvsiyan/prvsiyans-extended-name-that-llm-dataset', 'bbucxi/qwen35-9b-safetensors', 'pjmathematician/ultrabert-380m-ntl'], 'kernel_sources': ['lucasboesen/name-that-llm-glm4-9b-trunc25', 'lucasboesen/name-that-llm-extract-encoders2', 'lucasboesen/name-that-llm-electra-large-weights', 'lucasboesen/name-that-llm-desklib-weights', 'lucasboesen/name-that-llm-train-lora24-s7', 'lucasboesen/name-that-llm-train-tokattn', 'lucasboesen/name-that-llm-train-tokattn2', 'vnton37nguyn/ntl-train-lora-qwen25-instyle', 'estopadilla/ntl-glm-likelihood-train', 'estopadilla/ntl-glm-prefix-ab'], 'model_sources': ['alexchilton/llama-3-1-8b-base/Transformers/default/1', 'qwen-lm/qwen-3/Transformers/8b-base/1', 'qwen-lm/qwen-3/Transformers/14b-base/1', 'qwen-lm/qwen-3/Transformers/14b/1', 'qwen-lm/qwen-3/Transformers/4b-base/1', 'qwen-lm/qwen-3/Transformers/8b/1', 'qwen-lm/qwen-3-embedding/Transformers/8b/1', 'qwen-lm/qwen2.5/Transformers/14b/1', 'qwen-lm/qwen2.5/Transformers/7b/1']}
    PB115_FILES = ('spec.json', 'flash_features.csv', 'glm53_features.csv', 'paired_token_features.parquet')

    @classmethod
    def check(cls, root=Path('/kaggle/input')):
        missing = []
        for key, layout in (('dataset_sources', 'datasets'), ('kernel_sources', 'notebooks')):
            for ref in cls.SOURCES[key]:
                slug = ref.split('/')[-1]
                if not ((root / layout / ref).is_dir() or (root / slug).is_dir()):
                    missing.append(ref)
        for ref in cls.SOURCES['model_sources']:
            owner, name, framework, variant, version = ref.split('/')
            tail = Path(name, framework.lower(), variant, version)
            if not ((root / 'models' / owner / tail).is_dir() or (root / tail).is_dir()):
                missing.append(ref)
        for name in cls.SOURCES['competition_sources']:
            if not any(all((path / f).is_file() for f in ('train.csv', 'test.csv'))
                       for path in (root / 'competitions' / name, root / name)):
                missing.append(name)
        if missing:
            raise FileNotFoundError(f'Missing Kaggle sources: {missing}')
        pb115 = next(path for path in (root / 'datasets/lumibee/ntl-prefixb-full-115',
                                      root / 'ntl-prefixb-full-115') if path.is_dir())
        absent = [name for name in cls.PB115_FILES if not (pb115 / name).is_file()]
        if absent:
            raise FileNotFoundError(f'Incomplete PB115 source: {absent} in {pb115}')
        spec = json.loads((pb115 / 'spec.json').read_text())
        if (spec.get('format'), len(spec.get('features_54', [])), len(spec.get('features_61', []))) != ('ntl-prefixb-115-v1', 54, 61):
            raise ValueError(f'Unexpected PB115 feature specification in {pb115}')
        print(f'47 Kaggle sources mounted | PB115 files present at {pb115}', flush=True)
        return pb115

adapted_pb115_dir = KaggleInputs.check()


# 0.728 | Residual Sibling Gating + PB115 Hybrid Likelihood

Clean, **100% transparent**, offline Dual-T4 pipeline combining multi-backbone reader representations, unlocked Prefix-B 115-feature GLM-5.3 causal likelihoods, and mass-preserving intra-family sibling gating.

| Stage | Component | Key Configuration | Role |
| :--- | :--- | :--- | :--- |
| **Stage 1** | **Multi-Backbone Ensemble + GSQ-27B** | 5x L24 LoRA (`q14v5`, `q8v5`, `ll8v5`, `n14b_clean`, `q25_14`) + `DeBERTa` + 5-fold Native Q8 + `Qwen3.8-27B` (`gsq_share=0.035`, `timeout=5400s`) + Sinkhorn (`0.30`) | Calibrated 18-class prior log-probabilities |
| **Stage 2** | **PB-115 Hybrid GLM-5.3 Likelihood** | Dual-T4 `ntl-segrun` (`GLM-5.3` + `GLM-5.3-Flash`), `--max-cover 0.40`, `--hybrid-alpha 0.35`, `LR + 3-seed LightGBM` (18-class + direct 3-way GLM marginal odds) | Causal token-level perplexity disambiguation on uncertain passages |
| **Stage 3** | **Residual Sibling Gating & Invariants** | All-GLM-Out Guard (`fable_return`) + Stylometric `Residual` + `+0.55` Sibling Odds (`margin < 0.8`) + Joint Typographic Masking (`' — '`, curly quotes, `'—'`) | Preserves cross-family mass while refining close sibling pairs (`claude`, `gpt`, `gemini`) |

> **Execution Modes:** During interactive/public Save (`FAST_COMMIT = True`), end-to-end smoke checks validate Stage-1 prefit core votes, Stage-2 PB-115 `LR + LightGBM` hybrid calibration, Stage-3 residual sibling gating, and output schema in ~1 minute. During competition rerun (`FAST_COMMIT = False`), the full Dual-T4 inference pipeline executes automatically.

## 1. Deterministic Asset Resolution & Environment Setup
Explicit mount-alias resolution across all 44 attached Kaggle datasets, kernels, and models (`pinned_assets.py`), offline wheel installation, and clean training/open-pool alignment.

In [ ]:
%%writefile pinned_assets.py
"""Deterministic Asset Resolution and Mount-Alias Verification Engine.

Provides explicit, reproducible filesystem mappings across all 44 attached
Kaggle datasets, kernels, and pretrained model checkpoints, supporting both
legacy `/kaggle/input/<slug>` and namespaced mount hierarchies without silent
directory-discovery fallbacks."""
from pathlib import Path
import os
MOUNT_ALIASES = {'/kaggle/input/glm53-iq3xxs-segs-a-mirror': '/kaggle/input/datasets/jamesliebel/glm53-iq3xxs-segs-a-mirror', '/kaggle/input/glm53-iq3xxs-segs-b-mirror': '/kaggle/input/datasets/jamesliebel/glm53-iq3xxs-segs-b-mirror', '/kaggle/input/glm53-flash-iq3xxs-segs-mirror': '/kaggle/input/datasets/jamesliebel/glm53-flash-iq3xxs-segs-mirror', '/kaggle/input/nlm-l24-adapters': '/kaggle/input/datasets/dhruvtibarewal/nlm-l24-adapters', '/kaggle/input/ntl-olmo2-native24-final-inference-assets': '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets', '/kaggle/input/name-that-llm-open-pool': '/kaggle/input/datasets/crtproduction/name-that-llm-open-pool', '/kaggle/input/ntl-extra-api-passages': '/kaggle/input/datasets/seyitkaangunes/ntl-extra-api-passages', '/kaggle/input/ntl-generations': '/kaggle/input/datasets/abdullah1afify/ntl-generations', '/kaggle/input/ntl-proxy-generations': '/kaggle/input/datasets/dophuchung/ntl-proxy-generations', '/kaggle/input/ntl-qwen3-8b-lora-claude-head': '/kaggle/input/datasets/zbigniewkirchner/ntl-qwen3-8b-lora-claude-head', '/kaggle/input/nlm-matched-clean': '/kaggle/input/datasets/dhruvtibarewal/nlm-matched-clean', '/kaggle/input/nlm-offline-wheels': '/kaggle/input/datasets/dhruvtibarewal/nlm-offline-wheels', '/kaggle/input/ntl-atlas-qwen3-8b-v1': '/kaggle/input/datasets/josephadamski91/ntl-atlas-qwen3-8b-v1', '/kaggle/input/ntl-free-proxy-generations': '/kaggle/input/datasets/lucasboesen/ntl-free-proxy-generations', '/kaggle/input/ntl-q14-lora': '/kaggle/input/datasets/nuridudak/ntl-q14-lora', '/kaggle/input/ntl-q14-lora-v4': '/kaggle/input/datasets/nuridudak/ntl-q14-lora-v4', '/kaggle/input/ntl-spectral-style-heads-paired-4404': '/kaggle/input/datasets/kirilltym/ntl-spectral-style-heads-paired-4404', '/kaggle/input/ntl-svc-prefit-pool-v4n': '/kaggle/input/datasets/nuridudak/ntl-svc-prefit-pool-v4n', '/kaggle/input/nlm-enc-v5': '/kaggle/input/datasets/dhruvtibarewal/nlm-enc-v5', '/kaggle/input/ntl-two-stage-encoders': '/kaggle/input/datasets/nuridudak/ntl-two-stage-encoders', '/kaggle/input/ntl-two-stage-lora': '/kaggle/input/datasets/nishantdahal/ntl-two-stage-lora', '/kaggle/input/ntl-handwriting-blend': '/kaggle/input/datasets/nishantdahal/ntl-handwriting-blend', '/kaggle/input/nlm-l24-adapters-v5': '/kaggle/input/datasets/dhruvtibarewal/nlm-l24-adapters-v5', '/kaggle/input/prvsiyans-extended-name-that-llm-dataset': '/kaggle/input/datasets/prvsiyan/prvsiyans-extended-name-that-llm-dataset', '/kaggle/input/qwen35-9b-safetensors': '/kaggle/input/datasets/bbucxi/qwen35-9b-safetensors', '/kaggle/input/ultrabert-380m-ntl': '/kaggle/input/datasets/pjmathematician/ultrabert-380m-ntl', '/kaggle/input/name-that-llm-glm4-9b-trunc25': '/kaggle/input/notebooks/lucasboesen/name-that-llm-glm4-9b-trunc25', '/kaggle/input/name-that-llm-extract-encoders2': '/kaggle/input/notebooks/lucasboesen/name-that-llm-extract-encoders2', '/kaggle/input/name-that-llm-electra-large-weights': '/kaggle/input/notebooks/lucasboesen/name-that-llm-electra-large-weights', '/kaggle/input/name-that-llm-desklib-weights': '/kaggle/input/notebooks/lucasboesen/name-that-llm-desklib-weights', '/kaggle/input/name-that-llm-train-lora24-s7': '/kaggle/input/notebooks/lucasboesen/name-that-llm-train-lora24-s7', '/kaggle/input/name-that-llm-train-tokattn': '/kaggle/input/notebooks/lucasboesen/name-that-llm-train-tokattn', '/kaggle/input/name-that-llm-train-tokattn2': '/kaggle/input/notebooks/lucasboesen/name-that-llm-train-tokattn2', '/kaggle/input/ntl-train-lora-qwen25-instyle': '/kaggle/input/notebooks/vnton37nguyn/ntl-train-lora-qwen25-instyle', '/kaggle/input/ntl-glm-likelihood-train': '/kaggle/input/notebooks/estopadilla/ntl-glm-likelihood-train', '/kaggle/input/llama-3-1-8b-base': '/kaggle/input/models/alexchilton/llama-3-1-8b-base', '/kaggle/input/qwen-3': '/kaggle/input/models/qwen-lm/qwen-3', '/kaggle/input/qwen-3-embedding': '/kaggle/input/models/qwen-lm/qwen-3-embedding', '/kaggle/input/qwen2.5': '/kaggle/input/models/qwen-lm/qwen2.5'}
PATHS = {'/kaggle/input/**/': [], '/kaggle/input/**/bitsandbytes-*.whl': ['/kaggle/input/nlm-offline-wheels/bitsandbytes-0.50.2-py3-none-manylinux_2_24_x86_64.whl', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/bitsandbytes-0.50.2-py3-none-manylinux_2_24_x86_64.whl'], '/kaggle/input/**/config.json': ['/kaggle/input/models/alexchilton/llama-3-1-8b-base/transformers/default/1/config.json', '/kaggle/input/models/qwen-lm/qwen-3-embedding/transformers/8b/1/config.json', '/kaggle/input/models/qwen-lm/qwen-3/transformers/14b-base/1/config.json', '/kaggle/input/models/qwen-lm/qwen-3/transformers/14b/1/config.json', '/kaggle/input/models/qwen-lm/qwen-3/transformers/4b-base/1/config.json', '/kaggle/input/models/qwen-lm/qwen-3/transformers/8b-base/1/config.json', '/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1/config.json', '/kaggle/input/models/qwen-lm/qwen2.5/transformers/14b/1/config.json', '/kaggle/input/models/qwen-lm/qwen2.5/transformers/7b/1/config.json', '/kaggle/input/name-that-llm-desklib-weights/desklib-detector/config.json', '/kaggle/input/name-that-llm-electra-large-weights/electra-large-discriminator/config.json', '/kaggle/input/name-that-llm-extract-encoders2/deberta-v3-large/config.json', '/kaggle/input/name-that-llm-glm4-9b-trunc25/glm4_9b_trunc25/config.json', '/kaggle/input/nlm-enc-v5/encoder/config.json', '/kaggle/input/ntl-handwriting-blend/deberta_v3_large/encoder/config.json', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/document-deberta-07nov/document-deberta-07nov/final/config.json', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/document-modernbert-07nov/document-modernbert-07nov/final/config.json', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/gsq3-base/qwen38-27b-gsq3/config.json', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/llama31-prefix24/config.json', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/ntl-tabular-offline-assets/config.json', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-27b-nf4/config.json', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/ntl-lora24-fivefold-training/lora24/final/base_config/config.json', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/olmo2-7b-fp16-offline-weights/olmo2-fp16-base/config.json', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/smollm3-3b-base-offline-weights/smollm3-base/config.json', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/ntl-native-q8-multidepth-fivefold-0679/base/1_Pooling/config.json', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/ntl-native-q8-multidepth-fivefold-0679/base/config.json', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/ntl-native-wemm4-fivefold-0679/base/config.json', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/config.json', '/kaggle/input/ntl-two-stage-encoders/desklib/config.json', '/kaggle/input/ntl-two-stage-encoders/modernbert/config.json', '/kaggle/input/qwen35-9b-safetensors/config.json', '/kaggle/input/ultrabert-380m-ntl/fold0/config.json', '/kaggle/input/ultrabert-380m-ntl/fold1/config.json', '/kaggle/input/ultrabert-380m-ntl/fold2/config.json', '/kaggle/input/ultrabert-380m-ntl/fold3/config.json', '/kaggle/input/ultrabert-380m-ntl/fold4/config.json'], '/kaggle/input/**/deberta-v3-large/config.json': ['/kaggle/input/name-that-llm-extract-encoders2/deberta-v3-large/config.json'], '/kaggle/input/**/deberta_v3_large/meta.json': ['/kaggle/input/ntl-handwriting-blend/deberta_v3_large/meta.json'], '/kaggle/input/**/desklib-detector/config.json': ['/kaggle/input/name-that-llm-desklib-weights/desklib-detector/config.json'], '/kaggle/input/**/electra-large-discriminator/config.json': ['/kaggle/input/name-that-llm-electra-large-weights/electra-large-discriminator/config.json'], '/kaggle/input/**/gen_passages.csv': ['/kaggle/input/datasets/lucasboesen/ntl-free-proxy-generations/gen_passages.csv'], '/kaggle/input/**/ll8v5/heads.pt': ['/kaggle/input/nlm-l24-adapters-v5/ll8v5/heads.pt'], '/kaggle/input/**/ll8v5_f0/heads.pt': ['/kaggle/input/nlm-l24-adapters-v5/ll8v5_f0/heads.pt'], '/kaggle/input/**/ll8v5_f1/heads.pt': ['/kaggle/input/nlm-l24-adapters-v5/ll8v5_f1/heads.pt'], '/kaggle/input/**/ll8v5_f2/heads.pt': ['/kaggle/input/nlm-l24-adapters-v5/ll8v5_f2/heads.pt'], '/kaggle/input/**/ll8v5_f3/heads.pt': ['/kaggle/input/nlm-l24-adapters-v5/ll8v5_f3/heads.pt'], '/kaggle/input/**/ll8v5_f4/heads.pt': ['/kaggle/input/nlm-l24-adapters-v5/ll8v5_f4/heads.pt'], '/kaggle/input/**/lora24/final/COMPLETE.json': ['/kaggle/input/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/ntl-lora24-fivefold-training/lora24/final/COMPLETE.json'], '/kaggle/input/**/lora_qwen25_14b/meta.json': ['/kaggle/input/ntl-handwriting-blend/lora_qwen25_14b/meta.json'], '/kaggle/input/**/lora_qwen3_14b/meta.json': ['/kaggle/input/ntl-handwriting-blend/lora_qwen3_14b/meta.json'], '/kaggle/input/**/name-that-llm-train-lora24-s7/lora_out/job_final/COMPLETE.json': ['/kaggle/input/name-that-llm-train-lora24-s7/lora_out/job_final/COMPLETE.json'], '/kaggle/input/**/nlm_matched_clean.parquet': ['/kaggle/input/datasets/dhruvtibarewal/nlm-matched-clean/nlm_matched_clean.parquet'], '/kaggle/input/**/ntl-native-q8-multidepth-fivefold-0679/manifest.json': ['/kaggle/input/ntl-olmo2-native24-final-inference-assets/ntl-native-q8-multidepth-fivefold-0679/manifest.json'], '/kaggle/input/**/ntl-q14-lora-v4/heads.safetensors': ['/kaggle/input/ntl-q14-lora-v4/heads.safetensors'], '/kaggle/input/**/ntl-q14-lora/heads.safetensors': ['/kaggle/input/ntl-q14-lora/heads.safetensors'], '/kaggle/input/**/ntl-qwen3-8b-lora-claude-head/**/final/COMPLETE.json': ['/kaggle/input/ntl-qwen3-8b-lora-claude-head/final/COMPLETE.json'], '/kaggle/input/**/ntl-train-lora-qwen25-instyle/lora_out/job_final/COMPLETE.json': ['/kaggle/input/ntl-train-lora-qwen25-instyle/lora_out/job_final/COMPLETE.json'], '/kaggle/input/**/ntl666-cache-parts.json': ['/kaggle/input/ntl-olmo2-native24-final-inference-assets/ntl666-cache-parts.json'], '/kaggle/input/**/ntl_extra_passages_claude.csv': ['/kaggle/input/datasets/seyitkaangunes/ntl-extra-api-passages/ntl_extra_passages_claude.csv'], '/kaggle/input/**/ntl_generated*.csv': [], '/kaggle/input/**/ntl_generations.csv': ['/kaggle/input/datasets/abdullah1afify/ntl-generations/ntl_generations.csv'], '/kaggle/input/**/ntl_open_pool.parquet': ['/kaggle/input/name-that-llm-open-pool/ntl_open_pool.parquet'], '/kaggle/input/**/ntl_proxy_generations.csv': ['/kaggle/input/datasets/dophuchung/ntl-proxy-generations/ntl_proxy_generations.csv'], '/kaggle/input/**/passages_clean.csv': ['/kaggle/input/datasets/prvsiyan/prvsiyans-extended-name-that-llm-dataset/passages_clean.csv'], '/kaggle/input/**/q14v5/heads.pt': ['/kaggle/input/nlm-l24-adapters-v5/q14v5/heads.pt'], '/kaggle/input/**/q14v5_f0/heads.pt': ['/kaggle/input/nlm-l24-adapters-v5/q14v5_f0/heads.pt'], '/kaggle/input/**/q14v5_f1/heads.pt': ['/kaggle/input/nlm-l24-adapters-v5/q14v5_f1/heads.pt'], '/kaggle/input/**/q14v5_f2/heads.pt': ['/kaggle/input/nlm-l24-adapters-v5/q14v5_f2/heads.pt'], '/kaggle/input/**/q14v5_f3/heads.pt': ['/kaggle/input/nlm-l24-adapters-v5/q14v5_f3/heads.pt'], '/kaggle/input/**/q14v5_f4/heads.pt': ['/kaggle/input/nlm-l24-adapters-v5/q14v5_f4/heads.pt'], '/kaggle/input/**/q8v5/heads.pt': ['/kaggle/input/nlm-l24-adapters-v5/q8v5/heads.pt'], '/kaggle/input/**/q8v5_f0/heads.pt': ['/kaggle/input/nlm-l24-adapters-v5/q8v5_f0/heads.pt'], '/kaggle/input/**/q8v5_f1/heads.pt': ['/kaggle/input/nlm-l24-adapters-v5/q8v5_f1/heads.pt'], '/kaggle/input/**/q8v5_f2/heads.pt': ['/kaggle/input/nlm-l24-adapters-v5/q8v5_f2/heads.pt'], '/kaggle/input/**/q8v5_f3/heads.pt': ['/kaggle/input/nlm-l24-adapters-v5/q8v5_f3/heads.pt'], '/kaggle/input/**/q8v5_f4/heads.pt': ['/kaggle/input/nlm-l24-adapters-v5/q8v5_f4/heads.pt'], '/kaggle/input/**/tokattn_debl23_final.pt': ['/kaggle/input/name-that-llm-train-tokattn2/tokattn_debl23_final.pt'], '/kaggle/input/**/tokattn_final.pt': ['/kaggle/input/name-that-llm-train-tokattn/tokattn_final.pt'], '/kaggle/input/**/ultrabert-380m-ntl': ['/kaggle/input/ultrabert-380m-ntl'], '/kaggle/input/ultrabert-380m-ntl/fold*': ['/kaggle/input/ultrabert-380m-ntl/fold0', '/kaggle/input/ultrabert-380m-ntl/fold1', '/kaggle/input/ultrabert-380m-ntl/fold2', '/kaggle/input/ultrabert-380m-ntl/fold3', '/kaggle/input/ultrabert-380m-ntl/fold4']}
CHECKPOINTS = {('llama', None, None): '/kaggle/input/models/alexchilton/llama-3-1-8b-base/transformers/default/1', ('llama', 4096, None): '/kaggle/input/models/alexchilton/llama-3-1-8b-base/transformers/default/1', (None, None, '1'): '/kaggle/input/models/alexchilton/llama-3-1-8b-base/transformers/default/1', ('qwen3', None, None): '/kaggle/input/models/qwen-lm/qwen-3-embedding/transformers/8b/1', ('qwen3', 4096, None): '/kaggle/input/models/qwen-lm/qwen-3-embedding/transformers/8b/1', ('qwen3', 5120, None): '/kaggle/input/models/qwen-lm/qwen-3/transformers/14b-base/1', ('qwen3', 2560, None): '/kaggle/input/models/qwen-lm/qwen-3/transformers/4b-base/1', ('qwen2', None, None): '/kaggle/input/models/qwen-lm/qwen2.5/transformers/14b/1', ('qwen2', 5120, None): '/kaggle/input/models/qwen-lm/qwen2.5/transformers/14b/1', ('qwen2', 3584, None): '/kaggle/input/models/qwen-lm/qwen2.5/transformers/7b/1', ('deberta-v2', None, None): '/kaggle/input/name-that-llm-desklib-weights/desklib-detector', ('deberta-v2', 1024, None): '/kaggle/input/name-that-llm-desklib-weights/desklib-detector', (None, None, 'desklib-detector'): '/kaggle/input/name-that-llm-desklib-weights/desklib-detector', ('electra', None, None): '/kaggle/input/name-that-llm-electra-large-weights/electra-large-discriminator', ('electra', 1024, None): '/kaggle/input/name-that-llm-electra-large-weights/electra-large-discriminator', (None, None, 'electra-large-discriminator'): '/kaggle/input/name-that-llm-electra-large-weights/electra-large-discriminator', (None, None, 'deberta-v3-large'): '/kaggle/input/name-that-llm-extract-encoders2/deberta-v3-large', ('glm4', None, None): '/kaggle/input/name-that-llm-glm4-9b-trunc25/glm4_9b_trunc25', ('glm4', 4096, None): '/kaggle/input/name-that-llm-glm4-9b-trunc25/glm4_9b_trunc25', (None, None, 'glm4_9b_trunc25'): '/kaggle/input/name-that-llm-glm4-9b-trunc25/glm4_9b_trunc25', ('modernbert', None, None): '/kaggle/input/nlm-enc-v5/encoder', ('modernbert', 1024, None): '/kaggle/input/nlm-enc-v5/encoder', (None, None, 'encoder'): '/kaggle/input/nlm-enc-v5/encoder', ('deberta-v2', 768, None): '/kaggle/input/ntl-olmo2-native24-final-inference-assets/document-deberta-07nov/document-deberta-07nov/final', (None, None, 'final'): '/kaggle/input/ntl-olmo2-native24-final-inference-assets/document-deberta-07nov/document-deberta-07nov/final', ('qwen3_5', None, None): '/kaggle/input/ntl-olmo2-native24-final-inference-assets/gsq3-base/qwen38-27b-gsq3', (None, None, 'qwen38-27b-gsq3'): '/kaggle/input/ntl-olmo2-native24-final-inference-assets/gsq3-base/qwen38-27b-gsq3', (None, None, 'llama31-prefix24'): '/kaggle/input/ntl-olmo2-native24-final-inference-assets/llama31-prefix24', (None, None, None): '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/ntl-tabular-offline-assets', (None, None, 'ntl-tabular-offline-assets'): '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/ntl-tabular-offline-assets', (None, None, 'qwen38-27b-nf4'): '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-27b-nf4', (None, None, 'base_config'): '/kaggle/input/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/ntl-lora24-fivefold-training/lora24/final/base_config', ('olmo2', None, None): '/kaggle/input/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/olmo2-7b-fp16-offline-weights/olmo2-fp16-base', ('olmo2', 4096, None): '/kaggle/input/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/olmo2-7b-fp16-offline-weights/olmo2-fp16-base', (None, None, 'olmo2-fp16-base'): '/kaggle/input/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/olmo2-7b-fp16-offline-weights/olmo2-fp16-base', ('smollm3', None, None): '/kaggle/input/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/smollm3-3b-base-offline-weights/smollm3-base', ('smollm3', 2048, None): '/kaggle/input/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/smollm3-3b-base-offline-weights/smollm3-base', (None, None, 'smollm3-base'): '/kaggle/input/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/smollm3-3b-base-offline-weights/smollm3-base', (None, None, '1_Pooling'): '/kaggle/input/ntl-olmo2-native24-final-inference-assets/ntl-native-q8-multidepth-fivefold-0679/base/1_Pooling', (None, None, 'base'): '/kaggle/input/ntl-olmo2-native24-final-inference-assets/ntl-native-q8-multidepth-fivefold-0679/base', (None, None, 'qwen38-27b-mixed-tail8-text'): '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text', (None, None, 'desklib'): '/kaggle/input/ntl-two-stage-encoders/desklib', (None, None, 'modernbert'): '/kaggle/input/ntl-two-stage-encoders/modernbert', (None, None, 'qwen35-9b-safetensors'): '/kaggle/input/qwen35-9b-safetensors', ('ultrabert', None, None): '/kaggle/input/ultrabert-380m-ntl/fold0', ('ultrabert', 1024, None): '/kaggle/input/ultrabert-380m-ntl/fold0', (None, None, 'fold0'): '/kaggle/input/ultrabert-380m-ntl/fold0', (None, None, 'fold1'): '/kaggle/input/ultrabert-380m-ntl/fold1', (None, None, 'fold2'): '/kaggle/input/ultrabert-380m-ntl/fold2', (None, None, 'fold3'): '/kaggle/input/ultrabert-380m-ntl/fold3', (None, None, 'fold4'): '/kaggle/input/ultrabert-380m-ntl/fold4'}

def asset_path(value):
    if Path(value).exists():
        return value
    for flat,namespaced in MOUNT_ALIASES.items():
        for origin,target in ((flat,namespaced),(namespaced,flat)):
            if value==origin or value.startswith(origin+'/'):
                resolved=target+value[len(origin):]
                assert Path(resolved).exists(), ('missing exact input',value,resolved)
                return resolved
    return value

def asset_paths(key, recursive=False):
    values=[asset_path(v) for v in PATHS[key]]
    for value in values:
        assert Path(value).exists(), value
    return values.copy()

def checkpoint(model_type, hidden=None, dir_name=None):
    path=asset_path(CHECKPOINTS[(model_type,hidden,dir_name)])
    assert Path(path,'config.json').is_file(), path
    return path


In [ ]:
# Environment Bootstrap: install offline bitsandbytes, verify model/dataset assets & prepare open pool
from pinned_assets import asset_path,asset_paths
import glob
import json
import os
import re
import shutil
import subprocess
import sys
import time

import numpy as np
import pandas as pd

NB_T0 = time.time()
RUN_LIMIT_S = 7.0 * 3600  # Reserve ~1.8h for Stage-2 GLM-5.3/Flash likelihood
FINAL_LOGITS = '/kaggle/temp/final-logits.npz'

wheels = [asset_path('/kaggle/input/nlm-offline-wheels/bitsandbytes-0.50.2-py3-none-manylinux_2_24_x86_64.whl')]
if wheels:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '-q'] + wheels, capture_output=True, text=True, check=True)

from pinned_assets import asset_paths

def first(pattern):
    values=asset_paths(pattern)
    assert values, pattern
    return values[0]

def adapter_dir(name):
    return os.path.dirname(first(f'/kaggle/input/**/{name}/heads.pt'))

def meta_dir(name):
    filename=first(f'/kaggle/input/**/{name}/meta.json')
    return os.path.dirname(filename),json.load(open(filename))['cfg']


DATA = next(p for p in ['/kaggle/input/competitions/name-that-llm', '/kaggle/input/name-that-llm'] if os.path.exists(p + '/train.csv'))
FAST_COMMIT = os.getenv('KAGGLE_IS_COMPETITION_RERUN') is None and len(pd.read_csv(DATA + '/test.csv')) < 50   # ours: the scoring rerun never takes the fast path
print('FAST_COMMIT', FAST_COMMIT)

open_pool = pd.read_parquet(first('/kaggle/input/**/ntl_open_pool.parquet'), columns=['text', 'model', 'flag_train_prompt_regen', 'match_weight'])
open_pool = open_pool[~open_pool.flag_train_prompt_regen.astype(bool)]
open_pool[['text', 'model']].to_parquet('/kaggle/working/open_pool.parquet')
POOL = '/kaggle/working/open_pool.parquet'
print('svc pool rows', len(open_pool))


def draw(g, n):
    w = np.nan_to_num(g.match_weight.to_numpy(dtype=np.float64))
    return g.take(np.random.RandomState(0).choice(len(g), size=n, replace=False, p=w / w.sum()).astype(np.intp))


matched_counts = pd.read_parquet(first('/kaggle/input/**/nlm_matched_clean.parquet'), columns=['model']).model.value_counts()
extra = pd.concat([draw(g, min(len(g), int(matched_counts.get(m, 0)))) for m, g in open_pool.groupby('model')])
EXTRA = '/kaggle/working/extra_matched.parquet'
extra[['text', 'model']].to_parquet(EXTRA)
print('external rows', len(extra), 'matched_clean rows', int(matched_counts.sum()))
del open_pool, extra

B8 = asset_path('/kaggle/input/models/qwen-lm/qwen-3/transformers/8b-base/1')
B14 = asset_path('/kaggle/input/models/qwen-lm/qwen-3/transformers/14b-base/1')
B25_14 = asset_path('/kaggle/input/models/qwen-lm/qwen2.5/transformers/14b/1')
BL = asset_path('/kaggle/input/models/alexchilton/llama-3-1-8b-base/transformers/default/1')
SHARDS = {'model-00001-of-00004.safetensors': 4976698672, 'model-00002-of-00004.safetensors': 4999802720,
          'model-00003-of-00004.safetensors': 4915916176, 'model-00004-of-00004.safetensors': 1168138808}
assert all(os.path.getsize(os.path.join(BL, f)) == n for f, n in SHARDS.items()), BL
if not FAST_COMMIT:
    shutil.copytree(BL, '/tmp/llama-3.1/8b', dirs_exist_ok=True)
    BL = '/tmp/llama-3.1/8b'

AQ8V5 = [adapter_dir(f'q8v5_f{k}') for k in range(5)] + [adapter_dir('q8v5')]
AQ14V5 = [adapter_dir(f'q14v5_f{k}') for k in range(5)] + [adapter_dir('q14v5')]
ALL8V5 = [adapter_dir(f'll8v5_f{k}') for k in range(5)] + [adapter_dir('ll8v5')]
N14B, N14BC = meta_dir('lora_qwen3_14b')
Q25_14, Q25_14C = meta_dir('lora_qwen25_14b')
OQ1 = os.path.dirname(first('/kaggle/input/**/ntl-q14-lora/heads.safetensors'))
OQ4 = os.path.dirname(first('/kaggle/input/**/ntl-q14-lora-v4/heads.safetensors'))

NATIVE = os.path.dirname(first('/kaggle/input/**/ntl-native-q8-multidepth-fivefold-0679/manifest.json'))
NATIVE_BASE = asset_path('/kaggle/input/models/qwen-lm/qwen-3-embedding/transformers/8b/1')
assert all(os.path.exists(f'{NATIVE}/fold{k}/trainables.safetensors') and os.path.exists(f'{NATIVE}/calibration/fold{k}.json') for k in range(5)), NATIVE
assert os.path.exists(f'{NATIVE}/base/model.safetensors.index.json') and os.path.exists(f'{NATIVE_BASE}/model.safetensors.index.json'), (NATIVE, NATIVE_BASE)

LB_ROOT = os.path.dirname(first('/kaggle/input/**/ntl666-cache-parts.json'))
GSQ_ASSETS = LB_ROOT
PREFIT = '/kaggle/temp/ntl666-prefit'
import zipfile
from pathlib import PurePosixPath
parts_meta = json.loads(open(f'{LB_ROOT}/ntl666-cache-parts.json').read())
assert parts_meta['format'] == 'ntl666-cache-parts-v1'
assert [x['name'] for x in parts_meta['parts']] == [f'ntl666-prefit.part{i:03d}.bin' for i in range(len(parts_meta['parts']))]
os.makedirs('/kaggle/temp', exist_ok=True)
shutil.rmtree(PREFIT, ignore_errors=True)
prefit_zip = '/kaggle/temp/ntl666-prefit.zip'
try:
    with open(prefit_zip, 'wb') as joined:
        for part in parts_meta['parts']:
            assert os.path.getsize(f"{LB_ROOT}/{part['name']}") == part['bytes'], part['name']
            with open(f"{LB_ROOT}/{part['name']}", 'rb') as stream:
                shutil.copyfileobj(stream, joined, 16 * 1024 * 1024)
    assert os.path.getsize(prefit_zip) == parts_meta['archive_bytes']
    with zipfile.ZipFile(prefit_zip) as archive:
        entries = archive.infolist()
        assert len(entries) == parts_meta['files'] and len({e.filename for e in entries}) == len(entries)
        for entry in entries:
            path = PurePosixPath(entry.filename)
            assert not path.is_absolute() and '..' not in path.parts and chr(92) not in entry.filename and path.parts[0] == 'ntl666-prefit' and not entry.is_dir()
        assert archive.read('ntl666-prefit/COMPLETE') == b'ntl666-v14-data-numeric-v1' + chr(10).encode()
        provenance = json.loads(archive.read('ntl666-prefit/provenance.json'))
        assert provenance['status'] == 'COMPLETE' and provenance['pool_rows'] == 190588 and provenance['dataset_version'] == 13
        archive.extractall('/kaggle/temp')
finally:
    if os.path.exists(prefit_zip):
        os.remove(prefit_zip)
for name in ('core/svc4', 'core/tfidf', 'core/distort'):
    assert os.path.isdir(f'{PREFIT}/{name}'), name
assert os.path.exists(f'{GSQ_ASSETS}/gsq3-base/qwen38-27b-gsq3/config.json') and os.path.exists(f'{GSQ_ASSETS}/gsq120k-trained/gsq120k-inference/gsq/final.safetensors')
assert len([asset_path(p) for p in ['/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/accelerate-1.15.0-py3-none-any.whl', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/bitsandbytes-0.50.2-py3-none-manylinux_2_24_x86_64.whl', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/huggingface_hub-1.32.0-py3-none-any.whl', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/safetensors-0.8.0-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/transformers-5.14.1-py3-none-any.whl']]) >= 6

DEB = os.path.dirname(first('/kaggle/input/**/deberta_v3_large/meta.json'))
if not FAST_COMMIT:
    shutil.copytree(DEB, '/kaggle/working/deberta_v3_large', dirs_exist_ok=True)
    DEB = '/kaggle/working/deberta_v3_large'


_pb115_hits = sorted(glob.glob('/kaggle/input/**/paired_token_features.parquet', recursive=True))
assert _pb115_hits, 'lumibee/ntl-prefixb-full-115 dataset not attached'
PB115_DIR = os.path.dirname(_pb115_hits[0])
assert os.path.isfile(os.path.join(PB115_DIR, 'spec.json')), PB115_DIR
for name, value in dict(PB115_DIR=PB115_DIR, B8=B8, B14=B14, B25_14=B25_14, BL=BL, N14B=N14B, Q25_14=Q25_14, OQ1=OQ1, OQ4=OQ4, NATIVE=NATIVE, NATIVE_BASE=NATIVE_BASE, PREFIT=PREFIT, GSQ_ASSETS=GSQ_ASSETS, DEB=DEB).items():
    print(f'{name:10s} {value}')
print('setup', round(time.time() - NB_T0), 's')

## 2. Core Inference & Calibration Modules
Modular Python components exported to the working directory: stylometric sibling refinement (`pngy_residual.py`), dual-GPU queue (`pipe.py`), shared-prompt Bayesian Gaussian head (`gauss.py`), ATLAS (`atlas_infer.py`), Native Q8 (`native_q8.py`), spectral prefit reader (`ntl660_readout.py`), GSQ-27B (`gsq_*.py`), prompt-group propagation (`qwen_groups.py`), and the Stage-1 ensemble orchestrator (`infer.py`).

In [ ]:
%%writefile pngy_residual.py
"""Mass-Preserving Intra-Family Sibling Log-Ratio Refinement (Stylometry).

Extracts a dense structural, punctuation, and function-word stylistic vector
and fits regularized family-level logistic heads (`claude`, `gpt`, `gemini`,
`glm`). Applies confidence-gated probability residual adjustments strictly
within sibling families without altering cross-family mass."""
import re
import numpy as np
from scipy.special import logsumexp, softmax
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

MARKERS = (",", ".", ";", ":", "!", "?", "—", "–", "’", "“", "”",
           "'", '"', "(", ")", "[", "]", "*", "", "**", " - ", "  ")
FUNCTIONS = ("the", "and", "but", "which", "however", "therefore", "moreover", "because", "perhaps",
             "might", "would", "could", "should", "also", "although", "overall", "finally", "we", "you", "i")
WORDS = re.compile(r"[A-Za-z]+(?:['’][A-Za-z]+)?")

def features(texts):
    rows = []
    for value in texts:
        if not isinstance(value, str):
            value = str(value)
        counts = np.array([value.count(m) for m in MARKERS], dtype=np.float64)
        tokens = WORDS.findall(value)
        n, chars = max(1, len(tokens)), max(1, len(value))
        lengths = np.array([len(t) for t in tokens] or [0], dtype=float)
        lower = [t.lower() for t in tokens]
        shape = [np.log1p(len(tokens)), np.log1p(len(value)), chars/n,
                 len(set(lower))/n, lengths.mean(), lengths.std(),
                 sum(c.isupper() for c in value)/chars, sum(c.isdigit() for c in value)/chars,
                 sum(ord(c) > 127 for c in value)/chars, sum(c.isspace() for c in value)/chars]
        rows.append(np.r_[shape, counts/n, [lower.count(w)/n for w in FUNCTIONS]])
    return np.asarray(rows, dtype=np.float64)

def family_columns(labels):
    return [np.array([i for i, label in enumerate(labels) if label.startswith(f + "-")])
            for f in ("claude", "gpt", "gemini", "glm")]

def refine(log_probability, evidence, families, strength=0.10):
    z, style = np.asarray(log_probability, dtype=float), np.asarray(evidence, dtype=float)
    result = z.copy()
    for columns in families:
        if len(columns) < 2:
            continue
        p = softmax(z[:, columns], axis=1)
        q = np.maximum(style[:, columns], 1e-15)
        q /= q.sum(1, keepdims=True)
        top = np.partition(p, -2, axis=1)[:, -2:]
        gate = np.clip((0.65 - (top[:, 1] - top[:, 0])) / 0.65, 0, 1) * strength
        mixed = (1 - gate[:, None]) * p + gate[:, None] * q
        result[:, columns] = logsumexp(z[:, columns], axis=1, keepdims=True) + np.log(mixed)
    return result

class Residual:
    def __init__(self, labels):
        self.labels = list(labels)
        self.families = family_columns(labels)

    def fit(self, texts, target):
        x = features(texts)
        target = np.asarray(target)
        self.models = []
        for columns in self.families:
            if len(columns) < 2:
                continue
            selected = np.isin(target, np.asarray(self.labels)[columns])
            model = make_pipeline(StandardScaler(), LogisticRegression(
                C=0.03, max_iter=500, class_weight="balanced", random_state=166))
            model.fit(x[selected], target[selected])
            self.models.append((columns, model))
        return self

    def predict_log_probability(self, base, texts):
        x = features(texts)
        evidence = np.ones((len(x), len(self.labels)), dtype=float)
        for columns, model in self.models:
            p = model.predict_proba(x)
            for j, label in enumerate(model.classes_):
                evidence[:, self.labels.index(label)] = p[:, j]
        return refine(base, evidence, self.families)



In [ ]:
%%writefile pipe.py
"""Asynchronous Multi-Device Worker Queue and IPC Orchestration.

Coordinates dual Nvidia Tesla T4 GPU execution pipelines via bounded queues,
explicit memory barriers, and deterministic garbage collection to prevent CUDA
out-of-memory errors across sequential multi-backbone adapter passes."""
import gc
import os
import threading
import time
from collections import deque

import numpy as np


def _flat(x):
    if isinstance(x, dict):
        parts = [x[k] for k in sorted(x)]
    else:
        parts = x if isinstance(x, (tuple, list)) else [x]
    return np.concatenate([np.ravel(np.asarray(p, dtype=np.float64)) for p in parts])


def _agree(batches, work, workers, tag, log):
    probe = [batches[len(batches) // 2 + i] for i in range(workers)]
    try:
        ref = [work(b) for b in probe]
        if any(r is None for r in ref):
            return False
        got = [None] * workers
        err = []

        def go(i):
            try:
                got[i] = work(probe[i])
            except Exception as e:
                err.append(e)

        threads = [threading.Thread(target=go, args=(i,)) for i in range(workers)]
        [t.start() for t in threads]
        [t.join() for t in threads]
        ok = not err and all(g is not None and np.abs(_flat(r) - _flat(g)).max() <= 2e-2 * (np.abs(_flat(r)).mean() + 1e-6) for r, g in zip(ref, got))
    except Exception as e:
        log(f'{tag}: concurrent check raised {e!r:.100}')
        ok = False
    log(f'{tag}: concurrent check {"passed" if ok else "failed, single worker"}')
    return ok


def pipelined(batches, work, store, fallback, workers=2, deadline=None, tag='pipe', log=print):
    batches = list(batches)
    t0 = time.time()
    workers = min(workers, int(os.environ.get('NTL_PIPE_WORKERS', workers)))
    if workers < 2 or len(batches) < 6 or not _agree(batches, work, workers, tag, log):
        workers = 1
    queue = deque(batches)
    redo = []
    lock = threading.Lock()
    stop = threading.Event()

    def loop():
        while not stop.is_set():
            if deadline is not None and time.time() >= deadline:
                return
            with lock:
                if not queue:
                    return
                b = queue.popleft()
            try:
                r = work(b)
            except Exception:
                with lock:
                    redo.append(b)
                stop.set()
                return
            with lock:
                if r is None:
                    redo.append(b)
                else:
                    store(b, r)

    if workers == 1:
        while queue and not (deadline is not None and time.time() >= deadline):
            b = queue.popleft()
            r = work(b)
            if r is None:
                redo.append(b)
            else:
                store(b, r)
    else:
        threads = [threading.Thread(target=loop) for _ in range(workers)]
        [t.start() for t in threads]
        [t.join() for t in threads]
    rest = redo + list(queue)
    if rest:
        gc.collect()
        try:
            import torch
            torch.cuda.empty_cache()
        except Exception:
            pass
    for b in rest:
        if deadline is not None and time.time() >= deadline:
            break
        fallback(b)
    log(f'{tag}: {len(batches)} batches, {workers} worker(s), {len(redo)} redone sequentially, {time.time() - t0:.0f}s')


def spans_devices(model):
    return len({p.device for p in model.parameters()}) > 1

In [ ]:
%%writefile gauss.py
"""Shared-Prompt Bayesian Gaussian Head with Whitened PCA & Cavity-EM.

Projects multi-depth transformer embeddings via Whitened PCA (`d=128`), applies
Ledoit-Wolf covariance shrinkage, and computes prompt-grouped posterior likelihood
ratios to disambiguate completions sampled from shared prompts."""
import itertools
import numpy as np
K = 18
DEFAULT = {"red": "wpca", "d": 256, "se_shrink": "lw", "sp_shrink": "lw", "lam_floor": 1e-4, "n_exact": 3, "inj": True,
           "iters": 5, "repel": True, "sp_scale": 1.0}

def _lse(A, axis=None, keepdims=False):
    m = np.max(A, axis=axis, keepdims=True); m = np.where(np.isfinite(m), m, 0.0)
    out = np.log(np.sum(np.exp(A - m), axis=axis, keepdims=True)) + m
    if keepdims: return out
    return np.squeeze(out, axis=axis) if axis is not None else float(out.item())

def log_softmax(L): return L - _lse(L, axis=1, keepdims=True)

def _lw_shrinkage(X):
    from sklearn.covariance import ledoit_wolf_shrinkage
    return float(ledoit_wolf_shrinkage(X, assume_centered=True))

def _shrunk(S, a):
    d = S.shape[0]; return (1 - a) * S + a * (np.trace(S) / d) * np.eye(d)

def _two_way(Z, y, pinv, G, iters=4):
    P = np.zeros((G, Z.shape[1])); cnt_p = np.bincount(pinv, minlength=G).astype(float)
    cls = np.unique(y); mu = np.zeros((K, Z.shape[1]))
    for _ in range(iters):
        R = Z - P[pinv]
        for c in cls: mu[c] = R[y == c].mean(0)
        R = Z - mu[y]; P = np.zeros_like(P); np.add.at(P, pinv, R); P /= cnt_p[:, None]
    return mu, P, cnt_p

def fit(X, y, prompts, cfg=None):
    c = dict(DEFAULT); c.update(cfg or {})
    X = np.asarray(X, dtype=np.float64); y = np.asarray(y); N, D = X.shape
    pu, pinv = np.unique(np.asarray(prompts), return_inverse=True); G = len(pu)
    mx = X.mean(0); sx = X.std(0); sx = np.where(sx > 1e-8, sx, 1.0); Xs = (X - mx) / sx
    if c["red"] == "none":
        V = np.eye(D)
    else:
        if c["red"] == "wpca":
            Pm = np.zeros((G, D)); np.add.at(Pm, pinv, Xs); Pm /= np.bincount(pinv)[:, None]; R = Xs - Pm[pinv]
        elif c["red"] == "pca": R = Xs
        else: raise ValueError(c["red"])
        d = int(min(c["d"], N - G - 1, D))
        if N < D:
            w, U = np.linalg.eigh(R @ R.T); o = np.argsort(w)[::-1][:d]; w, U = np.clip(w[o], 1e-12, None), U[:, o]
            V = (R.T @ U) / np.sqrt(w)
        else:
            _, _, Vt = np.linalg.svd(R, full_matrices=False); V = Vt[:d].T
    Z = Xs @ V; d = Z.shape[1]
    mu, P, ng = _two_way(Z, y, pinv, G)
    E = Z - mu[y] - P[pinv]; dof = max(N - G - len(np.unique(y)) + 1, 1)
    Se_raw = E.T @ E / dof
    ae = _lw_shrinkage(E) if c["se_shrink"] == "lw" else float(c["se_shrink"]); Se = _shrunk(Se_raw, ae)
    Pc = P - P.mean(0); Sp_raw = Pc.T @ Pc / max(G - 1, 1) - np.mean(1.0 / ng) * Se_raw
    ap = _lw_shrinkage(Pc) if c["sp_shrink"] == "lw" else float(c["sp_shrink"])
    tr = max(np.trace(Sp_raw), 1e-8 * np.trace(Se)); Sp = (1 - ap) * Sp_raw + ap * (tr / d) * np.eye(d); Sp *= c["sp_scale"]
    L = np.linalg.cholesky(Se); Li = np.linalg.inv(L); M = Li @ Sp @ Li.T; M = 0.5 * (M + M.T)
    lam, Q = np.linalg.eigh(M); lam = np.clip(lam, c["lam_floor"], None); A = Q.T @ Li
    W = V @ A.T
    return {"mx": mx, "sx": sx, "W": W, "m": mu @ A.T, "lam": lam, "T1": 1.0, "Tg": 1.0,
            "info": {"d": int(d), "a_e": round(ae, 4), "a_p": round(ap, 4), "lam_pct": np.round(np.percentile(lam, [5, 50, 95, 100]), 3).tolist()}, "cfg": c}

def transform(X, prm): return ((np.asarray(X, dtype=np.float64) - prm["mx"]) / prm["sx"]) @ prm["W"]

_MASK = {}
def _inj_mask(n, C):
    if (n, C) not in _MASK:
        grids = np.meshgrid(*[np.arange(C)] * n, indexing="ij"); bad = np.zeros([C] * n, bool)
        for i, j in itertools.combinations(range(n), 2): bad |= grids[i] == grids[j]
        _MASK[(n, C)] = bad
    return _MASK[(n, C)]

def singleton_ll(U, m, lam):
    R = U[:, None, :] - m[None]; return -0.5 * (R ** 2 / (1 + lam)).sum(-1)

def joint_terms(U, m, lam):
    n = len(U); gam = lam / (1 + n * lam); R = U[:, None, :] - m[None]
    a = -0.5 * (R ** 2 * (1 - gam)).sum(-1); G = R * np.sqrt(gam); B = np.einsum("icd,jed->ijce", G, G)
    return a, B

def joint_table(a, B):
    n, C = a.shape; Q = np.zeros([C] * n)
    for i in range(n):
        sh = [1] * n; sh[i] = C; Q = Q + a[i].reshape(sh)
    for i, j in itertools.combinations(range(n), 2):
        sh = [1] * n; sh[i] = C; sh[j] = C; Q = Q + B[i, j].reshape(sh)
    return Q

def exact_marginals(Q, T=1.0, inj=True):
    n = Q.ndim; C = Q.shape[0]; Qt = Q / T
    if inj and n >= 2: Qt = np.where(_inj_mask(n, C), -np.inf, Qt)
    lz = _lse(Qt); out = np.empty((n, C))
    for i in range(n):
        ax = tuple(k for k in range(n) if k != i); out[i] = (_lse(Qt, axis=ax) if ax else Qt) - lz
    return out

def em_marginals(U, m, lam, T=1.0, iters=5, repel=False, q0=None):
    n = len(U); v = lam / (1 + (n - 1) * lam)
    L = singleton_ll(U, m, lam) / T if q0 is None else np.log(np.clip(q0, 1e-300, None))
    q = np.exp(log_softmax(L))
    for _ in range(iters):
        s = U - q @ m; S = s.sum(0); Pm = (S[None] - s) * v
        R = U[:, None, :] - Pm[:, None, :] - m[None]; L = -0.5 * (R ** 2 / (1 + v)).sum(-1) / T
        if repel:
            lo = np.log(np.clip(1 - q, 1e-6, None)); L = L + (lo.sum(0)[None] - lo)
        q = np.exp(log_softmax(L))
    return log_softmax(L)

LAM_DIFF = 1e6
def lam_g(prm, cfg=None):
    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {})
    return np.full_like(prm["lam"], LAM_DIFF) if c.get("diff") else prm["lam"]

def group_logpost(U, prm, T1=None, Tg=None, cfg=None):
    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {}); n = len(U); lg = lam_g(prm, cfg)
    T1 = prm["T1"] if T1 is None else T1; Tg = prm["Tg"] if Tg is None else Tg
    if n == 1: return log_softmax(singleton_ll(U, prm["m"], prm["lam"]) / T1)
    if n <= c["n_exact"]:
        a, B = joint_terms(U, prm["m"], lg); return exact_marginals(joint_table(a, B), Tg, c["inj"])
    return em_marginals(U, prm["m"], lg, Tg, c["iters"], c["repel"])

def scores(X_pool, lab, prm, T1=None, Tg=None, cfg=None, U=None):
    U = transform(X_pool, prm) if U is None else U; lab = np.asarray(lab); n = len(U); out = np.empty((n, prm["m"].shape[0]))
    sizes = np.bincount(lab); s1 = sizes[lab] == 1
    T1 = prm["T1"] if T1 is None else T1
    if s1.any(): out[s1] = log_softmax(singleton_ll(U[s1], prm["m"], prm["lam"]) / T1)
    for g in np.where(sizes >= 2)[0]:
        ii = np.where(lab == g)[0]; out[ii] = group_logpost(U[ii], prm, T1, Tg, cfg)
    return out


In [ ]:
%%writefile atlas_infer.py
"""ATLAS Multi-Layer Transformer Feature Extraction & Readout Module.

Pools hidden state representations across intermediate transformer layers
and executes fold-averaged linear/MLP classification heads under FP16/BF16
automatic numerical-stability fallback."""
import gc
import json
import os
import re
import threading
import time

import numpy as np
import torch
from safetensors.torch import load_file
from torch import nn
from torch.nn import functional as F

from pipe import pipelined

TEMPERATURES = (1.0, 2.0, 4.0, 8.0)
HEAD_CHUNK_TOKENS = 64


class Route(threading.local):
    labels = None


class SourceLinear(nn.Module):
    def __init__(self, base, sources, rank, alpha, route):
        super().__init__()
        self.base = base
        self.route = route
        self.scaling = alpha / rank
        dev = base.weight.device
        self.a = nn.Parameter(torch.zeros(rank, base.in_features, device=dev, dtype=torch.float32), requires_grad=False)
        self.b = nn.Parameter(torch.zeros(sources, base.out_features, rank, device=dev, dtype=torch.float32), requires_grad=False)

    def forward(self, x):
        labels = self.route.labels
        if labels is None or labels.numel() != x.shape[0]:
            raise ValueError('source label missing for a batch row')
        out = self.base(x)
        delta = torch.bmm(F.linear(x.float(), self.a), self.b.index_select(0, labels.to(self.b.device)).transpose(1, 2))
        return out + (delta * self.scaling).to(out.dtype)


def score_columns(logp, mask):
    lp = logp.float()
    valid = mask.float().unsqueeze(1)
    mean_lp = (lp * valid).sum(-1) / valid.sum(-1).clamp_min(1)
    gap = (lp.unsqueeze(1) - lp.unsqueeze(2)).clamp_min(0).sum(2)
    cols = [mean_lp] + [(torch.exp(-gap / t) * valid).sum(-1) / valid.sum(-1).clamp_min(1) for t in TEMPERATURES]
    return torch.stack(cols, -1)


class AtlasModel(nn.Module):
    def __init__(self, base, labels, rank, alpha):
        super().__init__()
        self.base = base
        self.labels = list(labels)
        self.route = Route()
        self.base.requires_grad_(False)
        self.base.config.use_cache = False
        hit = [(n, m) for n, m in self.base.named_modules() if n.split('.')[-1] in ('q_proj', 'k_proj', 'v_proj', 'o_proj') and isinstance(m, nn.Linear)]
        if not hit:
            raise ValueError('no attention projections found')
        for name, m in hit:
            parent, leaf = name.rsplit('.', 1)
            setattr(self.base.get_submodule(parent), leaf, SourceLinear(m, len(labels), rank, alpha, self.route))
        emb = self.base.get_input_embeddings().weight
        self.control = nn.Parameter(torch.zeros(len(labels), 1, emb.shape[1], device=emb.device, dtype=torch.float32), requires_grad=False)

    @property
    def device(self):
        return self.control.device

    def load_state(self, state):
        own = {n: p for n, p in self.named_parameters() if n.endswith(('.a', '.b')) or n == 'control'}
        if set(own) != set(state):
            raise ValueError(f'adapter key mismatch: {sorted(set(own) ^ set(state))[:6]}')
        with torch.no_grad():
            for n, p in own.items():
                t = state[n]
                if tuple(t.shape) != tuple(p.shape) or not torch.isfinite(t).all():
                    raise ValueError(f'bad adapter tensor {n}')
                p.copy_(t.to(device=p.device, dtype=p.dtype))

    def token_logp(self, ids, mask, labels):
        self.route.labels = labels
        tok = self.base.get_input_embeddings()(ids)
        ctrl = self.control.index_select(0, labels).to(tok.dtype)
        embeds = torch.cat([ctrl, tok], 1)
        attn = torch.cat([mask.new_ones(len(ids), 1), mask], 1)
        hid = self.base.model(inputs_embeds=embeds, attention_mask=attn, use_cache=False).last_hidden_state[:, 1:-1]
        tgt = ids[:, 1:]
        head = self.base.get_output_embeddings()
        parts = []
        for s in range(0, hid.shape[1], HEAD_CHUNK_TOKENS):
            h, y = hid[:, s:s + HEAD_CHUNK_TOKENS], tgt[:, s:s + HEAD_CHUNK_TOKENS]
            logits = head(h.to(head.weight.device)).float()
            parts.append(-F.cross_entropy(logits.reshape(-1, logits.shape[-1]), y.to(logits.device).reshape(-1), reduction='none').reshape(y.shape))
        return torch.cat(parts, 1).to(ids.device)

    @torch.no_grad()
    def score(self, ids, mask, chunk=3):
        outs = []
        for first in range(0, len(self.labels), chunk):
            last = min(len(self.labels), first + chunk)
            src = torch.arange(first, last, device=ids.device).repeat_interleave(len(ids))
            lp = self.token_logp(ids.repeat(last - first, 1), mask.repeat(last - first, 1), src)
            outs.append(lp.reshape(last - first, len(ids), -1).transpose(0, 1))
        cols = score_columns(torch.cat(outs, 1), mask[:, 1:])
        return cols, torch.isfinite(cols).flatten(1).all(1)


def split_device_map(base_path, n_gpu, head_weight=2.4):
    from accelerate import init_empty_weights
    from transformers import AutoConfig, AutoModelForCausalLM
    if n_gpu < 2:
        return {'': 0}
    cfg = AutoConfig.from_pretrained(base_path)
    with init_empty_weights():
        meta = AutoModelForCausalLM.from_config(cfg, torch_dtype=torch.float16)
    first = int(round((cfg.num_hidden_layers + head_weight) / 2.0))
    dm = {'model.embed_tokens': 0, 'model.rotary_emb': 0, 'model.norm': 1, 'lm_head': 1}
    for i in range(cfg.num_hidden_layers):
        dm[f'model.layers.{i}'] = 0 if i < first else 1
    tensors = [n for n, _ in meta.named_parameters()] + [n for n, _ in meta.named_buffers()]
    missing = [n for n in tensors if not any(n == k or n.startswith(k + '.') for k in dm)]
    if missing:
        raise RuntimeError(f'device map does not cover {missing[:3]}')
    return dm


class Scorer:
    def __init__(self, base_path, bundle_dir, log=print):
        self.log = log
        meta = json.load(open(os.path.join(bundle_dir, 'atlas_bundle.json'), encoding='utf-8'))
        self.labels = list(meta['labels'])
        cfg = meta['config']
        n_gpu = torch.cuda.device_count()
        t0 = time.time()
        from transformers import AutoModelForCausalLM, AutoTokenizer
        self.tok = AutoTokenizer.from_pretrained(base_path)
        self.tok.padding_side = 'right'
        if self.tok.pad_token_id is None:
            self.tok.pad_token = self.tok.eos_token
        try:
            dm = split_device_map(base_path, n_gpu)
        except Exception as e:
            self.log(f'explicit device map failed ({e!r:.100}), using device_map=auto')
            dm = 'auto'
        base = AutoModelForCausalLM.from_pretrained(base_path, dtype=torch.float16, attn_implementation='sdpa', device_map=dm)
        if any(p.device.type != 'cuda' for p in base.parameters()):
            raise RuntimeError('ATLAS base did not fit on the GPUs')
        self.model = AtlasModel(base, self.labels, cfg['rank'], cfg['alpha'])
        self.model.load_state(load_file(os.path.join(bundle_dir, 'adapters.safetensors')))
        self.model.eval()
        per_dev = {}
        for p in base.parameters():
            per_dev[str(p.device)] = per_dev.get(str(p.device), 0) + p.numel() * p.element_size()
        self.max_len = int(cfg['max_length'])
        head_path = os.path.join(bundle_dir, 'atlas_head.json')
        self.head = json.load(open(head_path)) if os.path.exists(head_path) else None
        lite_path = os.path.join(bundle_dir, 'atlas_head_meanlp.json')
        self.lite = json.load(open(lite_path)) if os.path.exists(lite_path) else None
        self.log(f'atlas loaded in {time.time() - t0:.0f}s, weights per device: ' + ', '.join(f'{k} {v / 2 ** 30:.1f} GiB' for k, v in sorted(per_dev.items())))
        self.n_gpu = n_gpu

    def encode(self, texts):
        return self.tok(list(texts), truncation=True, max_length=self.max_len)['input_ids']

    def _pack(self, enc, rows):
        pk = self.tok.pad({'input_ids': [enc[int(i)] for i in rows]}, padding=True, return_tensors='pt')
        return pk['input_ids'].to(self.model.device), pk['attention_mask'].to(self.model.device)

    def _chunks(self, enc, rows, batch, chunk, max_tokens):
        lens = np.array([len(enc[int(i)]) for i in rows])
        order = np.asarray(rows)[np.argsort(-lens, kind='stable')]
        out, a = [], 0
        while a < len(order):
            width = max(1, len(enc[int(order[a])]))
            size = max(1, min(batch, max_tokens // (chunk * width)))
            out.append(order[a:a + size])
            a += size
        return out

    def _score_rows(self, enc, rows, chunk):
        ids, mask = self._pack(enc, rows)
        p, ok = self.model.score(ids, mask, chunk)
        return p.float().cpu().numpy(), ok.cpu().numpy()

    def run(self, enc, rows, deadline, batch=4, chunk=3, max_tokens=14000, workers=2):
        out = np.zeros((len(enc), len(self.labels), 5), np.float32)
        done = np.zeros(len(enc), dtype=bool)
        chunks = self._chunks(enc, rows, batch, chunk, max_tokens)
        total = float(sum(len(enc[int(i)]) for i in rows))
        stat = {'invalid': 0, 'oom': 0, 'tokens': 0, 'rows': 0, 'next': 200, 't0': time.time()}

        def work(item):
            return self._score_rows(enc, item, chunk)

        def store(item, res):
            p, ok = res
            good = ok & np.isfinite(p).reshape(len(p), -1).all(1)
            out[item[good]] = p[good]
            done[item[good]] = True
            stat['invalid'] += int((~good).sum())
            stat['rows'] += len(item)
            stat['tokens'] += sum(len(enc[int(i)]) for i in item)
            if stat['rows'] >= stat['next']:
                stat['next'] += 200
                spent = time.time() - stat['t0']
                self.log(f"atlas {stat['rows']}/{len(rows)} rows, {spent:.0f}s elapsed, about {spent / stat['tokens'] * (total - stat['tokens']):.0f}s left")

        def fallback(item, c=chunk):
            try:
                store(item, self._score_rows(enc, item, c))
            except torch.cuda.OutOfMemoryError:
                gc.collect()
                torch.cuda.empty_cache()
                stat['oom'] += 1
                if len(item) > 1:
                    half = len(item) // 2
                    fallback(item[:half], c)
                    fallback(item[half:], c)
                elif c > 1:
                    fallback(item, 1)
                else:
                    stat['invalid'] += 1

        pipelined(chunks, work, store, fallback, workers if self.n_gpu >= 2 else 1, deadline, 'atlas', self.log)
        self.log(f"atlas scored {int(done.sum())}/{len(rows)} rows in {time.time() - stat['t0']:.0f}s (invalid {stat['invalid']}, oom retries {stat['oom']})")
        return out, done

    def run_candidates(self, enc, cand, deadline, max_seqs=16, max_tokens=8192, workers=1):
        n, k = cand.shape
        rows = np.repeat(np.arange(n), k)
        labels = cand.ravel()
        lens = np.array([len(enc[int(r)]) for r in rows])
        order = np.argsort(-lens, kind='stable')
        batches, a = [], 0
        while a < len(order):
            size = max(1, min(max_seqs, max_tokens // max(1, int(lens[order[a]]))))
            batches.append(order[a:a + size])
            a += size
        out = np.full(n * k, np.nan, np.float32)
        stat = {'done': 0, 'next': 2000, 't0': time.time(), 'oom': 0}

        @torch.no_grad()
        def work(item):
            pk = self.tok.pad({'input_ids': [enc[int(rows[i])] for i in item]}, padding=True, return_tensors='pt')
            ids, mask = pk['input_ids'].to(self.model.device), pk['attention_mask'].to(self.model.device)
            lp = self.model.token_logp(ids, mask, torch.as_tensor(labels[item], device=ids.device))
            valid = mask[:, 1:].float()
            return ((lp * valid).sum(1) / valid.sum(1).clamp_min(1)).float().cpu().numpy()

        def store(item, vals):
            out[item] = np.where(np.isfinite(vals), vals, np.nan)
            stat['done'] += len(item)
            if stat['done'] >= stat['next']:
                stat['next'] += 2000
                spent = time.time() - stat['t0']
                self.log(f"atlas lite {stat['done']}/{len(order)} sequences, {spent:.0f}s elapsed, about {spent / stat['done'] * (len(order) - stat['done']):.0f}s left")

        def fallback(item):
            try:
                store(item, work(item))
            except torch.cuda.OutOfMemoryError:
                gc.collect()
                torch.cuda.empty_cache()
                stat['oom'] += 1
                if len(item) > 1:
                    half = len(item) // 2
                    fallback(item[:half])
                    fallback(item[half:])

        pipelined(batches, work, store, fallback, workers if self.n_gpu >= 2 else 1, deadline, 'atlas lite', self.log)
        out = out.reshape(n, k)
        done = np.isfinite(out).all(1)
        self.log(f"atlas lite scored {int(done.sum())}/{n} rows ({n * k} sequences) in {time.time() - stat['t0']:.0f}s (oom retries {stat['oom']})")
        return out, done

    def candidate_logprobs(self, mean_lp, cand):
        z = self.lite['slope'] * mean_lp + np.asarray(self.lite['b'])[cand]
        z = z - z.max(1, keepdims=True)
        return z - np.log(np.exp(z).sum(1, keepdims=True))

    def calibrated(self, scores):
        feats = scores - scores.mean(1, keepdims=True)
        z = (feats / np.asarray(self.head['std'])) @ np.asarray(self.head['a']) + np.asarray(self.head['b'])
        z = z - z.max(1, keepdims=True)
        return z - np.log(np.exp(z).sum(1, keepdims=True))

    def close(self):
        del self.model
        gc.collect()
        torch.cuda.empty_cache()

In [ ]:
%%writefile native_q8.py
"""Five-Fold Native Qwen3-Embedding-8B Multi-Depth Inference Engine.

Loads calibrated 5-fold trainable states on top of Qwen3-Embedding-8B,
computes L2-normalized multi-depth passage embeddings, and applies prompt-aware
geometric centering with out-of-fold temperature calibration."""
import argparse
import gc
import json
import os
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd

INSTRUCTION = 'Instruct: Identify the language model that generated the text based on its writing style.\nQuery: '
DEPTHS = (3, 6, 9, 12, 18, 24, 30, 36)
SUFFIXES = ('q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj')
RANK = 16
SCALE = 2.0
TOKEN_LIMIT = 512
TERMINAL = 151643


def log_softmax(z):
    z = np.asarray(z, dtype=np.float64)
    z = z - z.max(axis=-1, keepdims=True)
    return z - np.log(np.exp(z).sum(axis=-1, keepdims=True))


def calibrate(plain, centered, bias, groups, selections, use_centered=True):
    plain, centered, bias = (np.asarray(x, dtype=np.float64) for x in (plain, centered, bias))
    groups = np.asarray(groups)
    folds = plain.shape[0]
    assert plain.shape == centered.shape and plain.ndim == 3 and plain.shape[-1] == 18
    assert bias.shape == (folds, 18) and groups.shape == (plain.shape[1],) and len(selections) == folds
    assert all(np.isfinite(x).all() for x in (plain, centered, bias))
    _, inverse = np.unique(groups, return_inverse=True)
    count = np.bincount(inverse)
    eligible = (count[inverse] >= 2) & (count[inverse] <= 18)
    out = []
    for k, state in enumerate(selections):
        assert not state['outer_labels_used'] and state['centered_head_trained']
        assert state['plain_scale'] > 0 and state['centered_scale'] > 0
        means = np.zeros((len(count), 18))
        np.add.at(means, inverse, centered[k])
        means /= count[:, None]
        relative = centered[k] - means[inverse] + bias[k]
        a = log_softmax(plain[k] * state['plain_scale'])
        b = log_softmax(relative * state['centered_scale'])
        weight = float(state['selected_centered_weight'])
        assert 0 <= weight <= 0.5
        fraction = np.where(eligible & use_centered, weight, 0.0)[:, None]
        out.append(log_softmax((1 - fraction) * a + fraction * b))
    return np.stack(out)


def windows(tokenizer, texts):
    prefix = tokenizer(INSTRUCTION, add_special_tokens=False)['input_ids']
    assert tokenizer('', add_special_tokens=True)['input_ids'] == [TERMINAL], 'unexpected end-of-text handling'
    capacity = TOKEN_LIMIT - len(prefix) - 1
    first_capacity = TOKEN_LIMIT - 1
    full = tokenizer([INSTRUCTION + str(t) for t in texts], truncation=False, add_special_tokens=False)['input_ids']
    seqs, owners, weights = [], [], []
    for row, tokens in enumerate(full):
        assert len(tokens) > 0
        seqs.append(tokens[:first_capacity] + [TERMINAL])
        owners.append(row)
        weights.append(max(1, min(len(tokens), first_capacity) - len(prefix)))
        pos = first_capacity
        while pos < len(tokens):
            chunk = tokens[pos:pos + capacity]
            seqs.append(prefix + chunk + [TERMINAL])
            owners.append(row)
            weights.append(len(chunk))
            pos += len(chunk)
    return seqs, np.asarray(owners), np.asarray(weights, dtype=np.float64)


class NativeQ8:
    def __init__(self, base, assets, work, log=print, gpus=None):
        import torch
        from torch import nn
        from transformers import AutoModel, AutoTokenizer
        self.torch, self.nn, self.log = torch, nn, log
        torch.set_num_threads(2)
        for key in ('allow_tf32', 'allow_fp16_reduced_precision_reduction', 'allow_bf16_reduced_precision_reduction'):
            setattr(torch.backends.cuda.matmul, key, False)
        torch.backends.cudnn.allow_tf32 = False
        self.assets = Path(assets)
        self.labels = json.loads((self.assets / 'manifest.json').read_text())['classes']
        origin = Path(base)
        index = json.loads((self.assets / 'base' / 'model.safetensors.index.json').read_text())
        shards = sorted(set(index['weight_map'].values()))
        small = ['config.json', 'model.safetensors.index.json', 'tokenizer.json', 'tokenizer_config.json', 'merges.txt', 'vocab.json']
        self.staged = Path(work) / 'native-q8-base'
        self.staged.mkdir(parents=True, exist_ok=True)
        for name in small:
            src = self.assets / 'base' / name
            assert src.is_file(), str(src)
            dst = self.staged / name
            if not dst.exists():
                shutil.copyfile(src, dst)
        for name in shards:
            src, dst = origin / name, self.staged / name
            assert src.is_file(), str(src)
            if not dst.exists():
                try:
                    os.symlink(src, dst)
                except OSError:
                    shutil.copyfile(src, dst)
        for k in range(5):
            assert (self.assets / f'fold{k}' / 'trainables.safetensors').is_file(), k
        cfg = json.loads((self.staged / 'config.json').read_text())
        self.layers, self.hidden = cfg['num_hidden_layers'], cfg['hidden_size']
        assert self.layers == 36
        gpus = gpus if gpus is not None else torch.cuda.device_count()
        assert gpus >= 1
        self.head_device = 'cuda:0'
        place = {'embed_tokens': 0, 'norm': gpus - 1, 'rotary_emb': gpus - 1}
        cut = 16 if gpus >= 2 else self.layers
        place.update({f'layers.{i}': 0 if i < cut else 1 for i in range(self.layers)})
        self.tokenizer = AutoTokenizer.from_pretrained(self.staged, local_files_only=True)
        self.tokenizer.padding_side = 'right'
        if self.tokenizer.pad_token_id is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        loaded = AutoModel.from_pretrained(self.staged, local_files_only=True, dtype=torch.float16, device_map=place,
                                           attn_implementation='sdpa', output_loading_info=True)
        if isinstance(loaded, tuple):
            self.model, info = loaded
            assert not any(info.get(k) for k in ('missing_keys', 'mismatched_keys', 'error_msgs', 'unexpected_keys')), info
        else:
            self.model = loaded
        self.model.config.use_cache = False
        self.loras = {}
        for name, mod in list(self.model.named_modules()):
            if isinstance(mod, nn.Linear) and name.rsplit('.', 1)[-1] in SUFFIXES:
                wrapped = self._lora(mod)
                parent, attr = name.rsplit('.', 1)
                setattr(self.model.get_submodule(parent), attr, wrapped)
                self.loras[name] = wrapped
        assert len(self.loras) == self.layers * len(SUFFIXES), len(self.loras)
        dev = self.head_device
        self.norms = nn.ModuleList([nn.LayerNorm(self.hidden, device=dev, dtype=torch.float32) for _ in DEPTHS])
        self.plain = nn.Linear(self.hidden * len(DEPTHS), 18, device=dev, dtype=torch.float32)
        self.centered = nn.Linear(self.hidden * len(DEPTHS), 18, device=dev, dtype=torch.float32)
        self.model.requires_grad_(False).eval()
        for m in (self.norms, self.plain, self.centered):
            m.requires_grad_(False).eval()
        self.expected = {}
        for name, mod in self.loras.items():
            self.expected[f'encoder.backbone.base_model.model.{name}.lora_A.default.weight'] = mod.a.weight
            self.expected[f'encoder.backbone.base_model.model.{name}.lora_B.default.weight'] = mod.b.weight
        for prefix, m in (('norms', self.norms), ('plain', self.plain), ('centered', self.centered)):
            self.expected.update({f'{prefix}.{n}': p for n, p in m.named_parameters()})
        assert len(self.expected) == 2 * len(self.loras) + 2 * len(DEPTHS) + 4
        self.current = None

    def _lora(self, base):
        torch, nn = self.torch, self.nn

        class TaskLoRA(nn.Module):
            def __init__(self, layer):
                super().__init__()
                self.base_layer = layer
                dev = layer.weight.device
                self.a = nn.Linear(layer.in_features, RANK, bias=False, device=dev, dtype=torch.float32)
                self.b = nn.Linear(RANK, layer.out_features, bias=False, device=dev, dtype=torch.float32)

            def forward(self, x):
                result = self.base_layer(x)
                return (result + self.b(self.a(x.float())) * SCALE).to(result.dtype)

        return TaskLoRA(base)

    def load_fold(self, fold):
        if self.current == fold:
            return
        from safetensors.torch import load_file
        torch = self.torch
        state = load_file(str(self.assets / f'fold{fold}' / 'trainables.safetensors'))
        assert set(state) == set(self.expected), (sorted(set(state) - set(self.expected))[:3], sorted(set(self.expected) - set(state))[:3])
        with torch.inference_mode():
            for name, param in self.expected.items():
                assert state[name].shape == param.shape and state[name].dtype == torch.float32, name
                assert torch.isfinite(state[name]).all(), name
                param.copy_(state[name].to(param.device))
        del state
        gc.collect()
        self.current = fold

    def _features(self, seqs, idx):
        torch = self.torch
        batch = self.tokenizer.pad({'input_ids': [seqs[i] for i in idx]}, padding=True, return_tensors='pt').to('cuda:0')
        out = self.model(**batch, output_hidden_states=True, use_cache=False, return_dict=True)
        assert len(out.hidden_states) == self.layers + 1
        pos = batch['attention_mask'].sum(1) - 1
        rows = torch.arange(len(idx))
        parts = []
        for depth, norm in zip(DEPTHS, self.norms):
            hs = out.last_hidden_state if depth == self.layers else out.hidden_states[depth]
            pooled = hs[rows.to(hs.device), pos.to(hs.device)].float()
            if depth == self.layers:
                pooled = self.nn.functional.normalize(pooled, dim=-1)
            parts.append(norm(pooled.to(self.head_device)))
        return torch.cat(parts, dim=-1)

    def predict(self, texts, folds=range(5), batch_size=4, deadline=None, on_fold=None):
        torch = self.torch
        seqs, owners, weights = windows(self.tokenizer, texts)
        order = sorted(range(len(seqs)), key=lambda i: (len(seqs[i]), seqs[i]))
        n = len(texts)
        res = {'plain': [], 'centered': [], 'centered_bias': [], 'folds': []}
        finite = np.ones(n, dtype=bool)
        for fold in folds:
            if deadline is not None and time.time() > deadline:
                self.log(f'native: deadline reached before fold {fold}')
                break
            self.load_fold(fold)
            t0 = time.time()
            vals = {'plain': np.zeros((n, 18)), 'centered': np.zeros((n, 18))}
            with torch.inference_mode():
                for s in range(0, len(order), batch_size):
                    idx = order[s:s + batch_size]
                    try:
                        feats = self._features(seqs, idx)
                    except torch.cuda.OutOfMemoryError:
                        torch.cuda.empty_cache()
                        feats = torch.cat([self._features(seqs, [i]) for i in idx], 0)
                    ok = torch.isfinite(feats).all(1).cpu().numpy()
                    if not ok.all():
                        for i, good in zip(idx, ok):
                            if not good:
                                finite[owners[i]] = False
                        feats = torch.nan_to_num(feats)
                    own = owners[idx]
                    for name, head in (('plain', self.plain), ('centered', self.centered)):
                        sc = head(feats).float().cpu().numpy().astype(np.float64)
                        np.add.at(vals[name], own, sc * weights[idx][:, None])
                    del feats
            denom = np.bincount(owners, weights=weights, minlength=n)[:, None]
            for name in ('plain', 'centered'):
                res[name].append(vals[name] / denom)
            res['centered_bias'].append(self.centered.bias.detach().cpu().numpy().astype(np.float64).copy())
            res['folds'].append(int(fold))
            self.log(f'native: fold {fold} done, {n} rows, {len(seqs)} windows, {time.time() - t0:.0f}s, non-finite rows {int((~finite).sum())}')
            if on_fold is not None:
                on_fold({k: np.stack(v) if k != 'folds' else np.asarray(v) for k, v in res.items()}, finite)
        return {k: np.stack(v) if k != 'folds' else np.asarray(v) for k, v in res.items()}, finite


def save(path, ids, labels, res, finite):
    tmp = str(path) + '.tmp.npz'
    np.savez(tmp, ids=np.asarray(ids, dtype=str), labels=np.asarray(labels), finite=finite, **res)
    os.replace(tmp, path)


def main():
    p = argparse.ArgumentParser()
    p.add_argument('--base', required=True)
    p.add_argument('--assets', required=True)
    p.add_argument('--query', required=True)
    p.add_argument('--out', required=True)
    p.add_argument('--work', required=True)
    p.add_argument('--batch', type=int, default=4)
    p.add_argument('--deadline', type=float, default=0.0)
    p.add_argument('--limit', type=int, default=0)
    a = p.parse_args()
    T0 = time.time()

    def log(*x):
        print(f'[{time.time() - T0:7.0f}s]', *x, flush=True)

    frame = pd.read_csv(a.query, dtype={'id': str})
    assert frame.id.is_unique and len(frame)
    if a.limit:
        frame = frame.iloc[:a.limit]
    texts = frame.text.fillna('').astype(str).tolist()
    model = NativeQ8(a.base, a.assets, a.work, log=log)
    log('native: backbone and heads built')
    res, finite = model.predict(texts, batch_size=a.batch, deadline=a.deadline or None,
                                on_fold=lambda r, f: save(a.out, frame.id.values, model.labels, r, f))
    assert len(res['folds']), 'no fold finished'
    save(a.out, frame.id.values, model.labels, res, finite)
    log('native: saved', a.out, 'folds', res['folds'].tolist())


if __name__ == '__main__':
    main()

In [ ]:
%%writefile ntl660_readout.py
"""Pre-Fitted CPU Spectral & Distortion Style Readout Engine.

Evaluates high-dimensional character/word N-gram TF-IDF classifiers, LinearSVC
margins, and complementary family gates from compressed prefit caches for fast
lexico-stylometric ensemble voting."""
import json
import re
from pathlib import Path

import joblib
import numpy as np
from scipy.sparse import csr_matrix, diags, hstack

LABELS = sorted(['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra',
                 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash',
                 'claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5',
                 'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3'])
FUNCTION_WORDS = frozenset('a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little'.split())
VIEWS = [('char', 3, 5, 250000, 1.), ('word', 1, 3, 180000, .7),
         ('shape', 2, 6, 60000, .5), ('function', 1, 3, 50000, .5)]


def shape_view(text):
    return re.sub(r'[^\W\d_]+', lambda m: 'X' if m[0].isupper() else 'x', text)


def function_view(text):
    return ' '.join(t.lower() if t.lower() in FUNCTION_WORDS or not t.isalnum() else 'CONTENT'
                    for t in re.findall(r'\w+|[^\w\s]', text))


def prepare(texts, view, keep=()):
    if view == 'shape':
        return [shape_view(t) for t in texts]
    if view == 'function':
        return [function_view(t) for t in texts]
    if view == 'distort':
        keep = set(keep)
        return [re.sub(r"[A-Za-z’']+", lambda m: m[0] if m[0].lower() in keep else '*', t) for t in texts]
    return texts


def log_softmax(z):
    z = z - z.max(1, keepdims=True)
    return z - np.log(np.exp(z).sum(1, keepdims=True))


def softmax(z):
    z = z - z.max(1, keepdims=True)
    z = np.exp(z)
    return z / z.sum(1, keepdims=True)


def sparse_group_centre(X, groups, lo=2, hi=18):
    X = csr_matrix(X)
    _, inv, cnt = np.unique(np.asarray(groups), return_inverse=True, return_counts=True)
    keep = np.flatnonzero((cnt >= lo) & (cnt <= hi))
    if not len(keep):
        return X.copy()
    col = np.full(len(cnt), -1)
    col[keep] = np.arange(len(keep))
    rows = np.flatnonzero(col[inv] >= 0)
    M = csr_matrix((np.ones(len(rows)), (rows, col[inv][rows])), shape=(X.shape[0], len(keep)))
    means = diags(1.0 / cnt[keep]) @ (M.T @ X)
    return (X - M @ means).tocsr()


def style_groups(texts):
    from scipy.cluster.hierarchy import fcluster, linkage
    from scipy.spatial.distance import squareform
    from sklearn.feature_extraction.text import TfidfVectorizer
    if len(texts) < 2:
        return np.ones(len(texts), dtype=int)
    try:
        X = TfidfVectorizer(stop_words='english', sublinear_tf=True, max_features=100000,
                            dtype=np.float32).fit_transform(texts)
        D = np.clip(1.0 - (X @ X.T).toarray(), 0., 1.)
    except ValueError as exc:
        if 'empty vocabulary' not in str(exc):
            raise
        D = np.ones((len(texts), len(texts)), dtype=np.float32) - np.eye(len(texts), dtype=np.float32)
    med = max(float(np.median(D[np.triu_indices(len(texts), 1)])), 1e-6)
    D = np.asarray(D / med, dtype=np.float64).copy()
    np.fill_diagonal(D, 0.)
    return fcluster(linkage(squareform(D, checks=False), method='average'), .90, criterion='distance')


class Readout:
    def __init__(self, folder):
        self.folder = Path(folder)
        self.meta = json.loads((self.folder / 'metadata.json').read_text())
        if self.meta['labels'] != LABELS:
            raise ValueError('Unexpected canonical class order')
        self.vectorizers = [joblib.load(self.folder / e['file']) for e in self.meta['views']]
        from sklearn.feature_extraction.text import TfidfVectorizer
        assert all(type(v) is TfidfVectorizer and not callable(v.analyzer) and
                   v.preprocessor is None and v.tokenizer is None for v in self.vectorizers)

    def matrix(self, texts):
        return hstack([v.transform(prepare(texts, e['view'], self.meta.get('keep', []))) * e['weight']
                       for v, e in zip(self.vectorizers, self.meta['views'])], format='csr')

    def raw(self, X, state='model'):
        with np.load(self.folder / (state + '.npz'), allow_pickle=False) as z:
            scores = np.asarray(X @ z['coef'].T) + z['intercept']
            classes = z['classes'].copy()
        if len(classes) == 2 and scores.shape[1] == 1:
            scores = np.column_stack([-scores[:, 0], scores[:, 0]])
        return scores, classes

    def scores(self, texts, state='model'):
        return self.raw(self.matrix(texts), state)


def core_votes(root, texts):
    votes = {}
    for key, weight in [('svc4', 12.0), ('tfidf', .25), ('distort', .10)]:
        r = Readout(Path(root) / 'core' / key)
        scores, classes = r.scores(texts)
        assert classes.tolist() == list(range(len(LABELS)))
        votes[key] = log_softmax(scores) * weight
    return votes


def complementary_style(root, texts):
    root = Path(root)
    r = Readout(root / 'comp' / 'stylegram')
    X = r.matrix(texts)
    d, classes = r.raw(X, 'plain')
    assert classes.tolist() == list(range(len(LABELS)))
    P = softmax(d)
    groups = style_groups(texts)
    sizes = np.bincount(groups)[groups]
    rows = (sizes >= 2) & (sizes <= 18)
    if rows.any():
        d, _ = r.raw(sparse_group_centre(X[rows], groups[rows]), 'twin')
        P[rows] = softmax(d)
    gates = {}
    for family in ['claude', 'gpt', 'gemini', 'glm']:
        g = Readout(root / 'comp' / 'gates' / family)
        d, classes = g.scores(texts)
        expected = [i for i, label in enumerate(LABELS) if label.startswith(family)]
        assert classes.tolist() == expected
        gates[family] = d
    return P, gates, groups


In [ ]:
%%writefile gsq_infer_model.py
"""Quantized-Adapter Qwen3.8-27B Model Loader and Layer-Split Architecture.

Constructs memory-efficient 4-bit/FP16 Qwen3.8-27B backbones across dual GPUs,
attaching Delta-Turing attention blocks and learned layer-mixing projections."""
import json
import math
import struct
import time
from pathlib import Path
import numpy as np
import torch
from torch import nn
from accelerate.utils import set_module_tensor_to_device
from transformers import AutoTokenizer
from transformers.models.qwen3_5.configuration_qwen3_5 import Qwen3_5TextConfig
from transformers.models.qwen3_5 import modeling_qwen3_5 as qm
from gsq_infer_layers import GSQLinear,DenseFrozenLinear
from delta_turing import triangular_chunk_gated_delta_rule

REVISION="b5ce0b76f60020a875dee4f6ec9d934cca4121e4"


class TensorReader:
    def __init__(self,base):
        self.base=Path(base)
        self.index=json.loads((self.base/"model.safetensors.index.json").read_text())["weight_map"]
        self.headers,self.maps={},{}
        for shard in sorted(set(self.index.values())):
            path=self.base/shard
            assert path.is_file(),f"Missing exact GSQ shard: {path}"
            with path.open("rb") as stream:
                size=struct.unpack("<Q",stream.read(8))[0];assert size<2**21
                self.headers[shard]=(8+size,json.loads(stream.read(size)))
            self.maps[shard]=np.memmap(path,dtype=np.uint8,mode="r")
        missing=[key for key,shard in self.index.items() if key not in self.headers[shard][1]]
        assert all(key.startswith("mtp.") for key in missing),missing
        self.stale_mtp_keys=missing

    def has(self,key):
        return key in self.index and key in self.headers[self.index[key]][1]

    def array(self,key):
        assert self.has(key),f"Required actual GSQ tensor is missing: {key}"
        shard=self.index[key];offset,header=self.headers[shard];item=header[key]
        lo,hi=item["data_offsets"]
        dtype={"I32":"<i4","I64":"<i8","BF16":"<u2","F16":"<f2","F32":"<f4"}[item["dtype"]]
        value=self.maps[shard][offset+lo:offset+hi].view(dtype).reshape(item["shape"])
        return value,item["dtype"]

    def tensor(self,key,device="cuda",bf16_dtype=None):
        value,dtype=self.array(key)
        if dtype=="BF16":
            assert bf16_dtype in (torch.float16,torch.float32)
            fp32=(value.astype(np.uint32)<<16).view(np.float32)
            result=torch.from_numpy(fp32).to(bf16_dtype)
        else:
            result=torch.from_numpy(value.copy())
        return result.to(device)


class QuantizedCPUEmbedding(nn.Module):
    def __init__(self,reader):
        super().__init__()
        prefix="model.language_model.embed_tokens."
        self.packed,dtype=reader.array(prefix+"weight_packed");assert dtype=="I32"
        self.scale_bits,dtype=reader.array(prefix+"weight_scale");assert dtype=="BF16"
        assert self.packed.shape==(248320,640) and self.scale_bits.shape==(248320,80)
        self.cpu_packed_bytes=self.packed.nbytes+self.scale_bits.nbytes
        self.last_lookup={}

    @torch.no_grad()
    def forward(self,ids):
        began=time.perf_counter()
        assert ids.device.type=="cpu"
        flat=ids.numpy().reshape(-1)
        unique,inverse=np.unique(flat,return_inverse=True)
        words=self.packed[unique].astype(np.uint32,copy=False)
        shifts=np.arange(8,dtype=np.uint32)*4
        q=((words[:,:,None]>>shifts)&15).reshape(len(unique),5120).astype(np.int16)-8
        scales=(self.scale_bits[unique].astype(np.uint32)<<16).view(np.float32)
        values=(q.reshape(len(unique),80,64).astype(np.float32)*scales[:,:,None]).reshape(len(unique),5120)
        decoded=torch.from_numpy(values).to(torch.float16)
        selected=decoded.index_select(0,torch.from_numpy(inverse)).reshape(*ids.shape,5120)
        result=selected.to("cuda",non_blocking=False)
        self.last_lookup=dict(unique_rows=len(unique),requested_rows=len(flat),
            transferred_bytes=selected.numel()*selected.element_size(),seconds=time.perf_counter()-began)
        return result


class GSQClassifier(nn.Module):
    def __init__(self,base,*,pad_multiple=8):
        super().__init__();began=time.perf_counter()
        assert torch.cuda.device_count()==1
        self.pad_multiple=pad_multiple
        self.depths=[8,16,24,32,40,48,56,64]
        config=json.loads((Path(base)/"config.json").read_text())
        quant=config["quantization_config"]
        assert quant["quant_method"]=="compressed-tensors" and quant["format"]=="pack-quantized"
        assert quant["config_groups"]["group_0"]["weights"]["num_bits"]==3
        assert quant["config_groups"]["group_0"]["weights"]["group_size"]==128
        cfg=Qwen3_5TextConfig(**config["text_config"])
        assert cfg.num_hidden_layers==64 and cfg.hidden_size==5120
        cfg._attn_implementation="sdpa";cfg.use_cache=False
        self.cfg=cfg
        delta=triangular_chunk_gated_delta_rule
        qm.FusedRMSNormGated=None
        with torch.device("meta"):self.model=qm.Qwen3_5TextModel(cfg)
        self.model.norm=nn.Identity()
        self.reader=TensorReader(base)
        prefix="model.language_model."
        packed_count=dense_count=packed_gpu_bytes=0
        for name,module in list(self.model.named_modules()):
            if not isinstance(module,nn.Linear):continue
            assert name.startswith("layers.") and module.bias is None
            layer=int(name.split(".")[1]);rank=16 if layer>=56 else 0
            key=prefix+name+"."
            if self.reader.has(key+"weight_packed"):
                shape,_=self.reader.array(key+"weight_shape")
                assert tuple(shape)==(module.out_features,module.in_features)
                packed=self.reader.tensor(key+"weight_packed")
                scales=self.reader.tensor(key+"weight_scale",bf16_dtype=torch.float32)
                replacement=GSQLinear(packed,scales,module.in_features,module.out_features,rank)
                packed_count+=1;packed_gpu_bytes+=packed.numel()*packed.element_size()+scales.numel()*scales.element_size()
            else:
                assert ".linear_attn.in_proj_a" in name or ".linear_attn.in_proj_b" in name,name
                replacement=DenseFrozenLinear(self.reader.tensor(key+"weight",bf16_dtype=torch.float16),module.in_features,module.out_features,rank)
                dense_count+=1
            parent,_,attribute=name.rpartition(".");setattr(self.model.get_submodule(parent),attribute,replacement)
        self.model.embed_tokens=QuantizedCPUEmbedding(self.reader)
        self.model.rotary_emb=qm.Qwen3_5TextRotaryEmbedding(cfg,device="cuda")
        for name,parameter in list(self.model.named_parameters()):
            if ".lora_A" in name or ".lora_B" in name:continue
            value=self.reader.tensor(prefix+name,bf16_dtype=torch.float16)
            set_module_tensor_to_device(self.model,name,"cuda",value=value,dtype=value.dtype)
        for name,parameter in self.model.named_parameters():parameter.requires_grad_(".lora_A" in name or ".lora_B" in name)
        assert not any(value.is_meta or value.device.type!="cuda" for value in self.model.parameters())
        assert not any(value.is_meta for value in self.model.buffers())
        for layer in self.model.layers:
            if hasattr(layer,"linear_attn"):
                layer.linear_attn.chunk_gated_delta_rule=delta
                layer.linear_attn.causal_conv1d_fn=None
        assert all(not p.requires_grad for layer in self.model.layers[:56] for p in layer.parameters())
        self.norms=nn.ModuleList([nn.LayerNorm(5120) for _ in self.depths]).to("cuda")
        self.plain=nn.Linear(40960,18).to("cuda");self.centered=nn.Linear(40960,18).to("cuda")
        self.tokenizer=AutoTokenizer.from_pretrained(base,local_files_only=True)
        self.tokenizer.pad_token_id=248044;self.tokenizer.padding_side="right"
        self.load_report=dict(revision=REVISION,seconds=time.perf_counter()-began,packed_linears=packed_count,
            ignored_original_BF16_linears=dense_count,packed_gpu_bytes=packed_gpu_bytes,
            CPU_packed_embedding_bytes=self.model.embed_tokens.cpu_packed_bytes,
            trainable_parameters=sum(p.numel() for p in self.parameters() if p.requires_grad),
            delta_backend=delta.__module__,causal_convolution="native PyTorch",pad_multiple=pad_multiple,
            all_decoder_layers=64,LoRA_target_layers=list(range(56,64)),
            dtype="FP16 compute/FP32 GSQ scales+LoRA+norms+heads; BF16 decoded on CPU",
            stale_unused_MTP_index_entries=len(self.reader.stale_mtp_keys))

    def forward(self,token_rows):
        assert all(ids and ids[-1]==248044 for ids in token_rows)
        batch=self.tokenizer.pad({"input_ids":token_rows},padding=True,pad_to_multiple_of=self.pad_multiple,return_tensors="pt")
        assert batch.attention_mask.sum(1).tolist()==[len(ids) for ids in token_rows]
        assert batch.input_ids.shape[1]<=self.cfg.max_position_embeddings
        raw=[]
        with torch.no_grad():
            hidden=self.model.embed_tokens(batch.input_ids)
            attention=batch.attention_mask.to("cuda")
            pos=torch.arange(hidden.shape[1],device="cuda")[None].expand(hidden.shape[0],-1)
            kwargs=dict(config=self.cfg,inputs_embeds=hidden,attention_mask=attention,past_key_values=None,position_ids=pos)
            masks={"full_attention":qm.create_causal_mask(**kwargs),"linear_attention":qm.create_recurrent_attention_mask(**kwargs)}
            rope=self.model.rotary_emb(hidden,pos[None].expand(3,-1,-1))
            mask=attention.unsqueeze(-1).float();denominator=mask.sum(1)
            for index in range(56):
                with torch.autocast("cuda",dtype=torch.float16):
                    hidden=self.model.layers[index](hidden,position_embeddings=rope,
                        attention_mask=masks[self.cfg.layer_types[index]],position_ids=pos,past_key_values=None,use_cache=False)
                if index+1 in self.depths:raw.append((hidden.float()*mask).sum(1)/denominator)
        assert len(raw)==7 and not hidden.requires_grad
        for index in range(56,64):
            layer=self.model.layers[index]
            def apply(value,block=layer,rotary=rope,position=pos,attention_mask=masks[self.cfg.layer_types[index]]):
                with torch.autocast("cuda",dtype=torch.float16):
                    return block(value,position_embeddings=rotary,attention_mask=attention_mask,
                                 position_ids=position,past_key_values=None,use_cache=False)
            hidden=apply(hidden)
        raw.append((hidden.float()*mask).sum(1)/denominator)
        features=torch.cat([norm(value) for norm,value in zip(self.norms,raw)],-1)
        assert features.shape==(len(token_rows),40960) and torch.isfinite(features).all()
        return features


In [ ]:
%%writefile gsq_infer_layers.py
"""Custom Linear, Normalization & Attention Layer Primitives for GSQ-27B.

Implements low-memory forward hooks, RMS normalization, and grouped-query
projections tailored for dual-T4 execution of the 27B GSQ reader."""
import math
import torch
import triton
import triton.language as tl
from torch import nn
from torch.nn import functional as F


@triton.jit
def unpack3_kernel(P,S,W,N:tl.constexpr,K:tl.constexpr,PACKED_COLS:tl.constexpr,GROUPS:tl.constexpr,BLOCK:tl.constexpr):
    element=tl.program_id(0)*BLOCK+tl.arange(0,BLOCK)
    valid=element<N*K
    row=element//K;col=element%K
    bit=col*3;word=bit//32;shift=bit%32
    low=tl.load(P+row*PACKED_COLS+word,mask=valid,other=0).to(tl.uint32)
    crosses=shift>29
    high=tl.load(P+row*PACKED_COLS+word+1,mask=valid&crosses,other=0).to(tl.uint32)
    unsigned=((low>>shift)|(high<<((32-shift)&31)))&7
    signed=unsigned.to(tl.int32)-4
    scale=tl.load(S+row*GROUPS+col//128,mask=valid,other=0).to(tl.float32)
    tl.store(W+element,signed.to(tl.float32)*scale,mask=valid)


def dequantize(packed,scales):
    n,groups=scales.shape;k=groups*128
    output=torch.empty((n,k),device=packed.device,dtype=torch.float16)
    unpack3_kernel[(triton.cdiv(n*k,512),)](packed,scales,output,n,k,packed.shape[1],groups,512)
    return output


class GSQLinear(nn.Module):
    def __init__(self,packed,scales,in_features,out_features,rank):
        super().__init__()
        assert packed.dtype==torch.int32 and scales.dtype==torch.float32
        assert tuple(packed.shape)==(out_features,in_features*3//32)
        assert tuple(scales.shape)==(out_features,in_features//128)
        self.register_buffer("packed",packed,persistent=False)
        self.register_buffer("scales",scales,persistent=False)
        self.in_features,self.out_features=in_features,out_features
        self.bias=None
        initialize_lora(self,rank)
    def forward(self,value):
        base=F.linear(value,dequantize(self.packed,self.scales))
        return add_lora(self,value,base)


class DenseFrozenLinear(nn.Module):
    def __init__(self,weight,in_features,out_features,rank):
        super().__init__()
        assert tuple(weight.shape)==(out_features,in_features)
        self.register_buffer("base_weight",weight,persistent=False)
        self.in_features,self.out_features=in_features,out_features
        self.bias=None
        initialize_lora(self,rank)
    def forward(self,value):
        return add_lora(self,value,F.linear(value,self.base_weight))


def initialize_lora(module,rank):
    module.scale=2.
    module.dropout=nn.Identity()
    if rank:
        module.lora_A=nn.Parameter(torch.empty(rank,module.in_features,device="cuda",dtype=torch.float32))
        module.lora_B=nn.Parameter(torch.zeros(module.out_features,rank,device="cuda",dtype=torch.float32))
        nn.init.kaiming_uniform_(module.lora_A,a=math.sqrt(5))
    else:
        module.register_parameter("lora_A",None);module.register_parameter("lora_B",None)


def add_lora(module,value,base):
    if module.lora_A is None:return base
    with torch.autocast("cuda",dtype=torch.float16):
        extra=F.linear(F.linear(module.dropout(value),module.lora_A),module.lora_B)
    return base+module.scale*extra.to(base.dtype)


In [ ]:
%%writefile delta_turing.py
"""Delta-Turing Attention Head and Sequence Pooling Block.

Computes multi-head content-aware pooling over transformer hidden states to
capture long-range discourse cadence across 18 frontier model families."""
import torch
from torch.nn import functional as F
from transformers.models.qwen3_5.modeling_qwen3_5 import l2norm

def triangular_chunk_gated_delta_rule(
    query,
    key,
    value,
    g,
    beta,
    chunk_size=64,
    initial_state=None,
    output_final_state=False,
    use_qk_l2norm_in_kernel=False,
    **kwargs,
):
    initial_dtype = query.dtype
    if use_qk_l2norm_in_kernel:
        query = l2norm(query, dim=-1, eps=1e-6)
        key = l2norm(key, dim=-1, eps=1e-6)
    query, key, value, beta, g = [
        x.transpose(1, 2).contiguous().to(torch.float32) for x in (query, key, value, beta, g)
    ]

    batch_size, num_heads, sequence_length, k_head_dim = key.shape
    v_head_dim = value.shape[-1]
    pad_size = (chunk_size - sequence_length % chunk_size) % chunk_size
    query = F.pad(query, (0, 0, 0, pad_size))
    key = F.pad(key, (0, 0, 0, pad_size))
    value = F.pad(value, (0, 0, 0, pad_size))
    beta = F.pad(beta, (0, pad_size))
    g = F.pad(g, (0, pad_size))
    total_sequence_length = sequence_length + pad_size
    scale = 1 / (query.shape[-1] ** 0.5)
    query = query * scale

    v_beta = value * beta.unsqueeze(-1)
    k_beta = key * beta.unsqueeze(-1)
    query, key, value, k_beta, v_beta = [
        x.reshape(x.shape[0], x.shape[1], -1, chunk_size, x.shape[-1]) for x in (query, key, value, k_beta, v_beta)
    ]
    g = g.reshape(g.shape[0], g.shape[1], -1, chunk_size)
    mask = torch.triu(torch.ones(chunk_size, chunk_size, dtype=torch.bool, device=query.device), diagonal=0)

    g = g.cumsum(dim=-1)
    decay_mask = ((g.unsqueeze(-1) - g.unsqueeze(-2)).tril().exp().float()).tril()
    attn = -((k_beta @ key.transpose(-1, -2)) * decay_mask).masked_fill(mask, 0)
    with torch.autocast(device_type=query.device.type, enabled=False):
        identity = torch.eye(chunk_size, dtype=torch.float32, device=attn.device)
        coefficient = identity - attn.float()
        identity = identity.expand_as(coefficient)
        attn = torch.linalg.solve_triangular(coefficient, identity,
                                             upper=False, unitriangular=True)
    value = attn @ v_beta
    k_cumdecay = attn @ (k_beta * g.exp().unsqueeze(-1))
    last_recurrent_state = (
        torch.zeros(batch_size, num_heads, k_head_dim, v_head_dim, dtype=value.dtype, device=value.device)
        if initial_state is None
        else initial_state.to(value)
    )
    core_attn_out = torch.zeros_like(value)
    mask = torch.triu(torch.ones(chunk_size, chunk_size, dtype=torch.bool, device=query.device), diagonal=1)

    for i in range(0, total_sequence_length // chunk_size):
        q_i, k_i, v_i = query[:, :, i], key[:, :, i], value[:, :, i]
        attn = q_i @ k_i.transpose(-1, -2) * decay_mask[:, :, i]
        v_prime = (k_cumdecay[:, :, i]) @ last_recurrent_state
        v_new = v_i - v_prime
        attn_inter = (q_i * g[:, :, i, :, None].exp()) @ last_recurrent_state
        core_attn_out[:, :, i] = attn_inter + attn @ v_new
        last_recurrent_state = (
            last_recurrent_state * g[:, :, i, -1, None, None].exp()
            + (k_i * (g[:, :, i, -1, None] - g[:, :, i]).exp()[..., None]).transpose(-1, -2) @ v_new
        )

    if not output_final_state:
        last_recurrent_state = None
    core_attn_out = core_attn_out.reshape(core_attn_out.shape[0], core_attn_out.shape[1], -1, core_attn_out.shape[-1])
    core_attn_out = core_attn_out[:, :, :sequence_length]
    core_attn_out = core_attn_out.transpose(1, 2).contiguous().to(initial_dtype)
    return core_attn_out, last_recurrent_state


In [ ]:
%%writefile adapter.py
"""Lightweight PEFT/LoRA Adapter Weight Injection Utility.

Loads safetensors LoRA low-rank matrices (`A`, `B`) and merges scaled delta
updates directly into base transformer linear weights in-place."""
import json
from pathlib import Path

import torch
from safetensors import safe_open

REVISION = "b5ce0b76f60020a875dee4f6ec9d934cca4121e4"


def load_final(model, final_file, metadata_file, complete_file):
    final_file, metadata_file, complete_file = map(Path, (final_file, metadata_file, complete_file))
    for path in (final_file, metadata_file, complete_file):
        if not path.is_file():
            raise FileNotFoundError(f"Required final training artifact missing: {path}")
    metadata = json.loads(metadata_file.read_text(encoding="utf-8"))
    complete = json.loads(complete_file.read_text(encoding="utf-8"))
    assert complete["status"] == "TRAINED_FULLTEXT_GSQ3_PUBLIC119899_OFFICIAL2_EPOCHS"
    assert complete["rows_seen"] == {"external": 119899, "official": 5398}
    assert complete["final_artifact"]["path"] == final_file.name == "final.safetensors"
    assert complete["contract_hash"] == metadata["checkpoint_contract_hash"]
    assert metadata["source_revision"] == REVISION
    assert metadata["rank"] == 16 and metadata["alpha"] == 32
    assert metadata["depths"] == [8, 16, 24, 32, 40, 48, 56, 64]
    assert metadata["lora_target_layer_indices"] == list(range(56, 64))
    assert metadata["executed_decoder_layers"] == 64 and metadata["text_truncation"] is False
    assert len(metadata["labels"]) == 18
    assert final_file.stat().st_size == complete["final_artifact"]["bytes"]
    expected = {name: param for name, param in model.named_parameters() if param.requires_grad}
    export_keys = {"backbone." + name.removeprefix("model.") if name.startswith("model.") else name: name
                   for name in expected}
    with safe_open(str(final_file), framework="pt", device="cpu") as weights:
        assert set(weights.keys()) == set(export_keys), "Final adapter tensor names differ from model trainable parameters"
        with torch.no_grad():
            for key, name in export_keys.items():
                source = weights.get_tensor(key)
                target = expected[name]
                assert source.dtype == target.dtype and source.shape == target.shape and torch.isfinite(source).all(), key
                target.copy_(source.to(target.device))
    model.eval()
    return metadata


In [ ]:
%%writefile gsq_infer.py
"""End-to-End Inference Driver for Qwen3.8-27B GSQ Adapter Checkpoints.

Runs batched tokenization, dual-GPU layer streaming, and calibrated softmax
probability extraction for the full-passage Qwen3.8-27B expert."""
import argparse
import json
import os
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd


def read_rows(path, text_column):
    frame = pd.read_parquet(path).reset_index(drop=True)
    assert text_column in frame and frame[text_column].notna().all()
    texts = frame[text_column].astype(str).tolist()
    assert all(texts)
    return frame, texts


def worker(args):
    assert os.environ.get("CUDA_VISIBLE_DEVICES") in ("0", "1")
    import torch
    from adapter import load_final
    from gsq_infer_model import GSQClassifier

    assert torch.cuda.device_count() == 1, "GSQ worker must see one isolated GPU"
    assert torch.cuda.get_device_capability(0) >= (7, 5), "GSQ T4 path requires SM 7.5 or newer"
    torch.set_grad_enabled(False)
    frame, texts = read_rows(args.input, args.text_column)
    indices = np.arange(len(texts), dtype=np.int64)[args.worker_id::2]
    assert len(indices)
    model = GSQClassifier(args.base)
    metadata = load_final(model, args.final, args.metadata, args.complete)
    tokenizer = model.tokenizer
    features = np.empty((len(indices), 40960), dtype=np.float32)
    plain = np.empty((len(indices), 18), dtype=np.float32)
    for begin in range(0, len(indices), args.batch_size):
        batch_ids = indices[begin:begin + args.batch_size]
        token_rows = tokenizer([texts[int(i)] for i in batch_ids], add_special_tokens=False,
                               truncation=False)["input_ids"]
        token_rows = [ids + [248044] for ids in token_rows]
        assert all(len(ids) <= model.cfg.max_position_embeddings for ids in token_rows), "Full text exceeds model context"
        with torch.inference_mode():
            value = model(token_rows)
            score = model.plain(value)
        stop = begin + len(batch_ids)
        features[begin:stop] = value.float().cpu().numpy()
        plain[begin:stop] = score.float().cpu().numpy()
        del value, score
    args.output.parent.mkdir(parents=True, exist_ok=True)
    np.savez(args.output, row_index=indices, features=features, plain_logits=plain)
    print(json.dumps(dict(worker=args.worker_id, rows=len(indices), labels=metadata["labels"],
                          load_report=model.load_report)), flush=True)


def group_codes(args, frame):
    assert (args.groups is None) != (args.prompt_column is None), "Provide exactly one group source"
    if args.groups is not None:
        codes = np.load(args.groups, allow_pickle=False)
        assert codes.ndim == 1 and len(codes) == len(frame) and np.issubdtype(codes.dtype, np.integer)
        assert np.all(codes >= 0), "Explicit group codes must be nonnegative"
        return codes
    assert args.prompt_column in frame, "Requested prompt column is missing"
    assert frame[args.prompt_column].notna().all(), "Missing prompt group is ambiguous"
    return pd.factorize(frame[args.prompt_column], sort=False)[0]


def combine(args, outputs):
    from safetensors import safe_open

    frame, _ = read_rows(args.input, args.text_column)
    n = len(frame)
    features = np.empty((n, 40960), dtype=np.float32)
    plain = np.empty((n, 18), dtype=np.float32)
    seen = np.zeros(n, dtype=bool)
    for path in outputs:
        with np.load(path, allow_pickle=False) as item:
            ids = item["row_index"]
            assert ids.ndim == 1 and np.issubdtype(ids.dtype, np.integer)
            assert len(np.unique(ids)) == len(ids)
            assert np.all((0 <= ids) & (ids < n)) and not np.any(seen[ids])
            assert item["features"].shape == (len(ids), 40960)
            assert item["plain_logits"].shape == (len(ids), 18)
            features[ids] = item["features"]
            plain[ids] = item["plain_logits"]
            seen[ids] = True
    assert seen.all() and np.isfinite(features).all() and np.isfinite(plain).all()
    metadata = json.loads(args.metadata.read_text(encoding="utf-8"))
    with safe_open(str(args.final), framework="pt", device="cpu") as weights:
        kernel = weights.get_tensor("centered.weight").numpy().astype(np.float32)
        bias = weights.get_tensor("centered.bias").numpy().astype(np.float32)
    assert kernel.shape == (18, 40960) and bias.shape == (18,)
    groups = group_codes(args, frame)
    centered = plain.copy()
    valid_centered = np.zeros(n, dtype=bool)
    for group in np.unique(groups):
        ids = np.flatnonzero(groups == group)
        if len(ids) == 1:
            continue
        centered[ids] = (features[ids] - features[ids].mean(0, keepdims=True)) @ kernel.T + bias
        valid_centered[ids] = True
    assert np.isfinite(centered).all() and len(groups) == n
    args.output.parent.mkdir(parents=True, exist_ok=True)
    np.savez(args.output, row_index=np.arange(n, dtype=np.int64), features=features,
             plain_logits=plain, centered_logits=centered, valid_centered=valid_centered,
             group_code=groups, labels=np.asarray(metadata["labels"]))


def parent(args):
    _, texts = read_rows(args.input, args.text_column)
    worker_count = min(2, len(texts))
    outputs = [args.work_dir / f"gsq-worker-{i}.npz" for i in range(worker_count)]
    args.work_dir.mkdir(parents=True, exist_ok=True)
    children = []
    for device in range(worker_count):
        command = [sys.executable, str(Path(__file__).resolve()), "--worker-id", str(device),
                   "--base", str(args.base), "--final", str(args.final), "--metadata", str(args.metadata),
                   "--complete", str(args.complete), "--input", str(args.input), "--output", str(outputs[device]),
                   "--text-column", args.text_column,
                   "--batch-size", str(args.batch_size)]
        env = os.environ.copy()
        env.update(CUDA_VISIBLE_DEVICES=str(device), HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1",
                   TOKENIZERS_PARALLELISM="false")
        children.append(subprocess.Popen(command, env=env))
    codes = [child.wait() for child in children]
    if any(code != 0 for code in codes):
        raise RuntimeError(f"GSQ workers failed: exit codes {codes}; outputs were not combined")
    combine(args, outputs)


def main():
    p = argparse.ArgumentParser(description=__doc__)
    for name in ("base", "final", "metadata", "complete", "input", "output"):
        p.add_argument("--" + name, type=Path, required=True)
    p.add_argument("--work-dir", type=Path)
    p.add_argument("--text-column", default="text")
    p.add_argument("--prompt-column")
    p.add_argument("--groups", type=Path)
    p.add_argument("--batch-size", type=int, default=1)
    p.add_argument("--worker-id", type=int, choices=(0, 1))
    args = p.parse_args()
    assert args.batch_size >= 1
    if args.worker_id is None:
        assert args.work_dir is not None and os.environ.get("CUDA_VISIBLE_DEVICES") in (None, "", "0,1")
        assert (args.groups is None) != (args.prompt_column is None), "Provide --groups or --prompt-column"
        parent(args)
    else:
        worker(args)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile gsq_stage.py
"""Stage-1 GSQ-27B Feature Blending and Logit Export Interface.

Wraps `gsq_infer.py` execution inside the master ensemble worker and aligns
27B class probabilities onto the canonical 18-label competition schema."""
import json
import os
import shutil
import signal
import subprocess
import sys
import tempfile
import time
from contextlib import contextmanager
from pathlib import Path

import numpy as np
import pandas as pd

BASE_FILES = {'config.json': 5038, 'merges.txt': 3353259, 'model.safetensors.index.json': 198539, 'tokenizer.json': 12809320,
              'tokenizer_config.json': 17928, 'vocab.json': 6722759, 'model-00001-of-00003.safetensors': 2272369784,
              'model-00002-of-00003.safetensors': 4777704120, 'model-00003-of-00003.safetensors': 4777704376}


@contextmanager
def staged_tree(source, files, label, temp_root='/tmp'):
    source, temp_root = Path(source), Path(temp_root)
    assert source.is_dir()
    for name, size in files.items():
        item = source / name
        assert item.is_file() and item.stat().st_size == size, (label, name, size)
    total = sum(files.values())
    free = shutil.disk_usage(temp_root).free
    assert free > total + 2 * 2 ** 30, ('not enough local disk to stage', label, total, free)
    destination = Path(tempfile.mkdtemp(prefix='ntl-stage-' + label + '-', dir=temp_root))
    start = time.monotonic()
    try:
        for name, size in files.items():
            with (source / name).open('rb') as reader, (destination / name).open('wb') as writer:
                while block := reader.read(16 * 1024 * 1024):
                    writer.write(block)
            assert (destination / name).stat().st_size == size
        print(json.dumps({'stage': label, 'staged_bytes': total, 'seconds': round(time.monotonic() - start)}), flush=True)
        yield destination
    finally:
        shutil.rmtree(destination, ignore_errors=True)


def run(assets, ids, texts, groups, work, labels, log=print, timeout=None):
    assets, work = Path(assets), Path(work)
    inference = assets / 'gsq120k-trained' / 'gsq120k-inference' / 'gsq'
    final, metadata, complete = inference / 'final.safetensors', inference / 'final.json', inference / 'COMPLETE.json'
    for path in (final, metadata, complete):
        assert path.is_file(), path
    wheels = sorted((assets / 'name-that-llm-artifacts' / 'qwen38-offline-linux').glob('*.whl'))
    assert len(wheels) >= 6, wheels
    work.mkdir(parents=True, exist_ok=True)
    target = work / 'python'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--disable-pip-version-check', '--target', str(target)] + [str(w) for w in wheels],
                   check=True, capture_output=True)
    query = work / 'query.parquet'
    pd.DataFrame({'id': list(ids), 'text': list(texts)}).to_parquet(query, index=False)
    group_file = work / 'groups.npy'
    np.save(group_file, np.asarray(groups, dtype=np.int64), allow_pickle=False)
    output = work / 'gsq.npz'
    if output.exists():
        output.unlink()
    started = time.monotonic()
    with staged_tree(assets / 'gsq3-base' / 'qwen38-27b-gsq3', BASE_FILES, 'gsq3-base') as base:
        env = dict(os.environ, PYTHONPATH=str(target), CUDA_VISIBLE_DEVICES='', HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1', TOKENIZERS_PARALLELISM='false')
        command = [sys.executable, '-u', str(Path(__file__).resolve().parent / 'gsq_infer.py'), '--base', str(base), '--final', str(final), '--metadata', str(metadata),
                   '--complete', str(complete), '--input', str(query), '--groups', str(group_file), '--output', str(output), '--work-dir', str(work / 'workers'), '--batch-size', '1']
        process = subprocess.Popen(command, env=env, start_new_session=True)
        try:
            code = process.wait(timeout=timeout)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL)
            process.wait()
            raise TimeoutError(f'GSQ stage exceeded {timeout:.0f} s')
        if code != 0:
            raise RuntimeError(f'GSQ stage exited with code {code}')
    with np.load(output, allow_pickle=False) as z:
        saved = [str(x) for x in z['labels'].tolist()]
        assert sorted(saved) == sorted(labels) and len(saved) == 18
        logits = z['plain_logits'][:, [saved.index(l) for l in labels]].astype(np.float64)
        assert len(logits) == len(ids) and np.isfinite(logits).all()
    logits -= logits.max(1, keepdims=True)
    probs = np.exp(logits)
    probs /= probs.sum(1, keepdims=True)
    log(f'GSQ 27B scored {len(ids)} rows in {time.monotonic() - started:.0f} s')
    return probs

In [ ]:
%%writefile qwen_groups.py
"""Dual-Embedding Prompt-Group Similarity Graph and Label Propagation.

Identifies high-similarity prompt clusters (`group_sim >= 0.70`) via L2-normalized
Qwen3 embeddings and runs soft neighborhood affinity smoothing across shared
evaluation groups."""
import argparse
import json
import os
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import AutoModel, AutoTokenizer


def stage(base, assets, work):
    base, assets, staged = Path(base), Path(assets), Path(work) / 'qgroup-base'
    staged.mkdir(parents=True, exist_ok=True)
    index = json.loads((assets / 'base' / 'model.safetensors.index.json').read_text())
    for name in ('config.json', 'model.safetensors.index.json', 'tokenizer.json', 'tokenizer_config.json', 'merges.txt', 'vocab.json'):
        target = staged / name
        if not target.exists():
            shutil.copyfile(assets / 'base' / name, target)
    for name in sorted(set(index['weight_map'].values())):
        source, target = base / name, staged / name
        assert source.is_file(), str(source)
        if not target.exists():
            try:
                os.symlink(source, target)
            except OSError:
                shutil.copyfile(source, target)
    return staged


def main():
    p = argparse.ArgumentParser()
    for key in ('base', 'assets', 'query', 'out', 'work'):
        p.add_argument('--' + key, required=True)
    p.add_argument('--max_len', type=int, default=512)
    p.add_argument('--tok_budget', type=int, default=8192)
    p.add_argument('--mem_frac', type=float, default=0.88)
    a = p.parse_args()
    started = time.time()
    frame = pd.read_csv(a.query, dtype={'id': str})
    texts = frame.text.fillna('').astype(str).tolist()
    staged = stage(a.base, a.assets, a.work)
    tok = AutoTokenizer.from_pretrained(staged, local_files_only=True)
    auto, plain = tok('hello')['input_ids'], tok('hello', add_special_tokens=False)['input_ids']
    eos = auto[-1] if (len(auto) > len(plain) and auto[-1] != plain[-1]) else (tok.eos_token_id if tok.eos_token_id is not None else 151643)
    pad = tok.pad_token_id if tok.pad_token_id is not None else eos
    gpus = torch.cuda.device_count()
    assert gpus >= 1
    max_memory = {i: f'{int(torch.cuda.get_device_properties(i).total_memory * a.mem_frac / 2 ** 30)}GiB' for i in range(gpus)}
    model = AutoModel.from_pretrained(staged, local_files_only=True, dtype=torch.float16, device_map='auto', max_memory=max_memory).eval()
    first = next(model.parameters()).device
    ids = []
    for t in texts:
        x = tok(t, truncation=True, max_length=a.max_len - 1)['input_ids']
        ids.append(x if x[-1] == eos else x + [eos])
    order = np.argsort([len(x) for x in ids], kind='stable')
    emb = np.zeros((len(texts), model.config.get_text_config().hidden_size), np.float16)
    budget, done, ooms = a.tok_budget, 0, 0
    with torch.inference_mode():
        while done < len(order):
            end = done + 1
            while end < len(order) and (end - done + 1) * len(ids[order[end]]) <= budget:
                end += 1
            idx = order[done:end]
            try:
                width = max(len(ids[i]) for i in idx)
                inp = torch.full((len(idx), width), pad)
                att = torch.zeros((len(idx), width), dtype=torch.long)
                for r, i in enumerate(idx):
                    inp[r, :len(ids[i])] = torch.tensor(ids[i])
                    att[r, :len(ids[i])] = 1
                hidden = model(input_ids=inp.to(first), attention_mask=att.to(first), use_cache=False).last_hidden_state
                vec = hidden[torch.arange(len(idx), device=hidden.device), (att.sum(1) - 1).to(hidden.device)].float()
                emb[idx] = torch.nn.functional.normalize(vec, dim=-1).cpu().numpy().astype(np.float16)
                done = end
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                ooms += 1
                budget = max(len(ids[order[done]]), budget // 2)
                print(f'qwen_groups: OOM, token budget -> {budget}', flush=True)
    assert np.isfinite(emb).all()
    os.makedirs(os.path.dirname(os.path.abspath(a.out)), exist_ok=True)
    np.save(a.out, emb)
    print(f'qwen_groups: {len(texts)} rows, dim {emb.shape[1]}, {ooms} OOM retries, {time.time() - started:.0f} s', flush=True)


if __name__ == '__main__':
    main()


In [ ]:
%%writefile infer.py
"""Master Stage-1 Multi-Backbone Ensemble & Sinkhorn-Knopp Balancer.

Fuses 5 L24 LoRA backbones (`q14v5`, `q8v5`, `ll8v5`, `n14b_clean`, `q25_14`),
DeBERTa-v3-Large, 5-fold Native Q8 embeddings, Bayesian Gaussian heads, and
GSQ-27B probabilities, followed by prior-damped Sinkhorn-Knopp scaling."""
import os, sys, time, gc, argparse, json, re
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import numpy as np, pandas as pd, scipy.sparse as sp, torch
from pipe import pipelined, spans_devices

LABELS = ['claude-fable-5','claude-opus-4-6','claude-opus-4-8','claude-opus-5','claude-sonnet-5',
 'gemini-3.1-pro-preview','gemini-3.6-flash','gemini-3.7-flash','gemini-3.8-flash','glm-5.3','glm-5.3-flash',
 'gpt-5.5','gpt-5.6-luna','gpt-5.6-sol','gpt-5.6-terra','gpt-6-astra','kimi-k3','qwen3.8-max']
L2I = {l: i for i, l in enumerate(LABELS)}

p = argparse.ArgumentParser()
p.add_argument('--data', required=True)
p.add_argument('--models', required=True)
p.add_argument('--out', default='submission.parquet')
p.add_argument('--save_final_logits', default='')
p.add_argument('--comp_path', default='')
p.add_argument('--comp_w', type=float, default=0.0)
p.add_argument('--tfidf_w', type=float, default=1.0)
p.add_argument('--distort_w', type=float, default=0.0)
p.add_argument('--adapt_thr', type=float, default=0.15)
p.add_argument('--adapt_w', type=float, default=0.0)
p.add_argument('--extra', default='')
p.add_argument('--extra_w', type=float, default=0.5)
p.add_argument('--svc_w', type=float, default=0.0)
p.add_argument('--pool', default='')
p.add_argument('--sk_div', type=int, default=1)
p.add_argument('--sinkhorn', type=float, default=0.0)
p.add_argument('--sinkhorn_target_mix', type=float, default=0.35)
p.add_argument('--atlas_dir', default='')
p.add_argument('--atlas_base', default='')
p.add_argument('--atlas_mode', default='off', choices=['off', 'head', 'lite', 'mix'])
p.add_argument('--atlas_member_w', type=float, default=0.0)
p.add_argument('--atlas_k', type=int, default=6)
p.add_argument('--atlas_w', type=float, default=0.0)
p.add_argument('--atlas_margin', type=float, default=0.0)
p.add_argument('--atlas_bs', type=int, default=4)
p.add_argument('--atlas_chunk', type=int, default=3)
p.add_argument('--atlas_workers', type=int, default=2)
p.add_argument('--atlas_deadline', type=float, default=0.0)
p.add_argument('--save_lp', default='')
p.add_argument('--native_dir', default='')
p.add_argument('--native_base', default='')
p.add_argument('--native_share', type=float, default=0.0)
p.add_argument('--native_cent', type=int, default=1)
p.add_argument('--native_deadline', type=float, default=0.0)
p.add_argument('--optional_deadline', type=float, default=0.0)
p.add_argument('--native_work', default='/kaggle/temp/native')
p.add_argument('--native_min_agree', type=float, default=0.35)
p.add_argument('--prefit_core', default='')
p.add_argument('--gsq_dir', default='')
p.add_argument('--gsq_share', type=float, default=0.0)
p.add_argument('--gsq_deadline', type=float, default=0.0)
p.add_argument('--gsq_timeout', type=float, default=0.0)
p.add_argument('--gsq_work', default='/kaggle/temp/gsq')
p.add_argument('--gsq_min_agree', type=float, default=0.30)
p.add_argument('--group_embed_base', default='')
p.add_argument('--group_embed_assets', default='')
p.add_argument('--group_work', default='/kaggle/temp/qgroups')
p.add_argument('--group_sim', type=float, default=0.70)
a = p.parse_args()
torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False
T0 = time.time()
def log(*x): print(f'[{time.time()-T0:7.0f}s]', *x, flush=True)

train = pd.read_csv(os.path.join(a.data, 'train.csv'))
test = pd.read_csv(os.path.join(a.data, 'test.csv'))
train['text'] = train.text.fillna(''); test['text'] = test.text.fillna('')
y = train.model.map(L2I).values; Gtr = train.prompt_id.values
log('train', len(train), 'test', len(test))

def capped_agglo(S, cap=18, thr=0.02):
    n = len(S); A = S.astype(np.float32).copy(); np.fill_diagonal(A, -np.inf)
    size = np.ones(n, np.int32); alive = np.ones(n, bool); lab = np.arange(n)
    while True:
        ok = (size[:, None] + size[None, :] <= cap) & alive[:, None] & alive[None, :]
        M = np.where(ok, A, -np.inf); k = int(np.argmax(M)); i, j = divmod(k, n)
        if not np.isfinite(M[i, j]) or M[i, j] < thr: break
        new = (size[i] * A[i] + size[j] * A[j]) / (size[i] + size[j])
        A[i] = new; A[:, i] = new; A[i, i] = -np.inf; A[j] = -np.inf; A[:, j] = -np.inf
        size[i] += size[j]; alive[j] = False; lab[lab == j] = i
    return lab

from sklearn.feature_extraction.text import TfidfVectorizer
try:
    v = TfidfVectorizer(sublinear_tf=True, stop_words='english', min_df=2, max_df=0.05)
    Xc = v.fit_transform(test.text.values)
    Gte = capped_agglo((Xc @ Xc.T).toarray(), 18, a.adapt_thr)
except ValueError as e:
    log('clustering fallback:', e)
    Gte = np.arange(len(test))
log('test clusters', len(np.unique(Gte)), 'sizes', np.bincount(np.bincount(Gte)[np.bincount(Gte) > 0]).tolist())

if a.group_embed_base:
    try:
        import subprocess
        os.makedirs(a.group_work, exist_ok=True)
        emb_path = os.path.join(a.group_work, 'group_emb.npy')
        if os.path.exists(emb_path):
            os.remove(emb_path)
        gcmd = [sys.executable, '-u', os.path.join(os.path.dirname(os.path.abspath(__file__)), 'qwen_groups.py'), '--base', a.group_embed_base, '--assets', a.group_embed_assets,
                '--query', os.path.join(a.data, 'test.csv'), '--out', emb_path, '--work', a.group_work]
        gcode = subprocess.run(gcmd, env=dict(os.environ, HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1', TOKENIZERS_PARALLELISM='false')).returncode
        E_q = np.load(emb_path).astype(np.float32)
        assert gcode == 0 and E_q.shape[0] == len(test) and np.isfinite(E_q).all()
        G_q = capped_agglo(E_q @ E_q.T, 18, a.group_sim)
        _sz_old, _sz_new = np.bincount(Gte)[Gte], np.bincount(G_q)[G_q]
        log('qwen groups', len(np.unique(G_q)), 'clusters (tf-idf', len(np.unique(Gte)), ') rows in groups of 2..18: qwen', round(float(((_sz_new >= 2) & (_sz_new <= 18)).mean()), 3),
            'tf-idf', round(float(((_sz_old >= 2) & (_sz_old <= 18)).mean()), 3))
        assert (np.bincount(G_q).max()) <= 18
        Gte = G_q
        del E_q
    except Exception as _e:
        log('QWEN GROUPS SKIPPED, keeping the TF-IDF groups:', repr(_e))
LP = {}
EXT_T, EXT_Y = np.array([], dtype=object), np.array([], dtype=int)
EXTRA_DF = pd.DataFrame(columns=['text', 'model'])
def _norm_text_key(value):
    import unicodedata
    return re.sub(r'\s+', ' ', unicodedata.normalize('NFKC', str(value or ''))).strip()
if a.extra:
    _E = pd.read_parquet(a.extra) if a.extra.endswith('.parquet') else pd.read_csv(a.extra)
    if not {'text', 'model'}.issubset(_E.columns):
        raise ValueError(f'External file must contain text/model columns: {a.extra}')
    _E = _E[_E.model.isin(LABELS)].copy()
    _E['text'] = _E.text.fillna('').astype(str)
    _official_keys = {_norm_text_key(t) for t in train.text.values}
    _seen = set(_official_keys)
    _keep = []
    for _t in _E.text.values:
        _k = _norm_text_key(_t)
        _ok = bool(_k) and _k not in _seen
        _keep.append(_ok)
        if _ok: _seen.add(_k)
    EXTRA_DF = _E.loc[_keep, ['text', 'model']].reset_index(drop=True)
    EXT_T = EXTRA_DF.text.astype(str).values
    EXT_Y = EXTRA_DF.model.map(L2I).values.astype(int)
    log('matched-clean extra rows after exact de-dup', len(EXT_T))
FIT_T = np.concatenate([train.text.values, EXT_T])
FIT_Y = np.concatenate([y, EXT_Y])
FIT_W = np.concatenate([np.ones(len(train), dtype=np.float32), np.full(len(EXT_T), a.extra_w, dtype=np.float32)])
if a.tfidf_w > 0:
    from sklearn.linear_model import LogisticRegression
    mats_tr, mats_te = [], []
    for kw in [dict(analyzer='char_wb', ngram_range=(2, 5), min_df=3, max_features=60000),
               dict(token_pattern=r"\w+|[^\w\s]", ngram_range=(1, 2), min_df=3, max_features=30000)]:
        vec = TfidfVectorizer(sublinear_tf=True, lowercase=False, **kw).fit(FIT_T)
        mats_tr.append(vec.transform(FIT_T)); mats_te.append(vec.transform(test.text.values))
    LP['tfidf'] = LogisticRegression(C=20, max_iter=3000).fit(sp.hstack(mats_tr).tocsr(), FIT_Y, sample_weight=FIT_W).predict_log_proba(sp.hstack(mats_te).tocsr()) * a.tfidf_w
    gc.collect(); log('tfidf done')
else:
    LP['tfidf'] = np.zeros((len(test), 18), dtype=np.float32)
    log('tfidf skipped (zero ensemble weight)')
if a.svc_w > 0:
    import re as _re
    from scipy.sparse import hstack as _hs
    from sklearn.svm import LinearSVC
    _FW = set("a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little".split())
    _shape = lambda t: _re.sub(r"[^\W\d_]+", lambda m_: "X" if m_[0].isupper() else "x", t)
    _func = lambda t: " ".join(w.lower() if w.lower() in _FW or not w.isalnum() else "CONTENT" for w in _re.findall(r"\w+|[^\w\s]", t))
    _parts = []
    _official = train[['text', 'model']].copy(); _official['text'] = _official.text.fillna('').astype(str)
    _official['_srcw'] = 1.0
    _parts.append(_official)
    _seen = {_norm_text_key(t) for t in _official.text.values}
    if len(EXTRA_DF):
        _e = EXTRA_DF[['text', 'model']].copy(); _e['_srcw'] = float(a.extra_w)
        _keep = []
        for _t in _e.text.values:
            _k = _norm_text_key(_t); _ok = bool(_k) and _k not in _seen
            _keep.append(_ok)
            if _ok: _seen.add(_k)
        _e = _e.loc[_keep].copy(); _parts.append(_e)
    if a.pool and os.path.exists(a.pool):
        _P = pd.read_parquet(a.pool)
        if not {'text', 'model'}.issubset(_P.columns):
            raise ValueError(f'Pool must contain text/model columns: {a.pool}')
        _P = _P[_P.model.isin(LABELS)][['text', 'model']].copy()
        _P['text'] = _P.text.fillna('').astype(str); _P['_srcw'] = 0.35
        _keep = []
        for _t in _P.text.values:
            _k = _norm_text_key(_t); _ok = bool(_k) and _k not in _seen
            _keep.append(_ok)
            if _ok: _seen.add(_k)
        _P = _P.loc[_keep].copy(); _parts.append(_P)
    _fitdf = pd.concat(_parts, ignore_index=True)
    _fit = _fitdf.text.astype(str).tolist(); _fy = _fitdf.model.map(L2I).to_numpy(dtype=int); _sw = _fitdf['_srcw'].to_numpy(dtype=np.float32)
    log(f'svc4 fit rows={len(_fit)} source_weighted; external clean={len(EXTRA_DF)}')
    _A, _B = [], []
    for view, lo, hi, mf, wv in [("char", 3, 5, 250000, 1.0), ("word", 1, 3, 180000, 0.7), ("shape", 2, 6, 60000, 0.5), ("function", 1, 3, 50000, 0.5)]:
        prep = _shape if view == "shape" else _func if view == "function" else (lambda s_: s_)
        an = "char" if view == "shape" else "word" if view == "function" else view
        o = dict(analyzer=an, ngram_range=(lo, hi), lowercase=False, sublinear_tf=True, min_df=2, max_features=mf, dtype=np.float32)
        if an == "word": o["token_pattern"] = r"(?u)\b\w+\b|[^\w\s]"
        _v = TfidfVectorizer(**o); _A.append(_v.fit_transform([prep(t) for t in _fit]) * wv); _B.append(_v.transform([prep(t) for t in test.text.values]) * wv)
    _svc = LinearSVC(C=0.5, random_state=42, max_iter=5000, class_weight='balanced')
    _svc.fit(_hs(_A, format='csr'), _fy, sample_weight=_sw)
    _d = _svc.decision_function(_hs(_B, format='csr'))
    LP['svc4'] = (_d - _d.max(1, keepdims=True) - np.log(np.exp(_d - _d.max(1, keepdims=True)).sum(1, keepdims=True))) * a.svc_w
    del _A, _B, _parts, _fitdf, _svc; gc.collect(); log('svc4 done')
if a.distort_w > 0:
    import re, collections
    wre = re.compile(r"[A-Za-z’']+")
    cnt = collections.Counter(w.lower() for t in FIT_T for w in wre.findall(t))
    keep = {w for w, _ in cnt.most_common(100)}
    dist = lambda arr: [wre.sub(lambda m_: m_.group(0) if m_.group(0).lower() in keep else '*', t) for t in arr]
    Dtr, Dte = dist(FIT_T), dist(test.text.values)
    vs = [TfidfVectorizer(sublinear_tf=True, lowercase=False, analyzer='char', ngram_range=(2, 6), min_df=2),
          TfidfVectorizer(sublinear_tf=True, lowercase=True, token_pattern=r"[^\s]+", ngram_range=(1, 3), min_df=2)]
    Xa = sp.hstack([v_.fit_transform(Dtr) for v_ in vs]).tocsr(); Xb = sp.hstack([v_.transform(Dte) for v_ in vs]).tocsr()
    LP['distort'] = LogisticRegression(C=10, max_iter=3000).fit(Xa, FIT_Y, sample_weight=FIT_W).predict_log_proba(Xb) * a.distort_w
    log('distort done')
if a.prefit_core:
    from ntl660_readout import core_votes
    _pv = core_votes(a.prefit_core, test.text.tolist())
    assert set(_pv) == {'svc4', 'tfidf', 'distort'} and all(v.shape == (len(test), 18) and np.isfinite(v).all() for v in _pv.values())
    LP.update(_pv)
    log('prefit core votes loaded', {k: round(float(abs(v).max()), 2) for k, v in _pv.items()})
from transformers import AutoTokenizer, AutoModel
@torch.no_grad()
def l24_predict(spec, texts, groups=None, train_texts=None):
    from transformers import AutoConfig
    from safetensors.torch import load_file
    from concurrent.futures import ThreadPoolExecutor
    adapters = spec.get('adapters') or [spec['adapter']]
    aw = spec.get('adapter_weights')
    if aw is None or len(aw) != len(adapters): aw = [1.0 / len(adapters)] * len(adapters)
    tok = AutoTokenizer.from_pretrained(spec['path'])
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    cfg = AutoConfig.from_pretrained(spec['path']); tc = cfg.text_config if hasattr(cfg, 'text_config') else cfg
    tc.num_hidden_layers = spec['keep']; tc.use_cache = False
    if getattr(tc, 'layer_types', None) is not None: tc.layer_types = tc.layer_types[:spec['keep']]
    _ms = []
    if spec.get('dp') and torch.cuda.device_count() >= 2:
        try:
            for _d in range(2):
                _ms.append(AutoModel.from_pretrained(spec['path'], config=cfg, dtype=torch.float16, device_map={'': _d}).eval())
                for _n in ('vision_tower', 'multi_modal_projector'):
                    if getattr(_ms[-1], _n, None) is not None: setattr(_ms[-1], _n, None)
                gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(_d)
                _ms[-1](input_ids=torch.full((spec.get('bs', 16), spec.get('max_len', 384)), tok.pad_token_id, device=f'cuda:{_d}'), output_hidden_states=True, use_cache=False)
                assert torch.cuda.max_memory_allocated(_d) < torch.cuda.get_device_properties(_d).total_memory - (1 << 30), 'under 1 GiB to spare'
        except Exception as _e:
            log(f'  l24 {spec["name"]}: one split copy instead of one per GPU ({_e!r:.80})'); _ms = []
        gc.collect(); torch.cuda.empty_cache()
    if not _ms: _ms = [AutoModel.from_pretrained(spec['path'], config=cfg, dtype=torch.float16, device_map='auto').eval()]
    m = _ms[0]
    if spec.get('no_final_norm'):
        for _m in _ms: _m.norm = torch.nn.Identity()
    mods = dict(m.named_modules()); first = next(m.parameters()).device; L = spec['layers']
    _mods = [dict(_m.named_modules()) for _m in _ms]
    _eos = spec.get('eos_id')
    _tk = lambda tx: [x + ([_eos] if _eos is not None else []) for x in tok(list(tx), truncation=True, max_length=spec.get('max_len', 384) - (1 if _eos is not None else 0))['input_ids']]
    enc = _tk(texts)
    order = np.argsort([len(x) for x in enc]); bs = spec.get('bs', 16)
    def lsm(z): return z - z.max(1, keepdims=True) - np.log(np.exp(z - z.max(1, keepdims=True)).sum(1, keepdims=True))
    P_acc, C_acc = 0.0, 0.0; FEATS = None; OVF = [0, 0]
    if train_texts is not None:
        enc_tr = _tk(train_texts)
    def featurize(enc_, norms, hid):
        order_ = np.argsort([len(x) for x in enc_])
        Z_ = np.zeros((len(enc_), hid * len(L)), np.float32)
        batches = [order_[i:i + bs] for i in range(0, len(order_), bs)]

        def feats(mm, b, check=True):
            dev = next(mm.parameters()).device
            width = max(len(enc_[j]) for j in b)
            ids = torch.full((len(b), width), tok.pad_token_id)
            att = torch.zeros((len(b), width), dtype=torch.long)
            for r, j in enumerate(b):
                ids[r, :len(enc_[j])] = torch.tensor(enc_[j])
                att[r, :len(enc_[j])] = 1
            with torch.no_grad():
                hs = mm(input_ids=ids.to(dev), attention_mask=att.to(dev), output_hidden_states=True, use_cache=False).hidden_states
                if check and not all(torch.isfinite(hs[l]).all() for l in L):
                    return None
                mask = att.unsqueeze(-1).float()
                mask[:, 0] = 0
                mg = {d_: mask.to(d_) for d_ in {hs[l].device for l in L}}
                return torch.cat([nm(((hs[l].float() * mg[hs[l].device]).sum(1) / mg[hs[l].device].sum(1)).cpu()) for l, nm in zip(L, norms)], -1).numpy()

        def put(b, r):
            Z_[b] = r

        def checked(k, b):
            mm = _ms[k]
            r = feats(mm, b)
            if r is None:
                mm.to(torch.bfloat16)
                r = feats(mm, b, check=False)
                mm.to(torch.float16)
                OVF[k] += 1
            put(b, r)

        def run_copy(k, n):
            for b in batches[k::n]:
                checked(k, b)

        if len(_ms) == 1:
            if spans_devices(_ms[0]):
                pipelined(batches, lambda b: feats(_ms[0], b), put, lambda b: checked(0, b), workers=2, tag='l24 ' + spec['name'], log=log)
            else:
                for b in batches:
                    checked(0, b)
        else:
            try:
                with ThreadPoolExecutor(len(_ms)) as _ex:
                    for _f in [_ex.submit(run_copy, k, len(_ms)) for k in range(len(_ms))]:
                        _f.result()
            except Exception as _e:
                log(f'  l24 {spec["name"]}: two-copy pass failed ({_e!r:.80}), one copy from here on')
                del _ms[1:], _mods[1:]
                gc.collect()
                torch.cuda.empty_cache()
                for b in batches:
                    checked(0, b)
        return Z_
    for ai, ad in enumerate(adapters):
        adp = os.path.join(ad, 'adapter') if os.path.exists(os.path.join(ad, 'adapter', 'adapter_config.json')) else ad
        acfg = json.load(open(os.path.join(adp, 'adapter_config.json'))); scale = acfg['lora_alpha'] / acfg['r']
        sd = load_file(os.path.join(adp, 'adapter_model.safetensors')); deltas = []
        for k, v in sd.items():
            if '.lora_A.' not in k: continue
            name = k.replace('base_model.model.', '', 1).split('.lora_A.')[0]
            if name not in mods and ('model.' + name) in mods: name = 'model.' + name
            B = sd[k.replace('.lora_A.', '.lora_B.')]
            for _md in _mods:
                W = _md[name].weight
                W += ((B.to(W.device, torch.float32) @ v.to(W.device, torch.float32)) * scale).to(W.dtype)
                deltas.append((W, B, v))
        if os.path.exists(os.path.join(ad, 'heads.pt')): H = torch.load(os.path.join(ad, 'heads.pt'), map_location='cpu')
        else: H = dict(load_file(os.path.join(ad, 'heads.safetensors')))
        if os.path.exists(os.path.join(ad, 'meta.json')):
            _lab = json.load(open(os.path.join(ad, 'meta.json')))['labels']; _perm = [_lab.index(l) for l in LABELS]
            for _p in ('plain', 'centered'): H[f'{_p}.weight'] = H[f'{_p}.weight'][_perm]; H[f'{_p}.bias'] = H[f'{_p}.bias'][_perm]
        hid = H['norms.0.weight'].shape[0]
        norms = [torch.nn.LayerNorm(hid) for _ in L]
        for i, nm in enumerate(norms): nm.load_state_dict({'weight': H[f'norms.{i}.weight'], 'bias': H[f'norms.{i}.bias']})
        Z = featurize(enc, norms, hid)
        if train_texts is not None and ai == len(adapters) - 1:
            FEATS = (Z.copy(), featurize(enc_tr, norms, hid))
        for W, B, A_ in deltas:
            W -= ((B.to(W.device, torch.float32) @ A_.to(W.device, torch.float32)) * scale).to(W.dtype)
        torch.cuda.empty_cache()
        del deltas, sd; gc.collect()
        hd = lambda p, X: lsm(X @ H[f'{p}.weight'].float().numpy().T + H[f'{p}.bias'].float().numpy())
        P_acc = P_acc + hd('plain', Z) * aw[ai]
        if groups is not None:
            Zc = Z.copy()
            for g in np.unique(groups):
                ii = groups == g; Zc[ii] -= Zc[ii].mean(0)
            C_acc = C_acc + hd('centered', Zc) * aw[ai]
        log(f'  l24 {spec["name"]}: adapter {os.path.basename(ad)} done (bf16 redo batches {sum(OVF)})')
    del m, mods, _ms, _mods; gc.collect(); torch.cuda.empty_cache()
    if train_texts is not None: return P_acc, (C_acc if groups is not None else None), FEATS
    return P_acc, (C_acc if groups is not None else None)

def gpu_mem():
    return ' '.join(f'gpu{i}:{torch.cuda.memory_allocated(i)/1e9:.1f}GB' for i in range(torch.cuda.device_count()))

specs = json.loads(a.models)
failed = []


def run_l24(spec):
    log('l24', spec['name'])
    out = {}
    if spec.get('gauss', 0) > 0:
        plain, cent, (Zte_g, Ztr_g) = l24_predict(spec, test.text.values, Gte if a.adapt_w > 0 else None, train_texts=train.text.values)
        import gauss as _ga
        prm = _ga.fit(Ztr_g, y, Gtr, dict(red='wpca', d=128, se_shrink='lw', sp_shrink='lw', lam_floor=1e-4, n_exact=3, inj=True, iters=5, repel=True, sp_scale=1.0))
        prm['T1'] = prm['Tg'] = 3.0
        _sz = np.bincount(Gte)[Gte]
        _rows = (_sz >= 2) & (_sz <= 18)
        _q = _ga.scores(Ztr_g[[0, 1]], np.zeros(2, dtype=int), prm)
        assert np.allclose(np.exp(_q).sum(1), 1)
        _G = np.zeros((len(test), 18))
        if _rows.any():
            _G[_rows] = _ga.scores(Zte_g[_rows], np.unique(Gte[_rows], return_inverse=True)[1], prm)
        out[spec['name'] + '_gauss'] = _G * spec['gauss']
        log(f"  gaussian head {spec['name']}: {int(_rows.sum())} grouped test rows, weight {spec['gauss']}")
    else:
        plain, cent = l24_predict(spec, test.text.values, Gte if a.adapt_w > 0 else None)
    out[spec['name']] = plain * spec.get('w', 1.0)
    if cent is not None:
        rows = np.bincount(Gte)[Gte] >= 2
        out[spec['name'] + '_twin'] = np.where(rows[:, None], cent, 0.0) * a.adapt_w * spec.get('w', 1.0) * spec.get('twin', 1.0)
    LP.update(out)


def run_enc(spec):
    log('enc', spec['name'])
    from safetensors.torch import load_file as _lf
    _ck = spec['ckpt']
    _meta = json.load(open(os.path.join(_ck, 'meta.json')))
    _perm = [_meta['labels'].index(l) for l in LABELS]
    _hd = _lf(os.path.join(_ck, 'head.safetensors'))
    _W = _hd['weight'].float().numpy()[_perm]
    _b = _hd['bias'].float().numpy()[_perm]
    _etok = AutoTokenizer.from_pretrained(os.path.join(_ck, 'encoder'))
    _enc = AutoModel.from_pretrained(os.path.join(_ck, 'encoder'), dtype=torch.float32).to('cuda:0').eval()
    _ids = _etok(list(test.text.values), truncation=True, max_length=_meta['max_len'])['input_ids']
    _ord = np.argsort([len(x) for x in _ids], kind='stable')
    Z_ = np.zeros((len(_ids), _enc.config.hidden_size), np.float32)
    with torch.inference_mode():
        for _s in range(0, len(_ord), 32):
            _idx = _ord[_s:_s + 32]
            _bt = _etok.pad({'input_ids': [_ids[i] for i in _idx]}, return_tensors='pt').to('cuda:0')
            _m = _bt['attention_mask'].unsqueeze(-1).float()
            Z_[_idx] = ((_enc(**_bt).last_hidden_state * _m).sum(1) / _m.sum(1).clamp(min=1)).cpu().numpy()
    del _enc
    _zz = Z_ @ _W.T + _b
    LP[spec['name']] = (_zz - _zz.max(1, keepdims=True) - np.log(np.exp(_zz - _zz.max(1, keepdims=True)).sum(1, keepdims=True))) * spec.get('w', 1.0)


RUNNERS = {'l24': run_l24, 'enc': run_enc}
for spec in specs:
    if spec.get('optional') and a.optional_deadline and time.time() > a.optional_deadline:
        log('MEMBER SKIPPED (time):', spec['name'])
        failed.append(spec['name'])
        continue
    try:
        RUNNERS[spec['type']](spec)
    except Exception as _e:
        failed.append(spec['name'])
        log(f"MEMBER FAILED {spec['name']}: {_e!r}")
    gc.collect()
    torch.cuda.empty_cache()
    log(spec['name'], 'done', gpu_mem())
log('members failed:', failed if failed else 'none')

tot = sum(LP.values())
if a.save_lp:
    np.save(a.save_lp, tot)
COMP_P = None
if a.comp_w > 0 and a.comp_path and os.path.exists(a.comp_path):
    try:
        _cm = json.load(open(a.comp_path.replace('comp_logit.npy', 'comp_meta.json')))
        assert _cm['ids'] == test['id'].astype(str).tolist(), 'comp stack row order differs'
        _cl = np.load(a.comp_path)[:, [_cm['labels'].index(l) for l in LABELS]]
        COMP_P = np.exp(_cl - _cl.max(1, keepdims=True))
        COMP_P /= COMP_P.sum(1, keepdims=True)
        assert np.isfinite(COMP_P).all()
        log(f"comp stack loaded ({_cm['seconds']:.0f}s)")
    except Exception as _e:
        log('COMP SKIPPED:', repr(_e))
        COMP_P = None
elif a.comp_w > 0:
    log('COMP SKIPPED: no stack output')


def blend(terms):
    total = sum(terms.values())
    active = sum(abs(v).max() > 0 for v in terms.values())
    lpn = total / max(1, active) if a.sk_div else total
    P = np.exp(lpn - lpn.max(1, keepdims=True))
    P /= P.sum(1, keepdims=True)
    if COMP_P is not None:
        P = (1 - a.comp_w) * P + a.comp_w * COMP_P
        P /= P.sum(1, keepdims=True)
    return P


def decide_scores(P):
    Q = P.copy()
    rho = float(np.clip(a.sinkhorn_target_mix, 0.0, 1.0))
    marg = np.clip(P.mean(0), 1e-6, None)
    marg /= marg.sum()
    prior = np.bincount(y, minlength=18).astype(np.float64) + 1.0
    prior /= prior.sum()
    target = np.clip((1.0 - rho) * marg + rho * prior, 1e-6, None)
    target /= target.sum()
    for _ in range(100):
        Q *= (len(Q) * target[None, :]) / np.clip(Q.sum(0, keepdims=True), 1e-12, None)
        Q /= np.clip(Q.sum(1, keepdims=True), 1e-12, None)
    return (1 - a.sinkhorn) * np.log(P + 1e-12) + a.sinkhorn * np.log(Q + 1e-12)


def decide(P):
    return decide_scores(P).argmax(1)


def save_final_logits(z):
    if a.save_final_logits:
        np.savez(a.save_final_logits, ids=test.id.astype(str).values.astype('U'),
                 labels=np.array(LABELS), logits=np.asarray(z, dtype=np.float64))


def write_pred(pred, tag):
    sub = pd.DataFrame({'id': test.id.values, 'model': [LABELS[i] for i in pred]})
    assert sub.model.isin(LABELS).all() and sub.id.is_unique and len(sub) == len(test)
    sub.to_parquet(a.out, index=False)
    log(f'wrote {tag}', sub.model.value_counts().to_dict())


def write(P, tag):
    save_final_logits(decide_scores(P))
    write_pred(decide(P), tag)


P = blend(LP)
write(P, 'blend without ATLAS')


def atlas_lite(k):
    import atlas_infer
    scorer = atlas_infer.Scorer(a.atlas_base, a.atlas_dir, log=log)
    try:
        enc = scorer.encode(test.text.values)
        to_b = np.array([scorer.labels.index(l) for l in LABELS])
        cand = np.argsort(-P, 1)[:, :k]
        mean_lp, done = scorer.run_candidates(enc, to_b[cand], a.atlas_deadline or None, workers=a.atlas_workers)
        lp = scorer.candidate_logprobs(mean_lp[done], to_b[cand][done])
    finally:
        scorer.close()
    rows = np.repeat(lp.min(1, keepdims=True), 18, 1)
    np.put_along_axis(rows, cand[done], lp, 1)
    term = np.zeros((len(test), 18))
    term[done] = rows
    return term, done


def atlas_scores(rows, calibrate):
    import atlas_infer
    scorer = atlas_infer.Scorer(a.atlas_base, a.atlas_dir, log=log)
    try:
        enc = scorer.encode(test.text.values)
        out, done = scorer.run(enc, rows, deadline=a.atlas_deadline or None, batch=a.atlas_bs, chunk=a.atlas_chunk, workers=a.atlas_workers)
        order = [scorer.labels.index(l) for l in LABELS]
        if calibrate:
            res = np.zeros((len(test), 18))
            res[done] = scorer.calibrated(out[done])
        else:
            raw = out[..., 1:].mean(-1)
            res = np.divide(raw, raw.sum(1, keepdims=True), out=np.zeros_like(raw), where=raw.sum(1, keepdims=True) > 0)
    finally:
        scorer.close()
    return res[:, order], done


if a.atlas_mode != 'off' and a.atlas_dir:
    if a.atlas_deadline and time.time() > a.atlas_deadline - 600:
        log('ATLAS SKIPPED: not enough time left')
    else:
        try:
            if a.atlas_mode == 'lite':
                term, done = atlas_lite(a.atlas_k)
                P = blend({**LP, 'atlas': a.atlas_member_w * term})
                log(f'atlas lite top-{a.atlas_k} on {int(done.sum())}/{len(test)} rows at weight {a.atlas_member_w}', gpu_mem())
            elif a.atlas_mode == 'head':
                lp, done = atlas_scores(np.arange(len(test)), True)
                P = blend({**LP, 'atlas': a.atlas_member_w * lp})
                log(f'atlas head on {int(done.sum())}/{len(test)} rows at weight {a.atlas_member_w}', gpu_mem())
            else:
                top2 = np.sort(P, 1)
                margin = top2[:, -1] - top2[:, -2]
                rows = np.where(margin < a.atlas_margin)[0] if a.atlas_margin > 0 else np.arange(len(test))
                A_, done = atlas_scores(rows, False)
                use = rows[done[rows]]
                mixed = P.copy()
                mixed[use] += a.atlas_w * (A_[use] - P[use])
                P = mixed / mixed.sum(1, keepdims=True)
                log(f'atlas mixed on {len(use)}/{len(rows)} rows at weight {a.atlas_w}', gpu_mem())
        except Exception as _e:
            log('ATLAS SKIPPED:', repr(_e))
        write(P, 'final blend')


if a.gsq_share > 0 and a.gsq_dir:
    if a.gsq_deadline and time.time() > a.gsq_deadline:
        log('GSQ SKIPPED: not enough time left')
    else:
        try:
            import gsq_stage as _gs
            import shutil as _sh
            _sh.rmtree('/tmp/llama-3.1', ignore_errors=True)
            gc.collect()
            torch.cuda.empty_cache()
            Q27 = _gs.run(a.gsq_dir, test['id'].astype(str).tolist(), test.text.tolist(), Gte, a.gsq_work, LABELS, log, a.gsq_timeout or None)
            _agree = float((Q27.argmax(1) == P.argmax(1)).mean())
            log(f'GSQ 27B agreement with the blend {_agree:.3f}')
            if _agree < a.gsq_min_agree:
                raise RuntimeError(f'GSQ agreement {_agree:.3f} below {a.gsq_min_agree}: refusing to use it')
            P = (1 - a.gsq_share) * P + a.gsq_share * Q27
            P /= P.sum(1, keepdims=True)
            write(P, 'blend with GSQ 27B')
        except Exception as _e:
            log('GSQ SKIPPED:', repr(_e))


def rowsoftmax(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def native_stage(P):
    import subprocess
    import native_q8
    os.makedirs(a.native_work, exist_ok=True)
    out = os.path.join(a.native_work, 'native.npz')
    if os.path.exists(out):
        os.remove(out)
    gc.collect()
    torch.cuda.empty_cache()
    cmd = [sys.executable, '-u', os.path.join(os.path.dirname(os.path.abspath(__file__)), 'native_q8.py'), '--base', a.native_base, '--assets', a.native_dir,
           '--query', os.path.join(a.data, 'test.csv'), '--out', out, '--work', a.native_work, '--deadline', str(a.native_deadline)]
    env = dict(os.environ, HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1', TOKENIZERS_PARALLELISM='false')
    code = subprocess.run(cmd, env=env).returncode
    if not os.path.exists(out):
        raise RuntimeError(f'native worker produced nothing (exit code {code})')
    z = np.load(out, allow_pickle=False)
    folds = z['folds'].tolist()
    assert z['ids'].astype(str).tolist() == test['id'].astype(str).tolist(), 'native row order differs'
    labels = z['labels'].astype(str).tolist()
    order = [labels.index(l) for l in LABELS]
    states = [json.load(open(os.path.join(a.native_dir, 'calibration', f'fold{k}.json')))['models']['qwen8'] for k in folds]
    lp = native_q8.calibrate(z['plain'][:, :, order], z['centered'][:, :, order], z['centered_bias'][:, order], Gte, states, bool(a.native_cent))
    share = (a.native_share * z['finite'])[:, None]
    logP = np.log(np.maximum(P, 1e-15))
    agree = [float((lp[i].argmax(1) == P.argmax(1)).mean()) for i in range(len(folds))]
    log(f'native folds {folds} (worker exit {code}); agreement with the blend per fold', [round(v, 3) for v in agree],
        'rows without native', int((~z['finite']).sum()))
    if min(agree) < a.native_min_agree:
        raise RuntimeError(f'native agreement {min(agree):.3f} below {a.native_min_agree}: refusing to use it')
    probs = np.zeros_like(P)
    for i in range(len(folds)):
        probs += rowsoftmax(decide_scores(rowsoftmax((1 - share) * logP + share * lp[i]))) / len(folds)
    assert np.isfinite(probs).all() and np.allclose(probs.sum(1), 1)
    return probs


if a.native_share > 0 and a.native_dir and a.native_base:
    if a.native_deadline and time.time() > a.native_deadline - 1500:
        log('NATIVE SKIPPED: not enough time left')
    else:
        try:
            probs = native_stage(P)
            changed = int((probs.argmax(1) != decide(P)).sum())
            write_pred(probs.argmax(1), 'final blend with native q8')
            save_final_logits(np.log(np.maximum(probs, 1e-15)))
            log(f'native q8 at share {a.native_share}: {changed}/{len(test)} decisions differ from the blend-only decision', gpu_mem())
        except Exception as _e:
            log('NATIVE SKIPPED:', repr(_e))

## 3. Stage-1 Multi-Backbone Ensemble Execution
Configures the five L24 LoRA backbones, DeBERTa encoder, Bayesian Gaussian weights, Sinkhorn-Knopp prior mix, Native Q8 share, and GSQ-27B share; runs a fast prefit smoke test on commit (`FAST_COMMIT=True`) and full dual-T4 inference during competition reruns.

### Complementary Multi-Architecture Stack (`comp_stack.py`)

Generates ensemble logits (`/kaggle/working/comp_logit.npy`) across 16 complementary experts (Stylegram TF-IDF, UltraBERT-380M, DeBERTa-v3-Large, ELECTRA-Large, Desklib, LoRA-24 variants, Token-Attention, Bayesian Gaussian heads, and conditional family gates) fused at `--comp_w 0.15`.

In [ ]:
%%writefile comp_stack.py
"""Complementary Multi-Architecture Style, Encoder & LoRA Ensemble Stack.

Trains and evaluates 16 complementary experts (Stylegram TF-IDF, UltraBERT-380M,
DeBERTa-v3-Large, ELECTRA-Large, Desklib, LoRA-24 variants, Token-Attention,
Bayesian Gaussian heads, and conditional family gates) into comp_logit.npy."""
import gc, glob, json, os, pathlib, re, time
import numpy as np, pandas as pd
try:
    import torch
except ImportError:
    torch = None
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.sparse import csr_matrix, diags, hstack, vstack
from scipy.spatial.distance import squareform
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from pipe import pipelined, spans_devices

SMOKE = os.environ.get('NTL_SMOKE') == '1'
T0 = time.time()
if torch is not None:
    torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False
LABELS = sorted(['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra',
                 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash',
                 'claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5',
                 'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3'])
K = len(LABELS)
LAB2I = {l: i for i, l in enumerate(LABELS)}
FAMILY_IDX = {f: np.array([i for i, l in enumerate(LABELS) if l.startswith(f)]) for f in ('claude', 'gpt', 'gemini', 'glm')}

GROUP_THRESH, GROUP_LAYER, GROUP_MIN, GROUP_MAX = 0.60, 24, 2, 18
STYLE_GROUP_THRESH = 0.90

W = {'stylegram': 1.516, 'stylometry_lgb': 0.063, 'ultrabert': 0.095,
     'q3_8b': 0.0, 'q25_7b': 0.0, 'q3_4b': 0.0, 'glm9b': 0.0, 'debv3l': 0.05, 'electl': 0.032, 'desk': 0.132,
     'lora24_public': 0.177, 'lora24_seed7': 0.113, 'lora24_claude': 0.177, 'lora20_q25_instyle': 0.17,
     'tokattn_q35': 0.153, 'tokattn_deb': 0.185}
TWIN_W = {'q3_8b': 1.5 * 0.219, 'q25_7b': 1.5 * 0.141, 'q3_4b': 1.5 * 0.083, 'glm9b': 1.5 * 0.110, 'debv3l': 0.25}
LORA_TWIN_W = {'lora24_public': 0.15, 'lora24_claude': 0.15}
GATE_LAM = {'claude_style': 1.5, 'claude_lora': 1.0, 'family': 0.25}
EXTRA_MIN_LABELS, EXTRA_SHARE = 12, 0.5

GAUSS_HEADS = {'gauss_q3_8b_L22': dict(src='q3_8b', part='L22_mean', d=384, T1=2.351, Tg=2.306, w=TWIN_W['q3_8b']),
               'gauss_glm9b_L8': dict(src='glm9b', part='L8_mean', d=384, T1=2.722, Tg=2.596, w=TWIN_W['glm9b']),
               'gauss_q3_4b_L4': dict(src='q3_4b', part='L4_mean', d=384, T1=2.725, Tg=2.578, w=TWIN_W['q3_4b']),
               'gauss_debv3l_L24': dict(src='debv3l', part='L24_mean', d=384, T1=3.112, Tg=3.069, w=TWIN_W['debv3l'])}
GAUSS_SETTING = dict(red='wpca', se_shrink='lw', sp_shrink='lw', lam_floor=1e-4, n_exact=3, inj=True, iters=5, repel=True, sp_scale=1.0)

LM_HEADS = [
    ('glm9b', 'glm4', 4096, [(8, 'mean'), (3, 'mean')], [(GROUP_LAYER, 'mean')], 0.0025),
    ('q3_8b', 'qwen3', 4096, [(7, 'mean'), (22, 'mean')], [], 0.0025),
    ('q25_7b', 'qwen2', 3584, [(6, 'mean'), (20, 'mean')], [], 0.0025),
    ('q3_4b', 'qwen3', 2560, [(4, 'style'), (7, 'style')], [(4, 'mean')], 0.0025)]
ENC_HEADS = [
    ('debv3l', 'deberta-v3-large', [(24, 'mean'), (6, 'mean')], 0.005),
    ('electl', 'electra-large-discriminator', [(21, 'mean'), (24, 'style')], 0.005),
    ('desk', 'desklib-detector', [(23, 'mean'), (23, 'style')], 0.005)]
LORA_RUNS = [
    (('qwen3', 4096, 24), [2, 4, 6, 8, 12, 16, 20, 24], [
        ('lora24_public', '**/lora24/final/COMPLETE.json', 'heads.safetensors', 'anchor_features.npz'),
        ('lora24_seed7', '**/name-that-llm-train-lora24-s7/lora_out/job_final/COMPLETE.json', 'norms.safetensors', 'valid_features.npz'),
        ('lora24_claude', '**/ntl-qwen3-8b-lora-claude-head/**/final/COMPLETE.json', 'heads.safetensors', 'anchor_features.npz')]),
    (('qwen2', 3584, 20), [2, 4, 6, 8, 10, 12, 16, 20], [
        ('lora20_q25_instyle', '**/ntl-train-lora-qwen25-instyle/lora_out/job_final/COMPLETE.json', 'norms.safetensors', 'valid_features.npz')])]
LORA_MAX_LEN, LORA_BS = 447, 8
TOKATTN_HEADS = [('tokattn_q35', 'tokattn_final.pt', 'qwen3_5'), ('tokattn_deb', 'tokattn_debl23_final.pt', 'deberta')]
if os.environ.get('NTL_DROP_Q35') == '1':
    TOKATTN_HEADS = [t for t in TOKATTN_HEADS if t[0] != 'tokattn_q35']

if SMOKE:
    _all = pd.read_csv(os.environ['NTL_DATA'] + '/train.csv')
    _p = _all['prompt_id'].drop_duplicates().sample(frac=1, random_state=0).tolist()
    train = _all[_all['prompt_id'].isin(_p[:30])].reset_index(drop=True)
    test = _all[_all['prompt_id'].isin(_p[30:42])].sample(frac=1, random_state=1).reset_index(drop=True)
    test = pd.concat([test[test['prompt_id'] == p].head(k) for p, k in zip(_p[30:42], [1, 1, 2, 2, 2, 3, 3, 4, 6, 9, 18, 18])]).reset_index(drop=True)
    if os.environ.get('NTL_TINY') == '1':
        test = test.head(3).reset_index(drop=True)
    SMOKE_Y = test['model'].map(LAB2I).to_numpy(dtype=int)
else:
    DATA_DIR = next(p for p in [pathlib.Path(__import__('os').environ.get('NTL_REHEARSAL_DATA', '/kaggle/input/competitions/name-that-llm')), pathlib.Path('/kaggle/input/name-that-llm')]
                    if (p / 'train.csv').exists())
    train, test = pd.read_csv(DATA_DIR / 'train.csv'), pd.read_csv(DATA_DIR / 'test.csv')
    if os.environ.get('NTL_FAKE_TEST'):
        test = pd.read_parquet(os.environ['NTL_POOL']).sample(int(os.environ['NTL_FAKE_TEST']), random_state=0)[['text']].reset_index(drop=True)
        test['id'] = [f'fake{i}' for i in range(len(test))]
test['text'] = test['text'].fillna('').astype(str)
TR, TE = train['text'].tolist(), test['text'].tolist()
y = train['model'].map(LAB2I).to_numpy(dtype=int)
PROMPT = train['prompt_id'].to_numpy(dtype=object)
TINY = len(TE) < 50
DEV = 'cuda' if torch is not None and torch.cuda.is_available() else 'cpu'
print(f'train {train.shape}  test {test.shape}  gpus {torch.cuda.device_count() if torch is not None else 0}  smoke {SMOKE}', flush=True)

LOGIT = np.zeros((len(TE), K))


def add_member(key, prob, rows=None, weight=None):
    w = W[key] if weight is None else weight
    lp = np.log(np.clip(prob, 1e-9, 1.0))
    if rows is None:
        LOGIT[:] += w * lp
    else:
        LOGIT[rows] += w * lp
    print(f'  + {key:20s} weight {w:.3f} on {len(TE) if rows is None else int(np.sum(rows))} rows  ({time.time() - T0:.0f}s)', flush=True)


def softmax_rows(S):
    S = S - S.max(1, keepdims=True)
    P = np.exp(S)
    return P / P.sum(1, keepdims=True)


def full_prob(P_part, classes):
    P = np.zeros((len(P_part), K))
    P[:, classes] = P_part
    return P


def group_centre_dense(X, groups):
    X = np.asarray(X, dtype=np.float32).copy()
    for g in np.unique(groups):
        i = groups == g
        X[i] -= X[i].mean(0)
    return X


def self_test_rows():
    idx = np.where(np.isin(PROMPT, pd.unique(PROMPT)[:3]))[0]
    return idx, np.unique(PROMPT[idx], return_inverse=True)[1]

def average_linkage_groups(D, thresh):
    if len(D) < 2:
        return np.ones(len(D), dtype=int)
    D = np.asarray(D, dtype=np.float64).copy()
    np.fill_diagonal(D, 0.0)
    return fcluster(linkage(squareform(D, checks=False), method='average'), thresh, criterion='distance')


def content_cosine_distance(texts, max_features=None):
    X = TfidfVectorizer(stop_words='english', sublinear_tf=True, max_features=max_features, dtype=np.float32).fit_transform(texts)
    return np.clip(1.0 - (X @ X.T).toarray(), 0.0, 1.0)


def mixed_groups(texts, E, thresh=GROUP_THRESH):
    if len(texts) < 2:
        return np.ones(len(texts), dtype=int)
    Z = np.asarray(E, dtype=np.float64)
    Z = Z - Z.mean(0)
    Z /= np.linalg.norm(Z, axis=1, keepdims=True) + 1e-12
    D = 0.5 * (content_cosine_distance(texts) + np.clip(1.0 - Z @ Z.T, 0.0, 2.0))
    return average_linkage_groups(D, thresh)


def normalised_content_groups(texts, thresh=STYLE_GROUP_THRESH):
    if len(texts) < 2:
        return np.ones(len(texts), dtype=int)
    D = content_cosine_distance(texts, max_features=100_000)
    med = max(float(np.median(D[np.triu_indices(len(texts), 1)])), 1e-6)
    return average_linkage_groups(D / med, thresh)


def eligible(groups, lo=GROUP_MIN, hi=GROUP_MAX):
    size = np.bincount(groups)[groups]
    return (size >= lo) & (size <= hi)


def describe_groups(name, groups):
    size = np.bincount(groups)[groups]
    print(f'{name}: {len(groups)} rows | singletons {int((size == 1).sum())}  2-3: {int(((size >= 2) & (size <= 3)).sum())}'
          f'  4-18: {int(((size >= 4) & (size <= 18)).sum())}  >18: {int((size > 18).sum())}', flush=True)


G_TE, GROUP_ROWS = None, None

FUNCTION_WORDS = frozenset("a an the and or but if while although because as so than that which who whom whose this these those it its "
                           "they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with "
                           "without for of by over under between through into about is are was were be been being have has had do does did "
                           "can could will would shall should may might must not no nor all any some each every both either neither other "
                           "another such only also just very too more most less least here there when where why how what whether then now "
                           "still yet even much many few little".split())
TOKEN_RE = re.compile(r"\w+|[^\w\s]")


def shape_view(text):
    return re.sub(r"[^\W\d_]+", lambda m: 'X' if m[0].isupper() else 'x', text)


def function_view(text):
    return ' '.join(t.lower() if (t.lower() in FUNCTION_WORDS or not t.isalnum()) else 'CONTENT' for t in TOKEN_RE.findall(text))


class StyleGrams:
    VIEWS = [('char', str, (3, 5), 250_000, 1.0), ('word', str, (1, 3), 180_000, 0.7),
             ('char', shape_view, (2, 6), 60_000, 0.5), ('word', function_view, (1, 3), 50_000, 0.5)]

    def fit(self, texts):
        self.vecs = []
        for analyzer, view, ngr, maxf, w in self.VIEWS:
            opts = dict(analyzer=analyzer, ngram_range=ngr, lowercase=False, sublinear_tf=True, min_df=2, max_features=maxf, dtype=np.float64)
            if analyzer == 'word':
                opts['token_pattern'] = r"(?u)\b\w+\b|[^\w\s]"
            self.vecs.append((TfidfVectorizer(**opts).fit([view(t) for t in texts]), view, w))
        return self

    def transform(self, texts):
        return hstack([v.transform([view(t) for t in texts]) * w for v, view, w in self.vecs], format='csr')


def sparse_group_centre(X, groups, lo=GROUP_MIN, hi=GROUP_MAX):
    X = csr_matrix(X)
    _, inv, cnt = np.unique(np.asarray(groups), return_inverse=True, return_counts=True)
    keep = np.flatnonzero((cnt >= lo) & (cnt <= hi))
    if not len(keep):
        return X.copy()
    col = np.full(len(cnt), -1); col[keep] = np.arange(len(keep))
    rows = np.flatnonzero(col[inv] >= 0)
    M = csr_matrix((np.ones(len(rows)), (rows, col[inv][rows])), shape=(X.shape[0], len(keep)))
    means = diags(1.0 / cnt[keep]) @ (M.T @ X)
    return (X - M @ means).tocsr()


def svc_probs(svc, X):
    return full_prob(softmax_rows(svc.decision_function(X)), svc.classes_)


def load_extra_passages():
    sources = [('**/ntl_extra_passages_claude.csv', 'model', None), ('**/gen_passages.csv', 'model', None),
               ('**/ntl_generations.csv', 'label', 'finish_reason'), ('**/ntl_proxy_generations.csv', 'label', None),
               ('**/ntl_generated*.csv', 'model', 'finish'), ('**/passages_clean.csv', 'model', None)]
    parts = []
    for pattern, label_col, finish_col in sources:
        for f in sorted(glob.glob('/kaggle/input/' + pattern, recursive=True)):
            d = pd.read_csv(f)
            _pin = {'ntl_generations.csv': lambda d: d['batch'].astype(str).isin(
                        {'b001', 'b001g', 'b003', 'b003a', 'd1028', 'd1029', 'd1030', 'pilot', 'pilot2', 'pilot3', 'pilotpro'}),
                    'gen_passages.csv': lambda d: d['batch'].astype(str).isin(
                        {f'b{i}' for i in (1, 2, *range(5, 16))} | {f'v2b{i:02d}' for i in range(1, 21)}),
                    'passages_clean.csv': lambda d: pd.to_numeric(d['dataset_version_added'], errors='coerce').isin([1, 2, 4, 5])
                    }.get(os.path.basename(f))
            if _pin is not None:
                try:
                    d = d[_pin(d)]
                except KeyError as _e:
                    print('extra-passage pin skipped for', f, repr(_e), flush=True)
            if finish_col and finish_col in d:
                d = d[d[finish_col].eq('stop')]
            d = d.rename(columns={label_col: 'model'})[['prompt_id', 'model', 'text']].copy()
            d['prompt_id'] = ('pv_' if f.endswith('passages_clean.csv') else '') + d['prompt_id'].astype(str)
            parts.append(d)
    for f in sorted(glob.glob('/kaggle/input/**/ntl_open_pool.parquet', recursive=True), key=lambda _p, _m={"glm53-iq3xxs-segs-a-mirror": "/kaggle/input/datasets/jamesliebel/glm53-iq3xxs-segs-a-mirror", "glm53-iq3xxs-segs-b-mirror": "/kaggle/input/datasets/jamesliebel/glm53-iq3xxs-segs-b-mirror", "glm53-flash-iq3xxs-segs-mirror": "/kaggle/input/datasets/jamesliebel/glm53-flash-iq3xxs-segs-mirror", "ntl-olmo2-native24-final-inference-assets": "/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets", "ntl-spectral-style-heads-paired-4404": "/kaggle/input/datasets/kirilltym/ntl-spectral-style-heads-paired-4404", "nlm-enc-v5": "/kaggle/input/datasets/dhruvtibarewal/nlm-enc-v5", "qwen35-9b-safetensors": "/kaggle/input/datasets/bbucxi/qwen35-9b-safetensors", "ntl-glm-likelihood-train": "/kaggle/input/notebooks/estopadilla/ntl-glm-likelihood-train"}: next((_m[_s] + _p[len(_s) + 14:] for _s in _m if _p.startswith("/kaggle/input/" + _s + "/")), _p))[:1]:
        d = pd.read_parquet(f, columns=['prompt_group', 'model', 'text']).rename(columns={'prompt_group': 'prompt_id'})
        d['prompt_id'] = 'op_' + d['prompt_id'].astype(str)
        parts.append(d)
    if not parts:
        return pd.DataFrame(columns=['prompt_id', 'model', 'text'])
    x = pd.concat(parts, ignore_index=True)
    x = x[x['model'].isin(LABELS)].copy()
    x['text'] = x['text'].astype(str).map(lambda t: re.sub(r'\s+', ' ', re.sub(r'^\*\*(.*)\*\*$', r'\1', t.strip())).strip())
    x = x[x['text'].str.split().str.len().between(30, 300)].drop_duplicates(subset=['model', 'text'])
    x = x.drop_duplicates(subset=['prompt_id', 'model'])
    x = x[x.groupby('prompt_id')['model'].transform('nunique') >= EXTRA_MIN_LABELS].reset_index(drop=True)
    return x


EXTRA = load_extra_passages()
EX_T, EX_Y, EX_G = EXTRA['text'].tolist(), EXTRA['model'].map(LAB2I).to_numpy(dtype=int), EXTRA['prompt_id'].to_numpy(dtype=object)
print(f'extra passages: {len(EXTRA)} rows, {EXTRA["prompt_id"].nunique()} prompts', flush=True)


def extra_sample_weights():
    n_tr, n_ex = np.bincount(y, minlength=K).astype(float), np.bincount(EX_Y, minlength=K).astype(float)
    w_tr = np.where(n_ex[y] > 0, 1.0 - EXTRA_SHARE, 1.0)
    w_ex = EXTRA_SHARE * n_tr[EX_Y] / np.maximum(n_ex[EX_Y], 1.0)
    return np.concatenate([w_tr, w_ex])


grams = StyleGrams().fit(TR + EX_T)
X_tr, X_ex, X_te = grams.transform(TR), grams.transform(EX_T) if len(EX_T) else None, grams.transform(TE)


def fit_svc(X, labels, C=0.5, **kw):
    return LinearSVC(C=C, random_state=42, max_iter=4000).fit(X, labels, **kw)


if X_ex is not None:
    svc_plain = fit_svc(vstack([X_tr, X_ex]).tocsr(), np.concatenate([y, EX_Y]), sample_weight=extra_sample_weights())
else:
    svc_plain = fit_svc(X_tr, y)
P_style = svc_probs(svc_plain, X_te)

X_tr_c = sparse_group_centre(X_tr, PROMPT)
if X_ex is not None:
    svc_centred = fit_svc(vstack([X_tr_c, sparse_group_centre(X_ex, EX_G, GROUP_MIN, 10 ** 6)]).tocsr(), np.concatenate([y, EX_Y]),
                          sample_weight=extra_sample_weights())
else:
    svc_centred = fit_svc(X_tr_c, y)
G_style = normalised_content_groups(TE)
rows_style = eligible(G_style)
describe_groups('stylegram groups', G_style)
if rows_style.any():
    P_style[rows_style] = svc_probs(svc_centred, sparse_group_centre(X_te[rows_style], G_style[rows_style]))
if TINY:
    st_idx, st_g = self_test_rows()
    acc = np.mean(svc_centred.classes_[svc_centred.decision_function(sparse_group_centre(X_tr[st_idx], st_g)).argmax(1)] == y[st_idx])
    print(f'  centred stylegram self-test on {len(st_idx)} training rows (seen in training): acc {acc:.2f}', flush=True)
add_member('stylegram', P_style)
del X_tr_c, svc_plain, svc_centred, P_style


def family_gate(texts, labels, classes, C, sample_weight=None, balanced=False):
    g = StyleGrams().fit(texts)
    svc = LinearSVC(C=C, class_weight='balanced' if balanced else None, random_state=42, max_iter=4000).fit(
        g.transform(texts), labels, sample_weight=sample_weight)
    assert list(svc.classes_) == list(classes)
    d = svc.decision_function(g.transform(TE))
    return np.column_stack([-d, d]) if len(classes) == 2 else d


GATES = {}
api_claude = sorted(glob.glob('/kaggle/input/**/ntl_extra_passages_claude.csv', recursive=True), key=lambda _p, _m={"glm53-iq3xxs-segs-a-mirror": "/kaggle/input/datasets/jamesliebel/glm53-iq3xxs-segs-a-mirror", "glm53-iq3xxs-segs-b-mirror": "/kaggle/input/datasets/jamesliebel/glm53-iq3xxs-segs-b-mirror", "glm53-flash-iq3xxs-segs-mirror": "/kaggle/input/datasets/jamesliebel/glm53-flash-iq3xxs-segs-mirror", "ntl-olmo2-native24-final-inference-assets": "/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets", "ntl-spectral-style-heads-paired-4404": "/kaggle/input/datasets/kirilltym/ntl-spectral-style-heads-paired-4404", "nlm-enc-v5": "/kaggle/input/datasets/dhruvtibarewal/nlm-enc-v5", "qwen35-9b-safetensors": "/kaggle/input/datasets/bbucxi/qwen35-9b-safetensors", "ntl-glm-likelihood-train": "/kaggle/input/notebooks/estopadilla/ntl-glm-likelihood-train"}: next((_m[_s] + _p[len(_s) + 14:] for _s in _m if _p.startswith("/kaggle/input/" + _s + "/")), _p))
if api_claude:
    api = pd.read_csv(api_claude[0])
    m = np.isin(y, FAMILY_IDX['claude'])
    GATES['claude'] = family_gate([t for t, k in zip(TR, m) if k] + api['text'].astype(str).tolist(),
                                  np.concatenate([y[m], api['model'].map(LAB2I).to_numpy(dtype=int)]), FAMILY_IDX['claude'], 0.5)
    print(f'claude gate: {int(m.sum())} official + {len(api)} API rows', flush=True)
    del api
else:
    print('Claude API passages not attached: Claude gate off', flush=True)
for fam, C in [('gpt', 1.0), ('gemini', 1.0), ('glm', 0.5)]:
    m, me = np.isin(y, FAMILY_IDX[fam]), np.isin(EX_Y, FAMILY_IDX[fam])
    GATES[fam] = family_gate([t for t, k in zip(TR, m) if k] + [t for t, k in zip(EX_T, me) if k], np.concatenate([y[m], EX_Y[me]]),
                             FAMILY_IDX[fam], C, sample_weight=np.concatenate([np.ones(int(m.sum())), np.full(int(me.sum()), 0.25)]), balanced=True)
    print(f'{fam} gate: {int(m.sum())} official + {int(me.sum())} extra rows', flush=True)
del grams, X_tr, X_ex, X_te

PUNCT = list(',.;:!?()[]"\'-') + ['—', '–', '‘', '’', '“', '”', '…', '*', '/', '%']
STOPS = sorted(ENGLISH_STOP_WORDS)
SENT_RE, WORD_RE = re.compile(r'(?<=[.!?])\s+'), re.compile(r"[A-Za-z’']+")


def stylometry(t):
    words = WORD_RE.findall(t)
    low = [w.lower() for w in words]
    nw, nc = max(len(words), 1), max(len(t), 1)
    sents = [s for s in SENT_RE.split(t.strip()) if s]
    sl = np.array([len(WORD_RE.findall(s)) for s in sents] or [0], dtype=float)
    wl = np.array([len(w) for w in words] or [0], dtype=float)
    f = [np.log1p(nw), len(sents), sl.mean(), sl.std(), sl.max(), sl.min(), wl.mean(), wl.std(), (wl >= 9).mean(), len(set(low)) / nw,
         sum(c.isupper() for c in t) / nc, sum(c.isdigit() for c in t) / nc, sum(ord(c) > 127 for c in t) / nc,
         t.count(', ') / nw, t.count(' and ') / nw, t.count(' but ') / nw,
         sum(1 for s in sents if s[:1].isupper() and s.split()[0].lower() in ('this', 'these', 'it', 'they')) / max(len(sents), 1),
         sum(w.endswith('ly') for w in words) / nw, sum(w.endswith('ing') for w in words) / nw, sum(w.endswith('tion') for w in words) / nw]
    f += [t.count(q) / nw for q in PUNCT]
    counts = pd.Series(low).value_counts() if low else pd.Series(dtype=float)
    f += [counts.get(s, 0) / nw for s in STOPS]
    return f


F_tr, F_te = (np.array([stylometry(t) for t in txt], dtype=np.float32) for txt in (TR, TE))
try:
    import lightgbm as lgb
    booster = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.03, num_leaves=15, min_child_samples=10, subsample=0.8, subsample_freq=1,
                                 colsample_bytree=0.5, reg_lambda=1.0, random_state=0, verbose=-1, n_jobs=4).fit(F_tr, y)
    add_member('stylometry_lgb', full_prob(booster.predict_proba(F_te), booster.classes_))
    del booster
except ImportError:
    assert SMOKE, 'lightgbm missing'
    print('smoke: lightgbm unavailable, stylometry member skipped', flush=True)
del F_tr, F_te

import itertools
K = 18
DEFAULT = {"red": "wpca", "d": 256, "se_shrink": "lw", "sp_shrink": "lw", "lam_floor": 1e-4, "n_exact": 3, "inj": True,
           "iters": 5, "repel": True, "sp_scale": 1.0}

def _lse(A, axis=None, keepdims=False):
    m = np.max(A, axis=axis, keepdims=True); m = np.where(np.isfinite(m), m, 0.0)
    out = np.log(np.sum(np.exp(A - m), axis=axis, keepdims=True)) + m
    if keepdims: return out
    return np.squeeze(out, axis=axis) if axis is not None else float(out.item())

def log_softmax(L): return L - _lse(L, axis=1, keepdims=True)

def _lw_shrinkage(X):
    from sklearn.covariance import ledoit_wolf_shrinkage
    return float(ledoit_wolf_shrinkage(X, assume_centered=True))

def _shrunk(S, a):
    d = S.shape[0]; return (1 - a) * S + a * (np.trace(S) / d) * np.eye(d)

def _two_way(Z, y, pinv, G, iters=4):
    P = np.zeros((G, Z.shape[1])); cnt_p = np.bincount(pinv, minlength=G).astype(float)
    cls = np.unique(y); mu = np.zeros((K, Z.shape[1]))
    for _ in range(iters):
        R = Z - P[pinv]
        for c in cls: mu[c] = R[y == c].mean(0)
        R = Z - mu[y]; P = np.zeros_like(P); np.add.at(P, pinv, R); P /= cnt_p[:, None]
    return mu, P, cnt_p

def fit(X, y, prompts, cfg=None):
    c = dict(DEFAULT); c.update(cfg or {})
    X = np.asarray(X, dtype=np.float64); y = np.asarray(y); N, D = X.shape
    pu, pinv = np.unique(np.asarray(prompts), return_inverse=True); G = len(pu)
    mx = X.mean(0); sx = X.std(0); sx = np.where(sx > 1e-8, sx, 1.0); Xs = (X - mx) / sx
    if c["red"] == "none":
        V = np.eye(D)
    else:
        if c["red"] == "wpca":
            Pm = np.zeros((G, D)); np.add.at(Pm, pinv, Xs); Pm /= np.bincount(pinv)[:, None]; R = Xs - Pm[pinv]
        elif c["red"] == "pca": R = Xs
        else: raise ValueError(c["red"])
        d = int(min(c["d"], N - G - 1, D))
        if N < D:
            w, U = np.linalg.eigh(R @ R.T); o = np.argsort(w)[::-1][:d]; w, U = np.clip(w[o], 1e-12, None), U[:, o]
            V = (R.T @ U) / np.sqrt(w)
        else:
            _, _, Vt = np.linalg.svd(R, full_matrices=False); V = Vt[:d].T
    Z = Xs @ V; d = Z.shape[1]
    mu, P, ng = _two_way(Z, y, pinv, G)
    E = Z - mu[y] - P[pinv]; dof = max(N - G - len(np.unique(y)) + 1, 1)
    Se_raw = E.T @ E / dof
    ae = _lw_shrinkage(E) if c["se_shrink"] == "lw" else float(c["se_shrink"]); Se = _shrunk(Se_raw, ae)
    Pc = P - P.mean(0); Sp_raw = Pc.T @ Pc / max(G - 1, 1) - np.mean(1.0 / ng) * Se_raw
    ap = _lw_shrinkage(Pc) if c["sp_shrink"] == "lw" else float(c["sp_shrink"])
    tr = max(np.trace(Sp_raw), 1e-8 * np.trace(Se)); Sp = (1 - ap) * Sp_raw + ap * (tr / d) * np.eye(d); Sp *= c["sp_scale"]
    L = np.linalg.cholesky(Se); Li = np.linalg.inv(L); M = Li @ Sp @ Li.T; M = 0.5 * (M + M.T)
    lam, Q = np.linalg.eigh(M); lam = np.clip(lam, c["lam_floor"], None); A = Q.T @ Li
    W = V @ A.T
    return {"mx": mx, "sx": sx, "W": W, "m": mu @ A.T, "lam": lam, "T1": 1.0, "Tg": 1.0,
            "info": {"d": int(d), "a_e": round(ae, 4), "a_p": round(ap, 4), "lam_pct": np.round(np.percentile(lam, [5, 50, 95, 100]), 3).tolist()}, "cfg": c}

def transform(X, prm): return ((np.asarray(X, dtype=np.float64) - prm["mx"]) / prm["sx"]) @ prm["W"]

_MASK = {}
def _inj_mask(n, C):
    if (n, C) not in _MASK:
        grids = np.meshgrid(*[np.arange(C)] * n, indexing="ij"); bad = np.zeros([C] * n, bool)
        for i, j in itertools.combinations(range(n), 2): bad |= grids[i] == grids[j]
        _MASK[(n, C)] = bad
    return _MASK[(n, C)]

def singleton_ll(U, m, lam):
    R = U[:, None, :] - m[None]; return -0.5 * (R ** 2 / (1 + lam)).sum(-1)

def joint_terms(U, m, lam):
    n = len(U); gam = lam / (1 + n * lam); R = U[:, None, :] - m[None]
    a = -0.5 * (R ** 2 * (1 - gam)).sum(-1); G = R * np.sqrt(gam); B = np.einsum("icd,jed->ijce", G, G)
    return a, B

def joint_table(a, B):
    n, C = a.shape; Q = np.zeros([C] * n)
    for i in range(n):
        sh = [1] * n; sh[i] = C; Q = Q + a[i].reshape(sh)
    for i, j in itertools.combinations(range(n), 2):
        sh = [1] * n; sh[i] = C; sh[j] = C; Q = Q + B[i, j].reshape(sh)
    return Q

def exact_marginals(Q, T=1.0, inj=True):
    n = Q.ndim; C = Q.shape[0]; Qt = Q / T
    if inj and n >= 2: Qt = np.where(_inj_mask(n, C), -np.inf, Qt)
    lz = _lse(Qt); out = np.empty((n, C))
    for i in range(n):
        ax = tuple(k for k in range(n) if k != i); out[i] = (_lse(Qt, axis=ax) if ax else Qt) - lz
    return out

def em_marginals(U, m, lam, T=1.0, iters=5, repel=False, q0=None):
    n = len(U); v = lam / (1 + (n - 1) * lam)
    L = singleton_ll(U, m, lam) / T if q0 is None else np.log(np.clip(q0, 1e-300, None))
    q = np.exp(log_softmax(L))
    for _ in range(iters):
        s = U - q @ m; S = s.sum(0); Pm = (S[None] - s) * v
        R = U[:, None, :] - Pm[:, None, :] - m[None]; L = -0.5 * (R ** 2 / (1 + v)).sum(-1) / T
        if repel:
            lo = np.log(np.clip(1 - q, 1e-6, None)); L = L + (lo.sum(0)[None] - lo)
        q = np.exp(log_softmax(L))
    return log_softmax(L)

LAM_DIFF = 1e6
def lam_g(prm, cfg=None):
    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {})
    return np.full_like(prm["lam"], LAM_DIFF) if c.get("diff") else prm["lam"]

def group_logpost(U, prm, T1=None, Tg=None, cfg=None):
    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {}); n = len(U); lg = lam_g(prm, cfg)
    T1 = prm["T1"] if T1 is None else T1; Tg = prm["Tg"] if Tg is None else Tg
    if n == 1: return log_softmax(singleton_ll(U, prm["m"], prm["lam"]) / T1)
    if n <= c["n_exact"]:
        a, B = joint_terms(U, prm["m"], lg); return exact_marginals(joint_table(a, B), Tg, c["inj"])
    return em_marginals(U, prm["m"], lg, Tg, c["iters"], c["repel"])

def scores(X_pool, lab, prm, T1=None, Tg=None, cfg=None, U=None):
    U = transform(X_pool, prm) if U is None else U; lab = np.asarray(lab); n = len(U); out = np.empty((n, prm["m"].shape[0]))
    sizes = np.bincount(lab); s1 = sizes[lab] == 1
    T1 = prm["T1"] if T1 is None else T1
    if s1.any(): out[s1] = log_softmax(singleton_ll(U[s1], prm["m"], prm["lam"]) / T1)
    for g in np.where(sizes >= 2)[0]:
        ii = np.where(lab == g)[0]; out[ii] = group_logpost(U[ii], prm, T1, Tg, cfg)
    return out

if not SMOKE:
    from transformers import AutoConfig, AutoModel, AutoTokenizer
CACHED_PARTS = {}


def find_checkpoint(model_type, hidden=None, dir_name=None):
    pattern = f'/kaggle/input/**/{dir_name}/config.json' if dir_name else '/kaggle/input/**/config.json'
    for cfg in sorted(glob.glob(pattern, recursive=True), key=lambda _p, _m={"glm53-iq3xxs-segs-a-mirror": "/kaggle/input/datasets/jamesliebel/glm53-iq3xxs-segs-a-mirror", "glm53-iq3xxs-segs-b-mirror": "/kaggle/input/datasets/jamesliebel/glm53-iq3xxs-segs-b-mirror", "glm53-flash-iq3xxs-segs-mirror": "/kaggle/input/datasets/jamesliebel/glm53-flash-iq3xxs-segs-mirror", "ntl-olmo2-native24-final-inference-assets": "/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets", "ntl-spectral-style-heads-paired-4404": "/kaggle/input/datasets/kirilltym/ntl-spectral-style-heads-paired-4404", "nlm-enc-v5": "/kaggle/input/datasets/dhruvtibarewal/nlm-enc-v5", "qwen35-9b-safetensors": "/kaggle/input/datasets/bbucxi/qwen35-9b-safetensors", "ntl-glm-likelihood-train": "/kaggle/input/notebooks/estopadilla/ntl-glm-likelihood-train"}: next((_m[_s] + _p[len(_s) + 14:] for _s in _m if _p.startswith("/kaggle/input/" + _s + "/")), _p)):
        try:
            c = json.load(open(cfg))
        except Exception:
            continue
        if dir_name or (c.get('model_type') == model_type and (hidden is None or c.get('hidden_size') == hidden)):
            return os.path.dirname(cfg)
    return None


def load_truncated_decoder(path, keep, dtype):
    cfg = AutoConfig.from_pretrained(path)
    cfg.num_hidden_layers = keep
    if getattr(cfg, 'layer_types', None):
        cfg.layer_types = cfg.layer_types[:keep]
    kw = dict(config=cfg, dtype=dtype)
    if DEV == 'cuda':
        kw['device_map'] = 'auto'
    return AutoModel.from_pretrained(path, **kw).eval()


def style_token_mask(tok):
    pieces = [tok.decode([i]) for i in range(len(tok))]
    return torch.tensor([(p.strip().lower() in ENGLISH_STOP_WORDS) or bool(re.fullmatch(r'\s*[^\w\s]+\s*', p)) for p in pieces])


@torch.no_grad()
def pooled_states(model, tok, texts, parts, decoder, bs, max_len=512):
    H = model.config.hidden_size
    out = {f'L{L}_{pool}': np.zeros((len(texts), H), dtype=np.float32) for L, pool in parts}
    smask = style_token_mask(tok) if any(pool == 'style' for _, pool in parts) else None
    dev = next(model.parameters()).device
    order = np.argsort([len(t) for t in texts])
    retried = [0]
    batches = [order[b0:b0 + bs] for b0 in range(0, len(texts), bs)]

    def compute(idx, check):
        if decoder:
            batch = [(tok.bos_token or '') + texts[i] for i in idx]
            enc = tok(batch, return_tensors='pt', padding=True, truncation=True, max_length=max_len, add_special_tokens=False).to(dev)
        else:
            enc = tok([texts[i] for i in idx], return_tensors='pt', padding=True, truncation=True, max_length=max_len).to(dev)
        with torch.no_grad():
            hs = model(**enc, output_hidden_states=True).hidden_states
            if check and decoder and not all(torch.isfinite(hs[L]).all() for L, _ in parts):
                return None
            m_all = enc['attention_mask'].float()
            res = {}
            for L, pool in parts:
                m = m_all * smask.to(m_all.device)[enc['input_ids']].float() if pool == 'style' else m_all
                m = m.unsqueeze(-1).to(hs[L].device)
                res[f'L{L}_{pool}'] = ((hs[L].float() * m).sum(1) / m.sum(1).clamp(min=1)).cpu().numpy()
        return res

    def put(idx, res):
        for k, v in res.items():
            out[k][idx] = v

    def checked(idx):
        res = compute(idx, True)
        if res is None:
            retried[0] += 1
            model.to(torch.bfloat16)
            res = compute(idx, False)
        put(idx, res)

    if decoder and spans_devices(model):
        pipelined(batches, lambda idx: compute(idx, True), put, checked, workers=2, tag='pooled states', log=print)
    else:
        for idx in batches:
            checked(idx)
    if retried[0]:
        print(f'  {retried[0]} batches redone in bf16', flush=True)
    return {k: np.nan_to_num(v) for k, v in out.items()}


def pseudo_parts(parts, dim=48):
    from sklearn.decomposition import TruncatedSVD
    tf = TfidfVectorizer(analyzer='char', ngram_range=(2, 4), sublinear_tf=True, min_df=2).fit(TR + TE)
    out = {}
    for j, (L, pool) in enumerate(parts):
        svd = TruncatedSVD(dim, random_state=L + 7 * j).fit(tf.transform(TR))
        out[f'L{L}_{pool}'] = (svd.transform(tf.transform(TR)).astype(np.float32), svd.transform(tf.transform(TE)).astype(np.float32))
    return out


def extract_parts(key, parts, loader):
    if SMOKE:
        return pseudo_parts(parts)
    model, tok, decoder, bs = loader()
    tr, te = pooled_states(model, tok, TR, parts, decoder, bs), pooled_states(model, tok, TE, parts, decoder, bs)
    del model, tok
    gc.collect()
    if DEV == 'cuda':
        torch.cuda.empty_cache()
    return {k: (tr[k], te[k]) for k in tr}


def decoder_loader(model_type, hidden, keep):
    def load():
        path = find_checkpoint(model_type, hidden)
        assert path, f'no {model_type} checkpoint with hidden size {hidden} under /kaggle/input'
        tok = AutoTokenizer.from_pretrained(path)
        tok.padding_side = 'right'
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token
        return load_truncated_decoder(path, keep, torch.float16 if DEV == 'cuda' else torch.float32), tok, True, 8
    return load


def encoder_loader(dir_name):
    def load():
        path = find_checkpoint(None, dir_name=dir_name)
        assert path, f'encoder directory {dir_name} not found under /kaggle/input'
        return AutoModel.from_pretrained(path, dtype=torch.float32).to(DEV).eval(), AutoTokenizer.from_pretrained(path), False, 16
    return load


def concat_parts(cache, parts):
    return (np.concatenate([cache[f'L{L}_{p}'][0] for L, p in parts], 1), np.concatenate([cache[f'L{L}_{p}'][1] for L, p in parts], 1))


def plain_and_twin(key, Etr, Ete, C):
    if W[key] > 0:
        sc = StandardScaler().fit(Etr)
        clf = LogisticRegression(C=C, max_iter=5000).fit(sc.transform(Etr), y)
        Xte = (Ete - Ete.mean(0)) / (Ete.std(0) + 1e-6) if len(Ete) >= 2 else sc.transform(Ete)
        add_member(key, full_prob(clf.predict_proba(Xte), clf.classes_))
    else:
        print(f'  {key}: plain head has weight 0 in the 0.563 blend, not fitted', flush=True)
    if key in TWIN_W:
        csc = StandardScaler().fit(group_centre_dense(Etr, PROMPT))
        twin = LogisticRegression(C=C, max_iter=5000).fit(csc.transform(group_centre_dense(Etr, PROMPT)), y)
        if GROUP_ROWS.any():
            pc = twin.predict_proba(csc.transform(group_centre_dense(Ete[GROUP_ROWS], G_TE[GROUP_ROWS])))
            add_member(key + '_twin', full_prob(pc, twin.classes_), rows=GROUP_ROWS, weight=TWIN_W[key])
        if TINY:
            st_idx, st_g = self_test_rows()
            ps = twin.predict_proba(csc.transform(group_centre_dense(Etr[st_idx], st_g)))
            print(f'  {key} twin self-test on {len(st_idx)} training rows (seen in training): acc {np.mean(twin.classes_[ps.argmax(1)] == y[st_idx]):.2f}', flush=True)


def gaussian_group_head(name, spec, Xtr, Xte):
    prm = fit(Xtr, y, PROMPT, dict(GAUSS_SETTING, d=spec['d']))
    prm['T1'], prm['Tg'] = spec['T1'], spec['Tg']
    print(f'  {name}: d {prm["info"]["d"]}, shrinkage e/p {prm["info"]["a_e"]}/{prm["info"]["a_p"]}, lam pct {prm["info"]["lam_pct"]}', flush=True)
    if GROUP_ROWS.any():
        LP = scores(Xte[GROUP_ROWS], np.unique(G_TE[GROUP_ROWS], return_inverse=True)[1], prm)
        add_member(name, np.exp(LP), rows=GROUP_ROWS, weight=spec['w'])
    if TINY:
        st_idx, st_g = self_test_rows()
        LP = scores(Xtr[st_idx], st_g, prm)
        print(f'  {name} self-test on {len(st_idx)} training rows (seen in training): acc {np.mean(LP.argmax(1) == y[st_idx]):.2f}', flush=True)


for key, model_type, hidden, parts, extra_parts, C in LM_HEADS:
    print(f'[{key}] {model_type} hidden {hidden} parts {parts + extra_parts}', flush=True)
    cache = extract_parts(key, parts + extra_parts, decoder_loader(model_type, hidden, max(L for L, _ in parts + extra_parts) + 1))
    if G_TE is None:
        G_TE = mixed_groups(TE, cache[f'L{GROUP_LAYER}_mean'][1])
        GROUP_ROWS = eligible(G_TE)
        describe_groups('test groups (content + GLM L24)', G_TE)
        if SMOKE:
            same_g, same_p = G_TE[:, None] == G_TE[None, :], test['prompt_id'].to_numpy(dtype=object)[:, None] == test['prompt_id'].to_numpy(dtype=object)[None, :]
            off = ~np.eye(len(TE), dtype=bool)
            print(f'  smoke grouping: pair precision {(same_g & same_p & off).sum() / max((same_g & off).sum(), 1):.2f} '
                  f'recall {(same_g & same_p & off).sum() / max((same_p & off).sum(), 1):.2f}', flush=True)
    CACHED_PARTS[key] = cache
    plain_and_twin(key, *concat_parts(cache, parts), C)
    for name, spec in GAUSS_HEADS.items():
        if spec['src'] == key:
            gaussian_group_head(name, spec, *cache[spec['part']])
    del cache
    CACHED_PARTS.pop(key)
    gc.collect()

for key, dir_name, parts, C in ENC_HEADS:
    print(f'[{key}] {dir_name} parts {parts}', flush=True)
    cache = extract_parts(key, parts, encoder_loader(dir_name))
    plain_and_twin(key, *concat_parts(cache, parts), C)
    for name, spec in GAUSS_HEADS.items():
        if spec['src'] == key:
            gaussian_group_head(name, spec, *cache[spec['part']])
    del cache
    gc.collect()

@torch.no_grad()
def score_lora_checkpoints(trunk_spec, pool_layers, checkpoints):
    import importlib.util as iu
    _find = iu.find_spec
    iu.find_spec = lambda name, *a, **k: None if name.split('.')[0] == 'torchao' else _find(name, *a, **k)
    from peft import PeftModel
    import peft.import_utils as pi
    pi.is_torchao_available = lambda *a, **k: False
    from safetensors.torch import load_file
    model_type, H, keep = trunk_spec
    base = find_checkpoint(model_type, H)
    assert base, f'no {model_type} base checkpoint under /kaggle/input'
    cfg = AutoConfig.from_pretrained(base)
    cfg.num_hidden_layers, cfg.use_cache = keep, False
    if getattr(cfg, 'layer_types', None) is not None:
        cfg.layer_types = cfg.layer_types[:keep]
    trunk = AutoModel.from_pretrained(base, config=cfg, dtype=torch.float16, device_map='auto').eval()
    for key, marker, norms_file, anchor_file in checkpoints:
        hits = sorted(glob.glob('/kaggle/input/' + marker, recursive=True))
        if not hits:
            print(f'  SKIP {key}: checkpoint not attached ({marker})', flush=True); continue
        ck = pathlib.Path(os.path.dirname(hits[0]))
        print(f'[{key}] {ck}', flush=True)
        tok = AutoTokenizer.from_pretrained(ck / 'tokenizer')
        tok.padding_side = 'right'
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token
        lm = PeftModel.from_pretrained(trunk, ck / 'adapter').eval()
        state = load_file(str(ck / norms_file))
        norms = torch.nn.ModuleList([torch.nn.LayerNorm(H) for _ in pool_layers])
        norms.load_state_dict({k[len('norms.'):]: v for k, v in state.items() if k.startswith('norms.')})
        ro = np.load(ck / 'readouts.npz')
        assert list(ro['classes']) == LABELS and list(ro['pool_layers']) == pool_layers
        dev = lm.get_input_embeddings().weight.device
        norms = norms.to(dev).float()

        def token_ids(texts):
            return [ids + [tok.eos_token_id] for ids in tok(list(texts), add_special_tokens=False, truncation=True, max_length=LORA_MAX_LEN)['input_ids']]

        def features(texts, bs):
            ids = token_ids(texts)
            Z = np.zeros((len(ids), H * len(pool_layers)), dtype=np.float32)
            order = np.argsort([len(t) for t in ids], kind='stable')
            batches = [order[b0:b0 + bs] for b0 in range(0, len(order), bs)]

            def work(idx):
                batch = tok.pad({'input_ids': [ids[i] for i in idx]}, return_tensors='pt').to(dev)
                with torch.no_grad():
                    with torch.autocast('cuda', dtype=torch.float16):
                        hs = lm(**batch, output_hidden_states=True, use_cache=False).hidden_states
                    m = batch['attention_mask'].unsqueeze(-1).float()
                    m[:, 0] = 0
                    return torch.cat([nm((hs[L].float() * m.to(hs[L].device)).sum(1).to(dev) / m.sum(1).to(dev))
                                      for L, nm in zip(pool_layers, norms)], -1).cpu().numpy()

            def put(idx, res):
                Z[idx] = res

            pipelined(batches, work, put, lambda idx: put(idx, work(idx)), workers=2 if spans_devices(lm) else 1, tag='lora ' + key, log=print)
            return Z

        if (ck / anchor_file).exists():
            saved = np.load(ck / anchor_file)['features'].astype(np.float32)
            ours = features(TR[:3], 3)
            rel = float(np.abs(ours - saved).mean() / (np.abs(saved).mean() + 1e-9))
            print(f'  parity with the checkpoint anchors: max abs {float(np.abs(ours - saved).max()):.4f}, mean rel {rel:.5f}', flush=True)
            assert rel < 0.02, f'{key}: our features do not match the checkpoint'
        Z = features(TE, LORA_BS)
        add_member(key, softmax_rows(Z @ ro['plain_coef'].T + ro['plain_intercept']))
        if 'claude_coef' in ro.files:
            lc = Z @ ro['claude_coef'].T + ro['claude_intercept']
            GATES['claude_lora'] = lc - np.log(np.exp(lc - lc.max(1, keepdims=True)).sum(1, keepdims=True)) - lc.max(1, keepdims=True)
        if key in LORA_TWIN_W and GROUP_ROWS.any():
            Zc = group_centre_dense(Z[GROUP_ROWS], G_TE[GROUP_ROWS])
            add_member(key + '_twin', softmax_rows(Zc @ ro['center_coef'].T + ro['center_intercept']), rows=GROUP_ROWS, weight=LORA_TWIN_W[key])
        trunk = lm.unload()
        del lm, Z
        gc.collect()
        torch.cuda.empty_cache()
    del trunk
    gc.collect()
    torch.cuda.empty_cache()


if not SMOKE:
    for trunk_spec, pool_layers, checkpoints in LORA_RUNS:
        score_lora_checkpoints(trunk_spec, pool_layers, checkpoints)

class TokenAttention(torch.nn.Module if torch is not None else object):
    def __init__(self, k, h=4096):
        super().__init__()
        self.drop, self.proj, self.score, self.out = torch.nn.Dropout(0.2), torch.nn.Linear(h, 64), torch.nn.Linear(64, 1), torch.nn.Linear(128, k)

    def forward(self, x, m):
        h = torch.tanh(self.proj(self.drop(x)))
        a = torch.softmax(self.score(h).squeeze(-1).masked_fill(~m, -1e4), -1).unsqueeze(-1)
        mf = m.unsqueeze(-1).float()
        return self.out(self.drop(torch.cat([(h * mf).sum(1) / mf.sum(1).clamp(min=1), (a * h).sum(1)], -1)))


@torch.no_grad()
def token_attention_probs(texts, head_file, backbone, bs=8):
    hits = sorted(glob.glob('/kaggle/input/**/' + head_file, recursive=True), key=lambda _p, _m={"glm53-iq3xxs-segs-a-mirror": "/kaggle/input/datasets/jamesliebel/glm53-iq3xxs-segs-a-mirror", "glm53-iq3xxs-segs-b-mirror": "/kaggle/input/datasets/jamesliebel/glm53-iq3xxs-segs-b-mirror", "glm53-flash-iq3xxs-segs-mirror": "/kaggle/input/datasets/jamesliebel/glm53-flash-iq3xxs-segs-mirror", "ntl-olmo2-native24-final-inference-assets": "/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets", "ntl-spectral-style-heads-paired-4404": "/kaggle/input/datasets/kirilltym/ntl-spectral-style-heads-paired-4404", "nlm-enc-v5": "/kaggle/input/datasets/dhruvtibarewal/nlm-enc-v5", "qwen35-9b-safetensors": "/kaggle/input/datasets/bbucxi/qwen35-9b-safetensors", "ntl-glm-likelihood-train": "/kaggle/input/notebooks/estopadilla/ntl-glm-likelihood-train"}: next((_m[_s] + _p[len(_s) + 14:] for _s in _m if _p.startswith("/kaggle/input/" + _s + "/")), _p))
    assert hits, f'training-kernel output holding {head_file} not attached'
    d = torch.load(hits[0], map_location='cpu')
    assert list(d['classes']) == LABELS
    decoder = d.get('decoder', True)
    path = find_checkpoint('qwen3_5') if backbone == 'qwen3_5' else find_checkpoint(None, dir_name='deberta-v3-large')
    assert path, f'backbone {backbone} not attached'
    cfg = AutoConfig.from_pretrained(path)
    if decoder:
        tc = getattr(cfg, 'text_config', None) or cfg
        tc.num_hidden_layers = d['keep']
        if getattr(tc, 'layer_types', None):
            tc.layer_types = tc.layer_types[:d['keep']]
        lm = AutoModel.from_pretrained(path, config=cfg, dtype=torch.float16, device_map='auto').eval()
    else:
        lm = AutoModel.from_pretrained(path, dtype=torch.float32).to(DEV).eval()
    tok = AutoTokenizer.from_pretrained(path)
    tok.padding_side = 'right'
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    H = d.get('hidden', 4096)
    net = TokenAttention(K, H)
    net.load_state_dict(d['state_dict'])
    dev = lm.get_input_embeddings().weight.device
    net, mu, sd = net.to(dev).eval(), d['mu'].float().to(dev), d['sd'].float().to(dev)
    ids = tok(list(texts), add_special_tokens=not decoder, truncation=True, max_length=d['max_len'])['input_ids']
    P = np.zeros((len(texts), K))
    order = np.argsort([len(t) for t in ids])
    for b0 in range(0, len(order), bs):
        idx = order[b0:b0 + bs]
        batch = tok.pad({'input_ids': [ids[i] for i in idx]}, return_tensors='pt').to(dev)
        out = lm(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'], output_hidden_states=True)
        hs = (out.hidden_states if getattr(out, 'hidden_states', None) is not None else out.language_model_outputs.hidden_states)[d['layer']]
        n = batch['attention_mask'].sum(1)
        T = max(int(n.max()) - 1, 1)
        x = torch.zeros(len(idx), T, H, device=dev)
        m = torch.zeros(len(idx), T, dtype=torch.bool, device=dev)
        for r in range(len(idx)):
            k = int(n[r]) - 1
            if k > 0:
                x[r, :k] = ((torch.nan_to_num(hs[r, 1:k + 1].to(dev).float()).half().float() - mu) / sd).clamp(-8, 8)
                m[r, :k] = True
        P[idx] = torch.softmax(net(x, m), -1).cpu().numpy()
        del hs, out, batch
    del lm
    gc.collect()
    torch.cuda.empty_cache()
    return P


@torch.no_grad()
def ultrabert_probs(texts):
    from transformers import AutoModelForSequenceClassification
    folds = None
    for root in glob.glob('/kaggle/input/**/ultrabert-380m-ntl', recursive=True):
        fs = [f for f in sorted(glob.glob(os.path.join(root, 'fold*'))) if os.path.exists(os.path.join(f, 'config.json'))
              and json.load(open(os.path.join(f, 'config.json'))).get('model_type') == 'ultrabert']
        if len(fs) == 5:
            folds = fs
            break
    assert folds, 'ultrabert-380m-ntl (5 folds) not attached'
    tok = AutoTokenizer.from_pretrained(folds[0], trust_remote_code=True)
    ids = tok(texts, truncation=True, max_length=512)['input_ids']
    order = np.argsort([len(e) for e in ids])
    P = np.zeros((len(texts), K))
    for fp in folds:
        em = AutoModelForSequenceClassification.from_pretrained(fp, trust_remote_code=True, dtype=torch.float16, attn_implementation='sdpa').to(DEV).eval()
        cols = [LAB2I[em.config.id2label[i]] for i in range(len(em.config.id2label))]
        for b0 in range(0, len(order), 32):
            idx = order[b0:b0 + 32]
            b = tok.pad({'input_ids': [ids[j] for j in idx]}, return_tensors='pt').to(DEV)
            P[np.ix_(idx, cols)] += em(**b).logits.float().softmax(-1).cpu().numpy() / len(folds)
        del em
        gc.collect()
        torch.cuda.empty_cache()
    return P


if not SMOKE:
    _available_tokattn = []
    for _key, _head_file, _backbone in TOKATTN_HEADS:
        _head_hits = glob.glob('/kaggle/input/**/' + _head_file, recursive=True)
        _base_hit = find_checkpoint('qwen3_5') if _backbone == 'qwen3_5' else find_checkpoint(None, dir_name='deberta-v3-large')
        if _head_hits and _base_hit:
            _available_tokattn.append((_key, _head_file, _backbone))
        else:
            print(f'  OPTIONAL HEAD SKIPPED {_key}: head={bool(_head_hits)} backbone={bool(_base_hit)}', flush=True)
    TOKATTN_HEADS = _available_tokattn
    for key, head_file, backbone in TOKATTN_HEADS:
        if TINY:
            acc = float(np.mean(token_attention_probs(TR[:40], head_file, backbone).argmax(1) == y[:40]))
            print(f'  {key} in-sample check on 40 training rows: acc {acc:.2f}', flush=True)
            assert acc > 0.5, f'{key} does not reproduce its training fit'
        add_member(key, token_attention_probs(TE, head_file, backbone))
    add_member('ultrabert', ultrabert_probs(TE))


def _np_logsoftmax(z):
    from scipy.special import logsumexp as _lse_np
    z = np.asarray(z, dtype=np.float64)
    return z - _lse_np(z, axis=1, keepdims=True)

def _apply_conditional_family_gate(gate_key, family, lam, scores_are_logprob=False):
    score = GATES.get(gate_key)
    if score is None:
        print(f'  family gate skipped {gate_key}: no fitted scores', flush=True)
        return
    idx = FAMILY_IDX[family]
    score = np.asarray(score, dtype=np.float64)
    if score.shape != (len(TE), len(idx)):
        print(f'  family gate skipped {gate_key}: shape {score.shape} expected {(len(TE), len(idx))}', flush=True)
        return
    if scores_are_logprob:
        gate_lp = _np_logsoftmax(score)
    else:
        score = score - score.mean(axis=1, keepdims=True)
        scale = np.sqrt(np.mean(score * score, axis=1, keepdims=True))
        score = score / np.maximum(scale, 0.25)
        gate_lp = _np_logsoftmax(score)
    base = LOGIT[:, idx].copy()
    base_lp = _np_logsoftmax(base)
    base_p = np.exp(base_lp)
    sp = np.sort(base_p, axis=1)
    margin = sp[:, -1] - sp[:, -2]
    confidence_gate = np.clip((0.45 - margin) / 0.45, 0.0, 1.0)
    strength = min(0.45, float(lam) / (1.0 + float(lam))) * confidence_gate
    mixed_lp = _np_logsoftmax((1.0 - strength[:, None]) * base_lp + strength[:, None] * gate_lp)
    from scipy.special import logsumexp as _lse_np
    family_mass = _lse_np(base, axis=1)
    LOGIT[:, idx] = family_mass[:, None] + mixed_lp
    print(f'  family gate applied {gate_key}: mean strength={strength.mean():.4f}, active rows={int((strength > 0).sum())}', flush=True)

if not SMOKE:
    _apply_conditional_family_gate('claude', 'claude', GATE_LAM['claude_style'])
    _apply_conditional_family_gate('claude_lora', 'claude', GATE_LAM['claude_lora'], scores_are_logprob=True)
    for _fam in ('gpt', 'gemini', 'glm'):
        _apply_conditional_family_gate(_fam, _fam, GATE_LAM['family'])

np.save('/kaggle/working/comp_logit.npy', LOGIT.astype(np.float32))
json.dump({'labels': LABELS, 'ids': test['id'].astype(str).tolist(), 'seconds': time.time() - T0,
           'family_gates_applied': bool(not SMOKE)}, open('/kaggle/working/comp_meta.json', 'w'))
print(f'COMP STACK DONE {time.time() - T0:.0f}s', flush=True)

In [ ]:
# Subprocess Streaming Runner: real-time stdout/stderr capture with exit-code checking
def run(cmd, env=None):
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env, bufsize=1)
    for line in proc.stdout:
        if not re.search(r'Warning|warn', line):
            print(line, end='')
    return proc.wait()

In [ ]:
# Stage-1 Execution Config: 5 L24 LoRA backbones + DeBERTa + Native Q8 + GSQ-27B + Sinkhorn (or fast commit smoke check)
LAYERS = [2, 4, 6, 8, 12, 16, 20, 24]
MODELS = [
    {"name": "q14v5", "type": "l24", "path": B14, "adapters": AQ14V5, "keep": 27, "layers": LAYERS, "w": 3.0, "bs": 8, "max_len": 384, "twin": 0.35, "gauss": 0.70},
    {"name": "q8v5", "type": "l24", "path": B8, "adapters": AQ8V5, "keep": 24, "layers": LAYERS, "w": 2.0, "bs": 16, "max_len": 384, "twin": 0.35, "gauss": 0.44, "dp": True},
    {"name": "ll8v5", "type": "l24", "path": BL, "adapters": ALL8V5, "keep": 24, "layers": LAYERS, "w": 1.0, "bs": 16, "max_len": 384, "twin": 0.35, "dp": True},
    {"name": "n14b_clean", "type": "l24", "path": B14, "adapters": [OQ4, N14B], "adapter_weights": [0.72, 0.28], "keep": N14BC['keep'], "layers": N14BC['layers'],
     "w": 2.5, "bs": 8, "max_len": N14BC['max_len'], "twin": 0.35, "no_final_norm": True, "eos_id": N14BC['eos_id']},
    {"name": "q25_14", "type": "l24", "path": B25_14, "adapters": [Q25_14], "keep": Q25_14C['keep'], "layers": Q25_14C['layers'], "w": 1.0, "bs": 8,
     "max_len": Q25_14C['max_len'], "twin": 0.35, "no_final_norm": True, "eos_id": Q25_14C['eos_id']},
    {"name": "deb", "type": "enc", "ckpt": DEB, "w": 1.0},
]
OUT = '/kaggle/working/submission.parquet'
COMP_SEC = 0.0  # Reserve 30+ min GPU runtime for PB-115 GLM likelihood
cmd = [sys.executable, 'infer.py', '--data', DATA, '--models', json.dumps(MODELS),
       '--prefit_core', PREFIT, '--tfidf_w', '0', '--distort_w', '0', '--svc_w', '0',
       '--adapt_w', '1.0', '--adapt_thr', '0.15', '--sinkhorn', '0.30', '--sinkhorn_target_mix', '0.35',
       '--native_dir', NATIVE, '--native_base', NATIVE_BASE, '--native_share', '0.40', '--native_cent', '1',
       '--native_deadline', str(NB_T0 + RUN_LIMIT_S),
       '--group_embed_base', NATIVE_BASE, '--group_embed_assets', NATIVE, '--group_sim', '0.70',
       '--gsq_dir', GSQ_ASSETS, '--gsq_share', '0.0325', '--gsq_deadline', str(NB_T0 + RUN_LIMIT_S - 9000), '--gsq_timeout', '5400',
       '--comp_path', '/kaggle/working/comp_logit.npy', '--comp_w', '0.0', '--out', OUT, '--save_final_logits', FINAL_LOGITS]
if FAST_COMMIT:
    ids = pd.read_csv(os.path.join(DATA, 'test.csv')).id
    pd.DataFrame({'id': ids, 'model': 'gpt-5.5'}).to_parquet(OUT, index=False)
    sys.path.insert(0, os.getcwd())
    from ntl660_readout import core_votes
    smoke = core_votes(PREFIT, pd.read_csv(os.path.join(DATA, 'test.csv')).text.fillna('').tolist())
    assert set(smoke) == {'svc4', 'tfidf', 'distort'} and all(len(v) == len(ids) and np.isfinite(v).all() for v in smoke.values())
    print('prefit core votes smoke ok', {k: v.shape for k, v in smoke.items()})
else:
    assert run(cmd, env=dict(os.environ, NTL_PIPE_WORKERS='1')) == 0, 'ensemble worker failed'

In [ ]:
# Stage-1 Integrity Check: verify submission schema, non-null predictions & row count
submission = pd.read_parquet(OUT)
test_ids = pd.read_csv(os.path.join(DATA, 'test.csv')).id
assert len(submission) == len(test_ids) and set(submission.id) == set(test_ids)
assert list(submission.columns) == ['id', 'model'] and submission.id.is_unique and submission.model.notna().all()
print(submission.model.value_counts().to_string())
for path in ['/kaggle/working/open_pool.parquet', '/kaggle/working/extra_matched.parquet']:
    if os.path.exists(path):
        os.remove(path)
shutil.rmtree('/kaggle/working/deberta_v3_large', ignore_errors=True)
print('total', round(time.time() - NB_T0), 's')


## 4. Stage-2 User-Conditioned GLM-5.3 & Flash Likelihood Engine (`ll_prefixB / L160`, `W = 1.5`)
Transparently stages `segio.py`, `features.py`, `run_pipeline.py`, `v1_stage.py`, and the refitted `ll_model.json` head directly into `/kaggle/temp/v1`, resolves the `llama.cpp b12247` (`ntl-glm-prefix-ab`) binary with CUDA 13 libraries and all 25 pinned IQ3_XXS GGUF segments, runs a real GPU smoke test on commit, and executes priority-ordered shrink-safe likelihood scoring during competition reruns.

In [ ]:
# Initialize Stage-2 GLM likelihood directories and robust shrink-on-overrun configuration
from pathlib import Path

V1_DIR = Path('/kaggle/temp/v1')
V1_DIR.mkdir(parents=True, exist_ok=True)
V1_OUT = Path('/kaggle/working/v1')
V1_OUT.mkdir(parents=True, exist_ok=True)

V1_ENABLED = True
V1_DEADLINE_H = 8.8   # Hard stop before Kaggle's 9.0h wall-clock limit
V1_ROBUST = True      # Enables --priority-order and --shrink on runtime overrun
V1_PACK, V1_UBATCH_DIV, V1_SEQ_BATCH, V1_PREFETCH = False, 1, 8, False
# Calibrated GLM runtime clock: refitted on dual-T4 timing runs (frees ~22 min safety slack, expanding GLM coverage under 8.8h deadline)
V1_COST = dict(fixed_s_flash=638.4, row_s_flash=0.01172, fixed_s_glm53=1325.7, row_s_glm53=0.019489)
V1_RERUN = os.getenv('KAGGLE_IS_COMPETITION_RERUN') is not None
V1_TEST_CSV = os.path.join(DATA, 'test.csv')

In [ ]:
%%writefile /kaggle/temp/v1/segio.py
"""Binary Packet Reader/Writer for Segmented GGUF Causal LM Tokens (`segio`).

Reads and writes `NTLT`/`NTLK`/`NTLS` binary streams exchanged with `ntl-segrun`."""
import struct

import numpy as np


def write_texts(path, texts):
    with open(path, 'wb') as f:
        f.write(b'NTLT')
        f.write(struct.pack('<I', len(texts)))
        for t in texts:
            b = t.encode('utf-8')
            f.write(struct.pack('<I', len(b)))
            f.write(b)


def write_tokens(path, seqs):
    """seqs: list of (ids, n_prefix)."""
    with open(path, 'wb') as f:
        f.write(b'NTLK')
        f.write(struct.pack('<I', len(seqs)))
        for ids, n_prefix in seqs:
            ids = np.asarray(ids, dtype='<i4')
            f.write(struct.pack('<II', len(ids), n_prefix))
            f.write(ids.tobytes())


def read_tokens(path):
    out = []
    with open(path, 'rb') as f:
        assert f.read(4) == b'NTLK'
        (n,) = struct.unpack('<I', f.read(4))
        for _ in range(n):
            nt, npre = struct.unpack('<II', f.read(8))
            ids = np.frombuffer(f.read(4 * nt), dtype='<i4')
            out.append((ids, npre))
    return out


def read_scores(path):
    """List of float32 arrays [n_scored, 4]: logprob, entropy, E[logp^2], rank."""
    out = []
    with open(path, 'rb') as f:
        assert f.read(4) == b'NTLP'
        n, nf = struct.unpack('<II', f.read(8))
        for _ in range(n):
            (ns,) = struct.unpack('<I', f.read(4))
            out.append(np.frombuffer(f.read(4 * ns * nf), dtype='<f4').reshape(ns, nf))
    return out


def read_state(path):
    with open(path, 'rb') as f:
        assert f.read(4) == b'NTLS'
        dt, w, nt, nr = struct.unpack('<IIIQ', f.read(20))
        dtype = {0: '<f4', 1: '<f2'}.get(dt)
        if dtype is None:  # bf16
            raw = np.frombuffer(f.read(2 * w * nr), dtype='<u2').astype('<u4') << 16
            return raw.view('<f4').reshape(nr, w)
        return np.frombuffer(f.read(np.dtype(dtype).itemsize * w * nr), dtype=dtype).reshape(nr, w).astype('f4')


In [ ]:
%%writefile /kaggle/temp/v1/features.py
"""User-Conditioned Autoregressive Perplexity Distributional Features (`features`).

Computes 24 per-model token likelihood, entropy, Fast-DetectGPT (`fdg`),
top-k rank mass, and BPC statistics from `ntl-segrun` score tensors."""
import numpy as np
import pandas as pd


def one(s, n_chars):
    lp, h, m2, rank = s[:, 0].astype(np.float64), s[:, 1].astype(np.float64), s[:, 2].astype(np.float64), s[:, 3]
    n = len(lp)
    var = np.maximum(m2 - h * h, 1e-9)
    half = max(n // 2, 1)
    q = np.quantile(lp, [0.05, 0.10, 0.25, 0.50, 0.75, 0.90]) if n else np.zeros(6)
    return {
        'n_tok': n,
        'lp_mean': lp.mean(),
        'lp_std': lp.std(),
        'lp_q05': q[0], 'lp_q10': q[1], 'lp_q25': q[2], 'lp_q50': q[3], 'lp_q75': q[4], 'lp_q90': q[5],
        'lp_min': lp.min(),
        'ent_mean': h.mean(),
        'ent_std': h.std(),
        # Fast-DetectGPT analytic curvature: how far the text's log-likelihood sits above the model's own expectation
        'fdg': (lp + h).sum() / np.sqrt(var.sum()),
        'surprise_mean': (-lp - h).mean(),          # observed surprisal minus expected surprisal
        'z_mean': ((lp + h) / np.sqrt(var)).mean(),
        'logrank_mean': np.log1p(rank).mean(),
        'top1': (rank == 0).mean(),
        'top5': (rank < 5).mean(),
        'top20': (rank < 20).mean(),
        'rank_ge100': (rank >= 100).mean(),
        'lp_mean_first': lp[:half].mean(),
        'lp_mean_second': lp[half:].mean() if n > half else lp.mean(),
        'bpc': -lp.sum() / max(n_chars, 1) / np.log(2),
        'tok_per_char': n / max(n_chars, 1),
    }


def text_features(scores, texts):
    return pd.DataFrame([one(s, len(t)) for s, t in zip(scores, texts)])


In [ ]:
%%writefile /kaggle/temp/v1/run_pipeline.py
"""Shrink-Safe Segmented Layer-Streaming Executor for Dual-T4 GGUF Inference.

Streams 3-bit IQ3_XXS GLM-5.3-Flash and GLM-5.3 shards chunk-by-chunk with
user-turn prefix formatting and dynamic `--shrink-min` budget projection."""
import argparse
import glob
import json
import os
import re
import shutil
import struct
import subprocess
import sys
import threading
import time

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import features  # noqa: E402
import segio  # noqa: E402

CHAT_PREFIX = '[gMASK]<sop><|user|><|assistant|><think></think>'


def seg_index(path):
    m = re.search(r'seg(\d+)-of-(\d+)', os.path.basename(path))
    assert m, path
    return int(m.group(1)), int(m.group(2))


def gguf_kv_u32(path, key):
    """Read one uint32 metadata value from a GGUF header without the gguf package."""
    with open(path, 'rb') as f:
        assert f.read(4) == b'GGUF'
        _ver, _nt, nkv = struct.unpack('<IQQ', f.read(20))

        def rstr():
            (n,) = struct.unpack('<Q', f.read(8))
            return f.read(n).decode('utf-8', 'replace')

        sizes = {0: 1, 1: 1, 2: 2, 3: 2, 4: 4, 5: 4, 6: 4, 7: 1, 10: 8, 11: 8, 12: 8}

        def skip(t):
            if t == 8:
                rstr()
            elif t == 9:
                it, n = struct.unpack('<IQ', f.read(12))
                if it in sizes:
                    f.seek(sizes[it] * n, 1)
                else:
                    for _ in range(n):
                        skip(it)
            else:
                f.seek(sizes[t], 1)

        for _ in range(nkv):
            k = rstr()
            (t,) = struct.unpack('<I', f.read(4))
            if k == key:
                assert t == 4, (key, t)
                return struct.unpack('<I', f.read(4))[0]
            skip(t)
    return None


class BudgetExceeded(RuntimeError):
    """The time budget (--budget-s) is not enough: stop and let the caller fall back."""


class EarlyFailure(RuntimeError):
    """ntl-segrun failed before writing any output (model load or context reservation, e.g. out of GPU memory)."""


_PREFETCHED = {}


def prefetch(path):
    """Read a file once in a daemon thread (page cache warm-up for the next mmap); each file at most once."""
    path = os.path.realpath(path)
    if path in _PREFETCHED:
        return
    info = _PREFETCHED[path] = {'s': None}

    def read():
        t0 = time.time()
        with open(path, 'rb', buffering=0) as f:
            try:
                os.posix_fadvise(f.fileno(), 0, 0, os.POSIX_FADV_SEQUENTIAL)
            except (AttributeError, OSError):
                pass
            buf = bytearray(16 << 20)
            while f.readinto(buf):
                pass
        info['s'] = time.time() - t0
    threading.Thread(target=read, daemon=True).start()


def run(cmd, log, timeout=None, on_ready=None):
    """Run one ntl-segrun call; on_ready() is called when it prints "context ready" (the model is loaded)."""
    t0 = time.time()
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors='replace')
    expired = threading.Event()

    def kill():
        expired.set()
        p.kill()
    timer = threading.Timer(timeout, kill) if timeout is not None else None
    if timer is not None:
        timer.daemon = True
        timer.start()
    lines = []
    for line in p.stdout:
        lines.append(line)
        if on_ready is not None and 'context ready' in line:
            on_ready()
            on_ready = None
    p.wait()
    if timer is not None:
        timer.cancel()
    out = ''.join(lines)
    if expired.is_set():
        with open(log, 'a') as lf:
            lf.write('$ ' + ' '.join(cmd) + '\n' + out + '\n[killed: timeout]\n')
        raise BudgetExceeded(f'timeout after {timeout:.0f} s: {" ".join(cmd)}')
    with open(log, 'a') as lf:
        lf.write('$ ' + ' '.join(cmd) + '\n' + out)
    dt = time.time() - t0
    if p.returncode != 0:
        print('\n'.join(out.splitlines()[-40:]), flush=True)
        err = EarlyFailure if 'context ready' not in out else RuntimeError
        raise err(f'failed ({p.returncode}): {" ".join(cmd)}')
    return dt


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--bin', required=True)
    ap.add_argument('--segs', required=True, help='glob of the segment files')
    ap.add_argument('--csv', required=True)
    ap.add_argument('--text-col', default='text')
    ap.add_argument('--id-col', default='id')
    ap.add_argument('--out', required=True)
    ap.add_argument('--tag', required=True)
    ap.add_argument('--work', default='/tmp/ntl')
    ap.add_argument('--prefix', default=CHAT_PREFIX)
    ap.add_argument('--max-text-tokens', type=int, default=320)
    ap.add_argument('--seq-batch', type=int, default=8)
    ap.add_argument('--state-dtype', default='auto', help='auto | f32 | f16 | bf16')
    ap.add_argument('--threads', type=int, default=min(4, os.cpu_count() or 4))
    ap.add_argument('--limit', type=int, default=-1)
    ap.add_argument('--disk-frac', type=float, default=0.8, help='share of the free work disk the state may use')
    ap.add_argument('--deadline-s', type=float, default=0, help='stop before starting a chunk after this many seconds')
    ap.add_argument('--pack', action='store_true', help='sort by length and pack padded decodes (ntl-segrun --pack)')
    ap.add_argument('--ubatch-div', type=int, default=1, help='n_ubatch = n_batch / P (pipeline parallel on 2 GPUs)')
    ap.add_argument('--budget-s', type=float, default=0, help='hard time limit of the whole call (0 = none)')
    ap.add_argument('--prefetch', action='store_true', help='read the next segment while the current one computes')
    ap.add_argument('--prefetch-after', default='', help='with --prefetch: file to read during the last segment')
    ap.add_argument('--shrink-min', type=int, default=0,
                    help='S14: on a budget overrun keep the first m >= N texts instead of stopping (0 = stop, as S13)')
    ap.add_argument('--shrink-fixed', type=float, default=0.5,
                    help='share of a segment time that does not scale with the rows (projection when shrinking)')
    ap.add_argument('--user-template', default='Write a single paragraph of about {n_words} words, with no title or headings.',
                    help="with '{user}' in --prefix: the user turn of each text ({n_words} = its word count)")
    a = ap.parse_args()
    try:
        main_run(a)
    except BudgetExceeded as e:
        print(f'[{a.tag}] BUDGET EXCEEDED: {e}', flush=True)
        sys.exit(3)


def user_prefix_tokens(a, segrun, tok_model, texts, tx, tk, log):
    """A prefix with a user turn that depends on the text: --prefix holds '{user}', replaced for each text by
    --user-template formatted with n_words (the text's word count), e.g. prvsiyan's GLM prefix
    '[gMASK]<sop><|system|>Reasoning Effort: Low<|user|>{user}<|assistant|><think></think>'.
    ntl-segrun tokenize takes one prefix for all texts, so the tokens are put together from two calls (exact: the
    special tokens split the string before BPE, so each plain-text piece is tokenized on its own):
      full = tokenize(texts, prefix with an empty user turn) = H + T + text   (H = up to '{user}', T = the rest)
      user = tokenize(user turns, prefix H)                    = H + U_i
    and each text gets H + U_i + T + text, with n_prefix = len(H + U_i + T). Checked on the first 3 texts
    against a tokenization with their whole prefix; if that check fails (a tokenizer where the pieces are not
    independent), the exact fallback tokenizes each group of texts that share the user turn with its whole prefix."""
    users = [a.user_template.format(n_words=len(t.split())) for t in texts]
    try:
        return _user_prefix_pieces(a, segrun, tok_model, texts, users, tx, tk, log)
    except AssertionError as e:
        # exact fallback: one tokenize call per distinct user turn (texts with the same word count share it)
        print(f'[{a.tag}] user-turn prefix: piecewise tokens rejected ({e}); tokenizing by groups', flush=True)
    groups = {}
    for i, u in enumerate(users):
        groups.setdefault(u, []).append(i)
    seqs = [None] * len(texts)
    for j, (u, idx) in enumerate(groups.items()):
        gx, gk = tk + f'.g{j}.bin', tk + f'.g{j}_tok.bin'
        segio.write_texts(gx, [texts[i] for i in idx])
        run([segrun, 'tokenize', '--model', tok_model, '--texts', gx, '--out', gk, '--prefix', a.prefix.replace('{user}', u),
             '--max-text-tokens', str(a.max_text_tokens)], log)
        for i, sq in zip(idx, segio.read_tokens(gk)):
            seqs[i] = sq
        os.remove(gx); os.remove(gk)
    print(f'[{a.tag}] user-turn prefix: {len(groups)} groups, prefix {min(s[1] for s in seqs)}-{max(s[1] for s in seqs)} '
          f'tokens; e.g. {users[0]!r}', flush=True)
    return seqs


def _user_prefix_pieces(a, segrun, tok_model, texts, users, tx, tk, log):
    head = a.prefix.split('{user}')[0]
    run([segrun, 'tokenize', '--model', tok_model, '--texts', tx, '--out', tk, '--prefix', a.prefix.replace('{user}', ''),
         '--max-text-tokens', str(a.max_text_tokens)], log)
    full = segio.read_tokens(tk)
    ux, uk = tk + '.user_texts.bin', tk + '.user_tok.bin'
    segio.write_texts(ux, users)
    run([segrun, 'tokenize', '--model', tok_model, '--texts', ux, '--out', uk, '--prefix', head,
         '--max-text-tokens', '100000'], log)
    user = segio.read_tokens(uk)
    n_h = user[0][1]
    seqs = []
    for (f_ids, f_pre), (u_ids, u_pre) in zip(full, user):
        assert u_pre == n_h and f_pre >= n_h and list(f_ids[:n_h]) == list(u_ids[:n_h]), 'prefix head differs'
        seqs.append((np.concatenate([u_ids, f_ids[n_h:]]).astype('<i4'), len(u_ids) + f_pre - n_h))
    for i in range(min(3, len(texts))):
        one_x, one_k = tk + f'.check{i}.bin', tk + f'.check{i}_tok.bin'
        segio.write_texts(one_x, [texts[i]])
        run([segrun, 'tokenize', '--model', tok_model, '--texts', one_x, '--out', one_k,
             '--prefix', a.prefix.replace('{user}', users[i]), '--max-text-tokens', str(a.max_text_tokens)], log)
        ids, pre = segio.read_tokens(one_k)[0]
        assert pre == seqs[i][1] and list(ids) == list(seqs[i][0]), f'user-prefix tokens differ for text {i}'
    print(f'[{a.tag}] user-turn prefix: head {n_h} tokens, prefix {min(s[1] for s in seqs)}-{max(s[1] for s in seqs)} '
          f'tokens (pieces checked on {min(3, len(texts))} texts); e.g. {users[0]!r}', flush=True)
    return seqs


def project_rest(a, seg_t, seg_rows, rows0, rows_kept, rest):
    """Projected time of the `rest` segments (the last counted double, as in the S13 projection) if only the first
    m texts are kept, for every m (array, increasing). Per-segment time t = unit * (phi + (1 - phi) * rows / rows0),
    unit = mean over the segments done (each normalised by the rows it ran with)."""
    phi = a.shrink_fixed
    unit = float(np.mean([t / (phi + (1 - phi) * r / rows0) for t, r in zip(seg_t, seg_rows)]))
    return unit * (phi + (1 - phi) * np.cumsum(rows_kept) / rows0) * (rest + (1 if rest > 0 else 0))


def main_run(a):

    t_start = time.time()

    def left():
        return None if not a.budget_s else a.budget_s - (time.time() - t_start)

    os.makedirs(a.out, exist_ok=True)
    os.makedirs(a.work, exist_ok=True)
    log = os.path.join(a.out, f'{a.tag}_run.log')
    segrun = os.path.join(a.bin, 'ntl-segrun')

    segs = sorted(glob.glob(a.segs), key=lambda p: seg_index(p)[0])
    assert segs, a.segs
    n_seg = seg_index(segs[0])[1]
    assert [seg_index(p)[0] for p in segs] == list(range(n_seg)), segs
    ranges = [(gguf_kv_u32(p, 'ntl.layer_start'), gguf_kv_u32(p, 'ntl.layer_end')) for p in segs]
    print(f'[{a.tag}] {n_seg} segments, layers {ranges}', flush=True)
    for (s0, e0), (s1, _e1) in zip(ranges, ranges[1:]):
        assert e0 == s1, ranges

    df = pd.read_csv(a.csv)
    if a.limit > 0:
        df = df.iloc[:a.limit]
    texts = df[a.text_col].astype(str).tolist()
    tx = os.path.join(a.work, f'{a.tag}_texts.bin')
    tk = os.path.join(a.work, f'{a.tag}_tok.bin')
    segio.write_texts(tx, texts)
    if '{user}' in a.prefix:
        seqs = user_prefix_tokens(a, segrun, segs[0], texts, tx, tk, log)
    else:
        run([segrun, 'tokenize', '--model', segs[0], '--texts', tx, '--out', tk, '--prefix', a.prefix,
             '--max-text-tokens', str(a.max_text_tokens)], log)
        seqs = segio.read_tokens(tk)
    n_rows = np.array([len(ids) for ids, _ in seqs])
    # processing order: longest first when packing (similar lengths share a decode, little padding)
    order = np.argsort(-n_rows, kind='stable') if a.pack else np.arange(len(seqs))
    print(f'[{a.tag}] {len(seqs)} texts, {n_rows.sum()} rows, max {n_rows.max()}, prefix {seqs[0][1]}', flush=True)

    # state width: glm5-next hc*n_embd, glm-dsa n_embd (read from the first segment's metadata)
    n_embd = gguf_kv_u32(segs[0], 'glm5-next.embedding_length')
    if n_embd is not None:
        width = gguf_kv_u32(segs[0], 'glm5-next.hyper_connection.count') * n_embd
    else:
        width = gguf_kv_u32(segs[0], 'glm-dsa.embedding_length')
    free = shutil.disk_usage(a.work).free
    if a.state_dtype == 'auto':
        # exact f32 if the whole state fits in one chunk, else bf16 (half the disk, no overflow risk)
        a.state_dtype = 'f32' if n_rows.sum() * width * 4 <= free * a.disk_frac else 'bf16'
        print(f'[{a.tag}] state dtype auto -> {a.state_dtype}', flush=True)
    el = 2 if a.state_dtype in ('f16', 'bf16') else 4
    max_rows = int(free * a.disk_frac / (width * el))
    chunks, cur, cur_rows = [], [], 0
    for i in order:
        r = n_rows[i]
        if cur and cur_rows + r > max_rows:
            chunks.append(cur)
            cur, cur_rows = [], 0
        cur.append(i)
        cur_rows += r
    chunks.append(cur)
    print(f'[{a.tag}] state width {width}, {el} B/value, {free/2**30:.1f} GiB free on {a.work} -> '
          f'{len(chunks)} chunk(s)', flush=True)

    multi_gpu = False
    try:
        out = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
        multi_gpu = out.count('GPU ') > 1
    except FileNotFoundError:
        pass

    timing = {'tag': a.tag, 'n_texts': len(seqs), 'n_rows': int(n_rows.sum()), 'chunks': [], 'segments': segs,
              'ranges': ranges, 'seq_batch': a.seq_batch, 'state_dtype': a.state_dtype, 'prefix': a.prefix,
              'pack': a.pack, 'ubatch_div': a.ubatch_div, 'budget_s': a.budget_s}
    all_scores = [None] * len(seqs)
    st = os.path.join(a.work, f'{a.tag}_state.bin')
    done = 0
    can_shrink = a.shrink_min > 0 and len(chunks) == 1 and not a.pack
    if can_shrink:
        timing['shrinks'] = []
    for ci, idx in enumerate(chunks):
        if a.deadline_s and time.time() - t_start > a.deadline_s:
            print(f'[{a.tag}] deadline reached before chunk {ci}, stopping', flush=True)
            break
        ctk = os.path.join(a.work, f'{a.tag}_tok_c{ci}.bin')
        segio.write_tokens(ctk, [seqs[i] for i in idx])
        sc = os.path.join(a.work, f'{a.tag}_scores_c{ci}.bin')
        seg_t = []
        rows0 = int(n_rows[idx].sum())
        seg_rows = []   # rows of the chunk when each segment ran (they drop after a shrink)
        for k, p in enumerate(segs):
            while True:
                cmd = [segrun, 'run', '--model', p, '--tokens', ctk, '--threads', str(a.threads),
                       '--seq-batch', str(a.seq_batch), '--state-dtype', a.state_dtype]
                if multi_gpu:
                    cmd.append('--split-mid')
                if a.pack:
                    cmd.append('--pack')
                if a.ubatch_div > 1:
                    cmd += ['--ubatch-div', str(a.ubatch_div)]
                if k > 0:
                    cmd += ['--state-in', st]
                cmd += ['--out', sc if k == n_seg - 1 else st]
                if left() is not None and left() <= 0:
                    raise BudgetExceeded('no time left')
                nxt = (segs[k + 1] if k + 1 < n_seg else segs[0] if ci + 1 < len(chunks) else a.prefetch_after)
                ready = (lambda f=nxt: prefetch(f)) if a.prefetch and nxt else None
                try:
                    dt = run(cmd, log, timeout=left(), on_ready=ready)
                    break
                except EarlyFailure:
                    # nothing written yet (the input state is intact): retry with half the texts per decode
                    if a.seq_batch == 1:
                        raise
                    a.seq_batch //= 2
                    timing['seq_batch_reduced_to'] = a.seq_batch
                    print(f'[{a.tag}] segment {k+1} failed before writing: retrying with seq-batch {a.seq_batch}', flush=True)
            seg_t.append(dt)
            seg_rows.append(int(n_rows[idx].sum()))
            print(f'[{a.tag}] chunk {ci+1}/{len(chunks)} segment {k+1}/{n_seg} {os.path.basename(p)}: {dt:.0f} s '
                  f'(total {time.time()-t_start:.0f} s)', flush=True)
            if a.budget_s:
                # projection: the remaining segments of this chunk at the mean so far (the last one, with the
                # softmax over the vocabulary, counts double), plus the other chunks at this chunk's pace
                rest = n_seg - k - 1
                per = float(np.mean(seg_t))
                proj = per * (rest + (1 if rest > 0 else 0)) + per * n_seg * 1.15 * (len(chunks) - ci - 1)
                if can_shrink:   # one chunk: the same projection, aware of the rows left after a shrink
                    proj_m_all = project_rest(a, seg_t, seg_rows, rows0, n_rows[idx], rest)
                    proj = float(proj_m_all[-1])
                if proj > left():
                    if not can_shrink:
                        raise BudgetExceeded(f'projected {proj:.0f} s for the rest, {left():.0f} s left')
                    m = int((proj_m_all <= 0.9 * left()).sum())   # increasing in m: the count is the largest prefix
                    proj_m = float(proj_m_all[m - 1]) if m else float('inf')
                    if m < a.shrink_min:
                        raise BudgetExceeded(f'projected {proj:.0f} s for the rest, {left():.0f} s left; '
                                             f'not even {a.shrink_min} texts fit')
                    timing['shrinks'].append({'after_segment': k + 1, 'from': len(idx), 'to': m,
                                              'left_s': round(left()), 'proj_all_s': round(proj),
                                              'proj_kept_s': round(proj_m)})
                    print(f'[{a.tag}] SHRINK after segment {k+1}/{n_seg}: {len(idx)} -> {m} texts '
                          f'(rest projected {proj:.0f} s with all, {proj_m:.0f} s with {m}; {left():.0f} s left)',
                          flush=True)
                    idx = idx[:m]
                    segio.write_tokens(ctk, [seqs[i] for i in idx])
        for i, s in zip(idx, segio.read_scores(sc)):
            all_scores[i] = s
        done += len(idx)
        timing['chunks'].append({'n_texts': len(idx), 'seg_seconds': seg_t})
        for f in (ctk, sc, st):
            if os.path.exists(f):
                os.remove(f)

    ok = [i for i, s in enumerate(all_scores) if s is not None]
    off = np.zeros(len(ok) + 1, dtype=np.int64)
    off[1:] = np.cumsum([len(all_scores[i]) for i in ok])
    np.savez_compressed(os.path.join(a.out, f'{a.tag}_scores.npz'),
                        ids=df[a.id_col].astype(str).values[ok],
                        scores=np.concatenate([all_scores[i] for i in ok]).astype(np.float32),
                        offsets=off,
                        n_prefix=np.array([seqs[i][1] for i in ok]),
                        tokens=np.concatenate([seqs[i][0][seqs[i][1]:] for i in ok]).astype(np.int32))
    feats = features.text_features([all_scores[i] for i in ok], [texts[i] for i in ok])
    feats.insert(0, a.id_col, df[a.id_col].astype(str).values[ok])
    feats.to_csv(os.path.join(a.out, f'{a.tag}_features.csv'), index=False)
    timing['total_s'] = time.time() - t_start
    if a.prefetch:
        timing['prefetch_s'] = {os.path.basename(k): v['s'] for k, v in _PREFETCHED.items()}
    timing['n_done'] = len(ok)
    with open(os.path.join(a.out, f'{a.tag}_timing.json'), 'w') as f:
        json.dump(timing, f, indent=1)
    print(f'[{a.tag}] done: {len(ok)}/{len(seqs)} texts in {timing["total_s"]:.0f} s', flush=True)


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /kaggle/temp/v1/v1_token_features.py
"""Token-aligned differential and high-entropy likelihood features from paired GLM-5.3-Flash / GLM-5.3 scores."""
import numpy as np
import pandas as pd


def _single_extra(s: np.ndarray, prefix: str) -> dict[str, float]:
    lp = s[:, 0].astype(np.float64)
    h = s[:, 1].astype(np.float64)
    m2 = s[:, 2].astype(np.float64)
    rk = s[:, 3].astype(np.float64)
    n = len(lp)
    if n == 0:
        keys = (
            'lp_head32', 'lp_tail32', 'fdg_head32', 'fdg_tail32',
            'lp_hi_ent', 'z_hi_ent', 'top1_hi_ent', 'logrk_hi_ent',
            'lp_lo_ent', 'z_lo_ent', 'z_q10', 'z_q50', 'z_q90',
        )
        return {f'{prefix}:{k}': 0.0 for k in keys}

    var = np.maximum(m2 - h * h, 1e-9)
    sd = np.sqrt(var)
    z = (lp + h) / sd
    k32 = min(32, n)
    hi = h > 1.5
    lo = ~hi
    zq = np.quantile(z, [0.10, 0.50, 0.90])

    def _m(arr, mask, fallback):
        return float(arr[mask].mean()) if mask.any() else float(fallback)

    return {
        f'{prefix}:lp_head32': float(lp[:k32].mean()),
        f'{prefix}:lp_tail32': float(lp[-k32:].mean()),
        f'{prefix}:fdg_head32': float((lp[:k32] + h[:k32]).sum() / np.sqrt(var[:k32].sum())),
        f'{prefix}:fdg_tail32': float((lp[-k32:] + h[-k32:]).sum() / np.sqrt(var[-k32:].sum())),
        f'{prefix}:lp_hi_ent': _m(lp, hi, lp.mean()),
        f'{prefix}:z_hi_ent': _m(z, hi, z.mean()),
        f'{prefix}:top1_hi_ent': _m((rk == 0).astype(np.float64), hi, (rk == 0).mean()),
        f'{prefix}:logrk_hi_ent': _m(np.log1p(rk), hi, np.log1p(rk).mean()),
        f'{prefix}:lp_lo_ent': _m(lp, lo, lp.mean()),
        f'{prefix}:z_lo_ent': _m(z, lo, z.mean()),
        f'{prefix}:z_q10': float(zq[0]),
        f'{prefix}:z_q50': float(zq[1]),
        f'{prefix}:z_q90': float(zq[2]),
    }


def _paired_token_features(sf: np.ndarray, sg: np.ndarray) -> dict[str, float]:
    n = min(len(sf), len(sg))
    if n == 0:
        keys = (
            'dlp_mean', 'dlp_std', 'dlp_tstat', 'dlp_pos', 'dlp_big_pos', 'dlp_big_neg',
            'dlp_q10', 'dlp_q25', 'dlp_q50', 'dlp_q75', 'dlp_q90',
            'dlp_first', 'dlp_second', 'dlp_head32', 'dlp_tail32',
            'excl_t1_f', 'excl_t1_g', 'both_t1', 'excl_t1_diff',
            'excl_t5_f', 'excl_t5_g', 'excl_t5_diff',
            'dlrk_mean', 'dlrk_std', 'dlrk_tstat',
            'dz_mean', 'dz_std', 'dz_tstat', 'dz_pos',
            'dent_mean', 'dent_std', 'dent_pos',
            'dlp_hi_ent', 'dlp_hi_ent_pos', 'dz_hi_ent',
        )
        return {f'p:{k}': 0.0 for k in keys}

    sf, sg = sf[:n], sg[:n]
    lpf, hf, m2f, rkf = sf[:, 0].astype(np.float64), sf[:, 1].astype(np.float64), sf[:, 2].astype(np.float64), sf[:, 3].astype(np.float64)
    lpg, hg, m2g, rkg = sg[:, 0].astype(np.float64), sg[:, 1].astype(np.float64), sg[:, 2].astype(np.float64), sg[:, 3].astype(np.float64)

    zf = (lpf + hf) / np.sqrt(np.maximum(m2f - hf * hf, 1e-9))
    zg = (lpg + hg) / np.sqrt(np.maximum(m2g - hg * hg, 1e-9))

    dlp = lpf - lpg
    dlrk = np.log1p(rkf) - np.log1p(rkg)
    dz = zf - zg
    dent = hf - hg

    sqn = np.sqrt(float(n))
    dq = np.quantile(dlp, [0.10, 0.25, 0.50, 0.75, 0.90])
    half = max(n // 2, 1)
    k32 = min(32, n)
    hi = (0.5 * (hf + hg)) > 1.5

    ex_t1_f = float(((rkf == 0) & (rkg > 0)).mean())
    ex_t1_g = float(((rkg == 0) & (rkf > 0)).mean())
    ex_t5_f = float(((rkf < 5) & (rkg >= 5)).mean())
    ex_t5_g = float(((rkg < 5) & (rkf >= 5)).mean())

    return {
        'p:dlp_mean': float(dlp.mean()),
        'p:dlp_std': float(dlp.std()),
        'p:dlp_tstat': float(dlp.mean() * sqn / (dlp.std() + 1e-6)),
        'p:dlp_pos': float((dlp > 0.0).mean()),
        'p:dlp_big_pos': float((dlp > 1.0).mean()),
        'p:dlp_big_neg': float((dlp < -1.0).mean()),
        'p:dlp_q10': float(dq[0]),
        'p:dlp_q25': float(dq[1]),
        'p:dlp_q50': float(dq[2]),
        'p:dlp_q75': float(dq[3]),
        'p:dlp_q90': float(dq[4]),
        'p:dlp_first': float(dlp[:half].mean()),
        'p:dlp_second': float(dlp[half:].mean()) if n > half else float(dlp.mean()),
        'p:dlp_head32': float(dlp[:k32].mean()),
        'p:dlp_tail32': float(dlp[-k32:].mean()),
        'p:excl_t1_f': ex_t1_f,
        'p:excl_t1_g': ex_t1_g,
        'p:both_t1': float(((rkf == 0) & (rkg == 0)).mean()),
        'p:excl_t1_diff': ex_t1_f - ex_t1_g,
        'p:excl_t5_f': ex_t5_f,
        'p:excl_t5_g': ex_t5_g,
        'p:excl_t5_diff': ex_t5_f - ex_t5_g,
        'p:dlrk_mean': float(dlrk.mean()),
        'p:dlrk_std': float(dlrk.std()),
        'p:dlrk_tstat': float(dlrk.mean() * sqn / (dlrk.std() + 1e-6)),
        'p:dz_mean': float(dz.mean()),
        'p:dz_std': float(dz.std()),
        'p:dz_tstat': float(dz.mean() * sqn / (dz.std() + 1e-6)),
        'p:dz_pos': float((dz > 0.0).mean()),
        'p:dent_mean': float(dent.mean()),
        'p:dent_std': float(dent.std()),
        'p:dent_pos': float((dent > 0.0).mean()),
        'p:dlp_hi_ent': float(dlp[hi].mean()) if hi.any() else float(dlp.mean()),
        'p:dlp_hi_ent_pos': float((dlp[hi] > 0.0).mean()) if hi.any() else float((dlp > 0.0).mean()),
        'p:dz_hi_ent': float(dz[hi].mean()) if hi.any() else float(dz.mean()),
    }


def extract_paired_score_features(flash_npz_path, glm53_npz_path) -> pd.DataFrame:
    """Build DataFrame indexed by string id containing extra single-model and token-aligned paired features."""
    with np.load(flash_npz_path, allow_pickle=True) as fz, np.load(glm53_npz_path, allow_pickle=True) as gz:
        f_ids = fz['ids'].astype(str).tolist()
        g_ids = gz['ids'].astype(str).tolist()
        f_off = fz['offsets']
        g_off = gz['offsets']
        f_sc = fz['scores']
        g_sc = gz['scores']

    g_pos = {tid: idx for idx, tid in enumerate(g_ids)}
    common_ids = [tid for tid in f_ids if tid in g_pos]
    f_pos = {tid: idx for idx, tid in enumerate(f_ids)}

    rows = []
    for tid in common_ids:
        fi = f_pos[tid]
        gi = g_pos[tid]
        sf = f_sc[f_off[fi]:f_off[fi + 1]]
        sg = g_sc[g_off[gi]:g_off[gi + 1]]
        row = {'id': tid}
        row.update(_single_extra(sf, 'flash'))
        row.update(_single_extra(sg, 'glm53'))
        row.update(_paired_token_features(sf, sg))
        rows.append(row)

    df = pd.DataFrame(rows).set_index('id')
    assert np.isfinite(df.to_numpy(np.float64)).all(), 'Non-finite paired token features'
    return df

In [ ]:
%%writefile /kaggle/temp/v1/v1_stage.py
"""V1 GLM-5.3 / GLM-5.3-Flash likelihood stage with PB-115 token-aligned LR+LGB and hybrid calibration."""
import argparse
import glob
import json
import os
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.special import log_softmax

PREFIX = '[gMASK]<sop><|system|>Reasoning Effort: Max<|user|><|assistant|><think></think>'


def entropy(z):
    return float(-(np.exp(z) * z).sum(1).mean())


def match_temperature(z, h_ref):
    """T' with mean entropy of log_softmax(T' z) equal to h_ref (entropy decreases with T')."""
    lo, hi = 1e-3, 1e3
    if entropy(log_softmax(z * hi, 1)) > h_ref or entropy(log_softmax(z * lo, 1)) < h_ref:
        raise ValueError('entropy_ref out of reach')
    for _ in range(100):
        mid = np.sqrt(lo * hi)
        if entropy(log_softmax(z * mid, 1)) > h_ref:
            lo = mid
        else:
            hi = mid
    return float(np.sqrt(lo * hi))


def build_54_matrix(feats, ids, features_54, diff_features):
    blocks = [feats[t].add_prefix(f'{t}:') for t in ('flash', 'glm53')]
    F = pd.concat(blocks, axis=1).reindex(ids)
    for c in diff_features:
        F[f'd:{c}'] = F[f'flash:{c}'] - F[f'glm53:{c}']
    X = F[features_54].to_numpy(np.float64)
    assert np.isfinite(X).all(), 'non-finite 54-dim likelihood features'
    return X


def member_logp(model, feats, ids):
    """feats: {'flash': DataFrame, 'glm53': DataFrame} indexed by id -> (n, 18) log-probs, model['classes'] order."""
    X = build_54_matrix(feats, ids, model['features'], model['diff_features'])
    z = ((X - np.asarray(model['mean'])) / np.asarray(model['scale'])) @ np.asarray(model['coef']).T + np.asarray(model['intercept'])
    return log_softmax(z, 1)


def marginal_logodds(ll_18, labels):
    """Compute 3-way GLM vs non-GLM log-odds update (zero perturbation among the 16 non-GLM classes)."""
    g53 = labels.index('glm-5.3')
    gf = labels.index('glm-5.3-flash')
    non_glm = np.ones(len(labels), dtype=bool)
    non_glm[[g53, gf]] = False
    p18 = np.clip(np.exp(ll_18), 1e-8, 1.0)
    p_other = np.clip(p18[:, non_glm].sum(1), 1e-8, 1.0)
    non_prior = np.log(1.0 / float(non_glm.sum()))
    out = np.zeros_like(ll_18)
    out[:, g53] = np.log(p18[:, g53]) - (np.log(p_other) + non_prior)
    out[:, gf] = np.log(p18[:, gf]) - (np.log(p_other) + non_prior)
    return out


def marginal_logodds_from_p3(p3, labels):
    """Convert direct 3-class [other, glm-5.3, glm-5.3-flash] probabilities into 18-class marginal log-odds."""
    g53 = labels.index('glm-5.3')
    gf = labels.index('glm-5.3-flash')
    p3 = np.clip(np.asarray(p3, dtype=np.float64), 1e-8, 1.0)
    non_prior = np.log(1.0 / float(len(labels) - 2))
    out = np.zeros((len(p3), len(labels)), dtype=np.float64)
    out[:, g53] = np.log(p3[:, 1]) - (np.log(p3[:, 0]) + non_prior)
    out[:, gf] = np.log(p3[:, 2]) - (np.log(p3[:, 0]) + non_prior)
    return out


def fit_and_predict_pb115(pb115_dir, train_csv, feats_te, sel_ids, flash_npz, glm53_npz, labels):
    """Fit PB-115 (54 + 61 token-aligned) 55% LR + 45% 3-seed LGB18 and 50% LR3 + 50% 3-seed LGB3."""
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    import lightgbm as lgb
    import v1_token_features as vtf

    pb_dir = Path(pb115_dir)
    spec = json.loads((pb_dir / 'spec.json').read_text())
    assert spec['format'] == 'ntl-prefixb-115-v1'
    tr = pd.read_csv(train_csv, dtype={'id': str})
    tr_ids = tr['id'].tolist()
    l2i = {l: i for i, l in enumerate(labels)}
    y = tr['model'].map(l2i).to_numpy(dtype=np.int64)
    g53, gf = l2i['glm-5.3'], l2i['glm-5.3-flash']
    y3 = np.where(y == g53, 1, np.where(y == gf, 2, 0))

    tr_feats = {
        t: pd.read_csv(pb_dir / f'{t}_features.csv', dtype={'id': str}).set_index('id').loc[tr_ids]
        for t in ('flash', 'glm53')
    }
    X_tr_54 = build_54_matrix(tr_feats, tr_ids, spec['features_54'], spec['diff_features'])
    p_tr_61 = pd.read_parquet(pb_dir / 'paired_token_features.parquet').loc[tr_ids][spec['features_61']].to_numpy(np.float64)
    X_tr = np.hstack([X_tr_54, p_tr_61])
    assert np.isfinite(X_tr).all() and X_tr.shape == (len(tr_ids), 115)

    X_te_54 = build_54_matrix(feats_te, sel_ids, spec['features_54'], spec['diff_features'])
    p_te_df = vtf.extract_paired_score_features(flash_npz, glm53_npz)
    p_te_61 = p_te_df.loc[sel_ids][spec['features_61']].to_numpy(np.float64)
    X_te = np.hstack([X_te_54, p_te_61])
    assert np.isfinite(X_te).all() and X_te.shape == (len(sel_ids), 115)

    sc = StandardScaler().fit(X_tr)
    X_tr_s = sc.transform(X_tr)
    X_te_s = sc.transform(X_te)

    lr18 = LogisticRegression(C=0.025, max_iter=1000).fit(X_tr_s, y)
    p18_lr = lr18.predict_proba(X_te_s)
    p18_lgb_list = []
    for sd in (42, 1042, 2042):
        clf = lgb.LGBMClassifier(
            objective='multiclass', num_class=len(labels), n_estimators=220, learning_rate=0.03,
            num_leaves=15, min_child_samples=25, subsample=0.8, colsample_bytree=0.7,
            reg_alpha=1.0, reg_lambda=5.0, random_state=sd, verbose=-1, n_jobs=4,
        ).fit(X_tr_s, y)
        p18_lgb_list.append(clf.predict_proba(X_te_s))
    p18 = np.clip(0.55 * p18_lr + 0.45 * np.mean(p18_lgb_list, axis=0), 1e-8, 1.0)
    ll_18 = np.log(p18)

    # Derive 3-way marginal log-odds directly from calibrated 18-class model (avoids 16:1 class imbalance of 3-class head)
    ll_m3 = marginal_logodds(ll_18, labels)
    assert np.isfinite(ll_18).all() and np.isfinite(ll_m3).all()
    return ll_18, ll_m3


def compute_hybrid_delta(zl, ll_18, labels, W, h_ref, hybrid_alpha=0.20, marg_w=1.2, marg_href=0.90, ll_m3_override=None):
    """Combine calibrated 18-class GLM logp with zero-non-GLM-noise marginal GLM log-odds."""
    tp18 = match_temperature(zl, h_ref)
    w18 = W / tp18
    if hybrid_alpha <= 0.0:
        return w18 * ll_18, {'tp18': tp18, 'w18': w18, 'tp3': None, 'w3': 0.0, 'hybrid_alpha': 0.0}
    tp3 = match_temperature(zl, marg_href)
    w3 = marg_w / tp3
    ll_m3 = ll_m3_override if ll_m3_override is not None else marginal_logodds(ll_18, labels)
    delta = (1.0 - hybrid_alpha) * w18 * ll_18 + hybrid_alpha * w3 * ll_m3
    return delta, {'tp18': tp18, 'w18': w18, 'tp3': tp3, 'w3': w3, 'hybrid_alpha': hybrid_alpha}


def main():
    ap = argparse.ArgumentParser()
    for k in ('--test-csv', '--final-logits', '--out', '--report', '--bin', '--pipeline', '--flash-segs', '--glm53-segs',
              '--model', '--work', '--outdir'):
        ap.add_argument(k, required=True)
    ap.add_argument('--deadline-mono', type=float, required=True, help='time.monotonic() by which all must be done')
    ap.add_argument('--post-logits', default='', help='optional path to save post-GLM log-probabilities')
    ap.add_argument('--pb115-dir', default='', help='optional dir with full 2699 Prefix-B 115-dim train features')
    ap.add_argument('--train-csv', default='', help='path to competition train.csv (required with --pb115-dir)')
    ap.add_argument('--hybrid-alpha', type=float, default=0.20, help='share of 3-way marginal GLM log-odds in blend')
    ap.add_argument('--w-override', type=float, default=0.0, help='if >0, overrides model["W"] for 18-class term')
    ap.add_argument('--marg-w', type=float, default=1.2, help='weight for 3-way marginal GLM log-odds term')
    ap.add_argument('--marg-href', type=float, default=0.90, help='reference entropy for 3-way marginal term')
    ap.add_argument('--pack', action='store_true')
    ap.add_argument('--ubatch-div', type=int, default=1)
    ap.add_argument('--seq-batch', type=int, default=8)
    ap.add_argument('--max-text-tokens', type=int, default=0)
    ap.add_argument('--fixed-s-flash', type=float, default=870.0)
    ap.add_argument('--row-s-flash', type=float, default=0.0141)
    ap.add_argument('--fixed-s-glm53', type=float, default=1140.0)
    ap.add_argument('--row-s-glm53', type=float, default=0.0231)
    ap.add_argument('--select-classes', default='glm-5.3,glm-5.3-flash')
    ap.add_argument('--max-cover', type=float, default=0.30)
    ap.add_argument('--min-texts', type=int, default=50)
    ap.add_argument('--reserve-s', type=float, default=180.0)
    ap.add_argument('--prefetch', action='store_true')
    ap.add_argument('--priority-order', action='store_true')
    ap.add_argument('--shrink', action='store_true')
    ap.add_argument('--max-changed', type=float, default=0.25)
    a = ap.parse_args()
    assert a.priority_order or not a.shrink, '--shrink keeps a prefix of the input: it needs --priority-order'

    t0 = time.monotonic()
    rep = {'status': 'started', 'deadline_left_s': a.deadline_mono - t0}
    out_report = Path(a.report)

    def save(**kw):
        rep.update(kw)
        out_report.write_text(json.dumps(rep, indent=1))
        print('V1', json.dumps({k: v for k, v in kw.items() if not isinstance(v, dict)}), flush=True)

    def left():
        return a.deadline_mono - time.monotonic() - a.reserve_s

    model = json.loads(Path(a.model).read_text())
    assert model['format'] == 'ntl-v1-ll-model-1'
    a.max_text_tokens = a.max_text_tokens or model['max_text_tokens']
    assert a.max_text_tokens == model['max_text_tokens'], 'the test must be cut like the train features'
    frame = pd.read_csv(a.test_csv, dtype={'id': str})
    frame['text'] = frame.text.fillna('').astype(str)
    ids = frame.id.tolist()
    with np.load(a.final_logits, allow_pickle=False) as f:
        assert f['ids'].astype(str).tolist() == ids, 'final-logits ids differ from the test order'
        labels = f['labels'].astype(str).tolist()
        z = f['logits'].astype(np.float64)
    assert sorted(labels) == sorted(model['classes']) and z.shape == (len(ids), 18) and np.isfinite(z).all()

    zl = log_softmax(z, 1)
    sel_cls = a.select_classes.split(',')
    assert all(c in labels for c in sel_cls), sel_cls
    mass = np.exp(zl[:, [labels.index(c) for c in sel_cls]]).sum(1)
    order = np.argsort(-mass, kind='stable')

    tpc = model.get('tok_per_char', 0.235)
    prefix = model.get('prefix', PREFIX)
    user_template = model.get('user_template', '')
    assert ('{user}' in prefix) == bool(user_template), 'prefix and user_template go together'
    rows_i = np.minimum(frame.text.str.len().to_numpy() * tpc * 1.05, a.max_text_tokens) + model.get('prefix_rows', 13)
    cum = np.cumsum(rows_i[order])
    est = lambda r: (a.fixed_s_flash + a.row_s_flash * r, a.fixed_s_glm53 + a.row_s_glm53 * r)
    fits = 1.1 * (a.fixed_s_flash + a.fixed_s_glm53 + (a.row_s_flash + a.row_s_glm53) * cum) <= left()
    n_sel = int(min(fits.sum(), int(a.max_cover * len(ids))))
    rows = int(cum[n_sel - 1]) if n_sel else 0
    est_f, est_g = est(rows)
    save(status='estimated', n_texts=len(ids), n_selected=n_sel, cover=n_sel / len(ids),
         mass_threshold=float(mass[order[n_sel - 1]]) if n_sel else None, est_rows=rows,
         est_rows_all=int(cum[-1]), est_flash_s=est_f, est_glm53_s=est_g, left_s=left())
    if n_sel < a.min_texts:
        save(status='fallback_s10', reason=f'only {n_sel} texts fit in the time left')
        return
    pick = (lambda n: order[:n]) if a.priority_order else (lambda n: np.sort(order[:n]))
    sel = pick(n_sel)
    sel_ids = [ids[i] for i in sel]

    Path(a.work).mkdir(parents=True, exist_ok=True)
    Path(a.outdir).mkdir(parents=True, exist_ok=True)
    times = {}
    reserve_flash = 1.05 * (a.fixed_s_glm53 + a.row_s_glm53 * cum[min(a.min_texts, n_sel) - 1])
    for tag, segs, reserve in (('flash', a.flash_segs, reserve_flash), ('glm53', a.glm53_segs, 0.0)):
        if tag == 'glm53':
            m = int(min(n_sel, (1.1 * (a.fixed_s_glm53 + a.row_s_glm53 * cum[:n_sel]) <= left()).sum()))
            if m < a.min_texts:
                save(status='fallback_s10', reason=f'after Flash only {m} texts fit for GLM-5.3', times=times)
                return
            if m < n_sel:
                save(status='glm53_shrunk', n_selected_flash=n_sel, n_selected=m, cover=m / len(ids),
                     mass_threshold=float(mass[order[m - 1]]), left_s=left())
                n_sel = m
                sel = pick(n_sel)
                sel_ids = [ids[i] for i in sel]
        budget = left() - reserve
        if budget <= 0:
            save(status='fallback_s10', reason=f'no time left for {tag}')
            return
        query = Path(a.work) / f'v1_query_{tag}.csv'
        frame.iloc[sel][['id', 'text']].to_csv(query, index=False)
        cmd = [sys.executable, a.pipeline, '--bin', a.bin, '--segs', segs, '--csv', str(query), '--out', a.outdir,
               '--tag', tag, '--work', str(Path(a.work) / 'state'), '--seq-batch', str(a.seq_batch),
               '--state-dtype', 'auto', '--max-text-tokens', str(a.max_text_tokens), '--prefix', prefix,
               '--budget-s', f'{budget:.0f}']
        if user_template:
            cmd += ['--user-template', user_template]
        if a.shrink:
            cmd += ['--shrink-min', str(a.min_texts)]
        if a.pack:
            cmd.append('--pack')
        if a.ubatch_div > 1:
            cmd += ['--ubatch-div', str(a.ubatch_div)]
        if a.prefetch:
            cmd.append('--prefetch')
            if tag == 'flash':
                cmd += ['--prefetch-after', sorted(glob.glob(a.glm53_segs))[0]]
        t = time.monotonic()
        with open(Path(a.outdir) / f'{tag}_pipeline.txt', 'w') as log:
            code = subprocess.call(cmd, stdout=log, stderr=subprocess.STDOUT)
        times[tag] = time.monotonic() - t
        if code == 3:
            save(status='fallback_s10', reason=f'{tag} time budget exceeded', times=times)
            return
        if code != 0:
            raise RuntimeError(f'V1 {tag} pipeline failed with exit code {code}; see {tag}_pipeline.txt')
        if a.shrink:
            done = set(pd.read_csv(Path(a.outdir) / f'{tag}_features.csv', dtype={'id': str}).id)
            n_done = len(done)
            assert n_done >= a.min_texts and set(sel_ids[:n_done]) == done, f'{tag}: kept texts are not the top'
            if n_done < n_sel:
                shrinks = json.loads((Path(a.outdir) / f'{tag}_timing.json').read_text()).get('shrinks', [])
                save(**{f'{tag}_shrunk': {'from': n_sel, 'to': n_done, 'shrinks': shrinks}},
                     n_selected=n_done, cover=n_done / len(ids), mass_threshold=float(mass[order[n_done - 1]]))
                n_sel = n_done
                sel = pick(n_sel)
                sel_ids = [ids[i] for i in sel]
        save(status=f'{tag}_done', times=times, left_s=left())

    feats = {t: pd.read_csv(Path(a.outdir) / f'{t}_features.csv', dtype={'id': str}).set_index('id')
             for t in ('flash', 'glm53')}
    for t, f in feats.items():
        assert set(sel_ids) <= set(f.index), f'{t}: features missing for some texts'
        feats[t] = f.loc[sel_ids]

    ll_m3_override = None
    member_mode = 'll_model_54'
    if a.pb115_dir and a.train_csv:
        try:
            t_pb = time.monotonic()
            ll, ll_m3_override = fit_and_predict_pb115(
                a.pb115_dir, a.train_csv, feats, sel_ids,
                Path(a.outdir) / 'flash_scores.npz', Path(a.outdir) / 'glm53_scores.npz', labels,
            )
            member_mode = f'pb115_lr_lgb_direct3 ({time.monotonic() - t_pb:.1f}s)'
        except Exception as _e_pb:
            print(f'PB115 head failed, falling back to 54-dim ll_model.json: {_e_pb!r}', flush=True)
            ll = member_logp(model, feats, sel_ids)[:, [model['classes'].index(c) for c in labels]]
    else:
        ll = member_logp(model, feats, sel_ids)[:, [model['classes'].index(c) for c in labels]]

    eff_w = a.w_override if a.w_override > 0 else model['W']
    delta, cal_info = compute_hybrid_delta(
        zl, ll, labels, eff_w, model['entropy_ref'],
        hybrid_alpha=a.hybrid_alpha, marg_w=a.marg_w, marg_href=a.marg_href,
        ll_m3_override=ll_m3_override,
    )
    # Vary marginal odds near the end of the likelihood queue (0.50 -> 0.60 across 30-40% tail)
    positions = np.empty(len(ids), dtype=int)
    positions[order] = np.arange(len(ids))
    fraction = positions[sel] / len(ids)
    ramp = np.clip((fraction - 0.30) / 0.10, 0.0, 1.0)
    if np.any(ramp):
        marginal = ll_m3_override if ll_m3_override is not None else marginal_logodds(ll, labels)
        evidence_delta = cal_info['w3'] * marginal - cal_info['w18'] * ll
        delta += 0.10 * ramp[:, None] * evidence_delta
    zl_post = zl.copy()
    zl_post[sel] = zl[sel] + delta
    assert np.isfinite(zl_post).all()

    before = zl.argmax(1)
    after = zl_post.argmax(1)
    changed = float((before != after).mean())
    changed_sel = float((before[sel] != after[sel]).mean())
    agrees = float((ll.argmax(1) == before[sel]).mean())
    if changed > a.max_changed:
        save(status='error_sanity', changed=changed, changed_selected=changed_sel,
             temperature=cal_info['tp18'], weight=cal_info['w18'], member_argmax_agrees_s10=agrees)
        raise RuntimeError(f'V1 blend changes {changed:.1%} of the predictions (> {a.max_changed:.0%}): not submitted')

    if a.post_logits:
        _sel_mask = np.zeros(len(ids), dtype=bool)
        _sel_mask[sel] = True
        np.savez_compressed(a.post_logits, ids=np.asarray(ids, dtype=str),
                            labels=np.asarray(labels, dtype=str), logits=zl_post.astype(np.float64),
                            sel_mask=_sel_mask)

    sub = pd.DataFrame({'id': ids, 'model': np.asarray(labels)[after]})
    assert len(sub) == len(frame) and sub.id.is_unique and sub.model.isin(labels).all()
    sub.to_parquet(a.out, index=False)
    assert pd.read_parquet(a.out).equals(sub)
    count = lambda p: {labels[i]: int(n) for i, n in zip(*np.unique(p, return_counts=True))}
    save(status='applied', member_mode=member_mode, n_blended=len(sel_ids), entropy_s10=entropy(zl),
         temperature=cal_info['tp18'], weight=cal_info['w18'], hybrid_alpha=a.hybrid_alpha,
         changed=changed, changed_selected=changed_sel, member_argmax_agrees_s10=agrees,
         counts_s10=count(before), counts_v1=count(after), times=times, total_s=time.monotonic() - t0)


if __name__ == '__main__':
    main()

In [ ]:
# Stage 54-feature User-Prefix GLM Logistic Head (ll_prefixB/L160: C=0.05, W=1.5, prefix_rows=30)
import json as _j
Path('/kaggle/temp/v1/ll_model.json').write_text(_j.dumps(_j.loads(r'''{"format": "ntl-v1-ll-model-1", "features": ["flash:n_tok", "flash:lp_mean", "flash:lp_std", "flash:lp_q05", "flash:lp_q10", "flash:lp_q25", "flash:lp_q50", "flash:lp_q75", "flash:lp_q90", "flash:lp_min", "flash:ent_mean", "flash:ent_std", "flash:fdg", "flash:surprise_mean", "flash:z_mean", "flash:logrank_mean", "flash:top1", "flash:top5", "flash:top20", "flash:rank_ge100", "flash:lp_mean_first", "flash:lp_mean_second", "flash:bpc", "flash:tok_per_char", "glm53:n_tok", "glm53:lp_mean", "glm53:lp_std", "glm53:lp_q05", "glm53:lp_q10", "glm53:lp_q25", "glm53:lp_q50", "glm53:lp_q75", "glm53:lp_q90", "glm53:lp_min", "glm53:ent_mean", "glm53:ent_std", "glm53:fdg", "glm53:surprise_mean", "glm53:z_mean", "glm53:logrank_mean", "glm53:top1", "glm53:top5", "glm53:top20", "glm53:rank_ge100", "glm53:lp_mean_first", "glm53:lp_mean_second", "glm53:bpc", "glm53:tok_per_char", "d:lp_mean", "d:lp_q50", "d:ent_mean", "d:top1", "d:logrank_mean", "d:fdg"], "diff_features": ["lp_mean", "lp_q50", "ent_mean", "top1", "logrank_mean", "fdg"], "mean": [121.38532789922193, -2.4260019501236796, 2.977902222946341, -8.391377165278668, -6.508596439686649, -3.626004445559451, -1.2642393094367061, -0.2138849301170523, -0.022619060259504085, -14.894307147135953, 1.5255739335159824, 1.1579562513662212, -5.638152610411089, 0.9004280166076971, -0.6082919878361244, 1.1036535455983696, 0.5002127088667474, 0.7553078489867455, 0.8790082703416411, 0.049843302664741476, -2.785688071305161, -2.069375010471187, 0.6219789663417534, 0.17556811209707204, 121.38532789922193, -2.5812442425346966, 3.2158897458871207, -9.066232017209328, -7.024507404195241, -3.854065645277699, -1.2853007953774105, -0.21743587987209614, -0.02422422299792892, -15.983028637298437, 1.41995134286995, 1.0999446624988136, -7.8392950573889415, 1.1612928996647467, -0.9224403182872035, 1.1451894410855874, 0.4923752879199216, 0.7471018048206269, 0.8714704465538589, 0.05376077384672739, -2.9355549012736515, -2.229978006170103, 0.6611323240273932, 0.17556811209707204, 0.15524229241101686, 0.02106148594070437, 0.10562259064603255, 0.007837420946825857, -0.041535895487217485, 2.2011424469778524], "scale": [38.592979005558206, 0.5393137892044101, 0.4004671083600173, 1.3604275989678145, 1.2496975058838489, 1.0145578148724628, 0.5807562839256815, 0.19003109256245265, 0.04366605490416353, 2.856073261174058, 0.38003040846010067, 0.1832553083175586, 1.6211732061520345, 0.31575523872262196, 0.31718755107040475, 0.31254367165309466, 0.07798596643535212, 0.07870559959041155, 0.057241111741293794, 0.03160431312388449, 0.6720116954953225, 0.5398873747255816, 0.203152673044373, 0.03392095256315838, 38.592979005558206, 0.5637662708338721, 0.46233328448677286, 1.5212671145338694, 1.3239677570473984, 1.0497230506203583, 0.6234540311275364, 0.19750661004161096, 0.04614845713113082, 3.2465782509115204, 0.3888506410580733, 0.1881766155893005, 2.3263538090801212, 0.3523878334710465, 0.7358605765826416, 0.32851707276490555, 0.08039721571282175, 0.0793708152300281, 0.05852354681249675, 0.03372514760296747, 0.6773229570182222, 0.6234700953075106, 0.21227468616264136, 0.03392095256315838, 0.24464169712519823, 0.2835598734855828, 0.16578395252326922, 0.042042815001667996, 0.12838625838174414, 2.16440010550044], "coef": [[-0.031476498133259714, 0.10028463146029108, 0.023551787684195916, -0.09040435008702723, -0.03908015848825789, 0.09026628237677363, 0.19966401542739987, 0.008231865358579765, 0.1490881600357796, -0.08336997005033941, -0.09777861119955773, -0.19392673789615608, 0.40216771886913605, -0.0536049349726127, -0.14594911765899907, -0.0026164762597721187, 0.07902879638636295, 0.019823967062145204, -0.15735114528507657, 0.1819920949595595, 0.07841152686258508, 0.1034102592236056, 0.04021483974681796, 0.1850309389465541, -0.031476498133259714, 0.07678634641704742, -0.007964892239584874, 0.031576684657081064, 0.07825440281464306, 0.09479658336848466, 0.17338086314815376, 0.14347009088819002, 0.026255386127865335, 0.05415312618220648, -0.12142705079108655, -0.1633176699959422, 0.1005029515142811, 0.01114520422774843, 0.02022806263551967, -0.024137822609494704, 0.07814724090911519, 0.18253226280632873, 0.07021581842441799, 0.1577777947338406, -0.004764497408323129, 0.14649109111321962, 0.06380261763957557, 0.1850309389465541, 0.04412711548430751, 0.027723011322363073, 0.06067017245028081, -0.0028467057287689277, 0.05539474254651778, 0.1932073950043523], [-0.23488641737371743, 0.03775135402638909, -0.12354617758854304, 0.03296810432397082, -0.044172398401426505, 0.07940378113372078, 0.027905857737650624, -0.06384567418475307, 0.02175859048330302, -0.1721024178155427, -0.09384433905734615, 0.002256613964251526, 0.010060795773377932, 0.0484675306670253, 0.20111310614053912, -0.06318357060705973, -0.017865478122284596, 0.02426223158972952, 0.034640308555689364, -0.08599547459226346, -0.166555888277028, 0.27884247791070577, -0.03994674160733936, 0.043128054411826794, -0.23488641737371743, 0.10813131017820159, 0.049104386635942174, -0.07219014285439408, 0.08175399802475032, 0.20843705674144378, 0.01277591865183336, 0.08678281061209833, 0.15282657142913514, 0.00997568835991234, -0.11301780137856762, -0.0891175831192379, -0.105255942240858, -0.048281295114331685, -0.06444048482607083, -0.09126623936615601, -0.03501511142483295, 0.10899907988167419, -0.006546296310138534, -0.044079659331990484, -0.021804643076097956, 0.21717865324964203, -0.10070507478156114, 0.043128054411826794, -0.1659609142230508, 0.029063718195842442, 0.049964679254681836, 0.033819355083106656, 0.07971953371142905, 0.12066754849853331], [-0.09027004305316196, 0.09532843588238626, -0.04545983666303509, 0.12235931379203885, 0.022174811346888534, 0.16121859447243891, 0.10317049931061426, -0.10756795563649417, 0.0076465932616408645, -0.08547148136791798, -0.08253760502743232, -0.04837289772473218, 0.024747353839867172, -0.0634831596266078, 0.04024074544560749, 0.00788531568936723, -0.08380082467801561, -0.08197855098494607, 0.04304606964517915, 0.037556462376617236, 0.002901527012454225, 0.19361518914818002, 0.029394393208411085, 0.04502929886135963, -0.09027004305316196, 0.1055825376563887, -0.03581989506776347, -0.08944109659670402, -0.10798442700949358, 0.25947832618684696, 0.08847933877061225, -0.10655112430016582, -0.07070519559832386, 0.09748341928437405, -0.08410297164554664, 0.03165716281684261, -0.09827972302055323, -0.07611040034009335, -0.008345881465937748, -0.007221702726160872, -0.09321236054387382, 0.1205057485389388, 0.248740058142094, 0.2594530046736544, -0.014189464254818544, 0.21214596027025945, 0.033349302724152534, 0.04502929886135963, -0.033158425731931604, 0.016765825516543043, 0.008062871387859835, 0.022803562442584708, 0.03767504574588553, 0.1241698123602793], [0.13778889280154608, 0.03699979530397428, 0.05094547812305795, 0.037674181003225574, 0.05079050812811399, 0.2965523121690108, 0.06543873844121238, -0.07479041817645242, -0.11827089550512167, 0.04505769057576572, -0.09568810837524754, -0.02080122496743072, 0.24055747629400476, 0.05197028930329554, 0.1708564084366948, 0.005393646935892943, 0.08320790810708799, 0.07521597163755717, -0.02754912882434592, 0.0693146547425146, -0.143107175721388, 0.24193667698299226, 0.08709220389334844, 0.103097856902268, 0.13778889280154608, 0.006903100381874301, 0.04888892841176577, -0.024158041769738077, -0.0345887722191143, 0.08869032548610407, 0.10200925098295069, -0.05073447318753438, -0.00422373932991831, 0.021054698159201534, -0.13395518979532417, 0.3621961940942221, 0.18469925405212442, 0.13677210643382348, -0.08750520242221525, -0.03740397309015754, 0.051993518416279125, 0.1830878828377808, 0.01913563898996661, 0.1279233326325505, -0.0356205853907969, 0.04517152435117619, 0.110289148869157, 0.103097856902268, 0.06565832740063249, -0.09026002114159981, 0.09484736173207697, 0.05491818311693594, 0.10884026174165091, -0.0183378655314965], [-0.09571772598695984, 0.029891526855180997, 0.08415037306101363, 0.06927900433694709, -0.0798579142558381, 0.0665432366790329, -0.060886116710503986, 0.19225290685186144, 0.006100748477771084, -0.22024783627478223, -0.05352701822126062, -0.062523860485515, 0.08433444022558667, 0.01336789217490057, 0.1417369746798173, 0.021661180242237916, -0.011801532662339569, 0.05621502059226296, 0.22242052442520965, 0.09701160088871831, -0.023880675274506453, 0.10182962592005051, 0.0841616312539866, -0.09768548466317492, -0.09571772598695984, 0.046937135184769135, -0.12603289161189077, -0.04745825954102325, 0.08986636546205184, 0.12018412065082483, 0.009897528765256078, -0.13529841908315596, -0.013393827130654016, 0.06413310736895739, -0.06609291181823342, -0.172487698211519, 0.007283814584954842, -0.002160410957804698, -0.14491506722731218, 0.03683315040364548, -0.0073166623932793325, 0.06748442399327977, 0.024902939872419042, 0.07831660601450138, -0.09698241289057268, 0.198258493143971, 0.07896506195256206, -0.09768548466317492, -0.04226859597069283, -0.14646165756759974, 0.03232144275777088, -0.007899438830330173, -0.04151732446487884, 0.055339123640226605], [-0.00864208500710048, 0.0026749487019297443, 0.08572769530137019, -0.08633121139610232, 0.04525934568784551, 0.11084938398112192, 0.07376953727238854, -0.13900548526957793, 0.09829411763906257, 0.14847008708439965, 0.11771142744957112, -0.3017084501061023, -0.06855764792625567, -0.14624162297750926, 0.09176781868242516, 0.16332196710474772, 0.02870217994710692, -0.20550311894630924, -0.04478452505145586, -0.08792650980852193, -0.03735573803524979, 0.05511096700759692, -0.04934895846527344, -0.10300941995924938, -0.00864208500710048, -0.022217385200165116, 0.18895474885736976, -0.028193832349130026, -0.23823045276632604, 0.010623077725745705, 0.03153115618551901, 0.012854697259336469, 0.16113367719437355, -0.0642244934564892, 0.15957857146582735, 0.07674572799471586, -0.09487388901108353, -0.14054633193149424, 0.09629791575474918, 0.07425028187283218, -0.03186898924492477, -0.13905204329068133, -0.1359590638256985, 0.10488013678022823, 0.09149795072658393, -0.13572282002438124, -0.11269029872044552, -0.10300941995924938, 0.0570959459752016, 0.08176014334564968, -0.10446311416725235, 0.11418229832279035, 0.20759824566615595, 0.05062206886327765], [-0.08259478470566303, -0.05139343421037613, 0.008781825167728135, -0.14029894030472362, -0.08377531068804708, 0.030319298961760152, 0.020247054479144024, 0.024396947747034485, 0.2638325433300283, 0.12373125200831364, 0.12425491814936841, -0.02957984293232846, -0.2721164249792365, -0.061767651527037185, -0.003624796271095008, 0.1133357989222381, 0.0014962428066886122, -0.1288545628746828, -0.03258757783024489, 0.00033155112919324055, -0.1889137945893927, 0.1216351250107066, -0.11095528287975148, -0.09992556507041984, -0.08259478470566303, -0.0496384631812202, -0.09306156023576856, 0.32079583511046156, 0.022246552935974915, -0.13718392863772358, 0.054807861076361666, 0.05526871764768316, -0.1244127340124587, -0.02048458197544872, 0.1503139363767561, 0.03213346993076278, -0.2799063947611695, -0.08645355017617565, -0.12513254414101152, 0.15991128282922587, -0.010120103788325343, -0.21096281969081762, -0.2590666806880791, -0.020893552033366996, 0.15004449150165342, -0.2563056136475348, -0.11278803219205868, -0.09992556507041984, 0.001092632762594095, -0.07903649247088036, -0.06773287191959726, 0.022127778755318555, -0.13327905983027105, 0.09703078925869539], [-0.029578747941580653, -0.0544138946541167, -0.09789900931902272, 0.05018338843941251, -0.05911696888154548, -0.16448578117413665, -0.08849847270879618, 0.14959836063736318, 0.2916392558093849, 0.0527756962942427, 0.0441986618663107, 0.23203478258831683, -0.25243868694709976, 0.039743847923490644, -0.03351371619154146, 0.06143569063875552, -0.006962671031488783, -0.17583990914391168, -0.23292746987127877, -0.044921531202419186, -0.12839194680636665, 0.04432278862424783, -0.10690255256388259, -0.22259502780821835, -0.029578747941580653, -0.04213096561373265, -0.1272003396831082, -0.11166852922477186, 0.02781261785247103, -0.17899696033144213, -0.0792326447705277, 0.0948667797509799, 0.29577494241602015, 0.08256859583238478, 0.05755091217537189, 0.13683934098531134, -0.25229275842927806, 0.0038971500925181054, 0.09113540157912117, 0.06983490857407364, -0.07862651475283629, -0.18540784498013432, -0.03421363565248193, 0.0814873464074042, -0.1495327197494241, 0.07861060168234994, -0.11464804247034592, -0.22259502780821835, -0.022866692008493915, -0.0070465977425358895, -0.03366956503008204, 0.1374399463437411, -0.029135699302465602, 0.08208989809643373], [-0.0766681792885646, -0.01933082324993079, -0.10801579354702974, -0.15760026609939223, 0.011443637969413768, -0.1604370969307811, -0.0009397739273462057, 0.13218086822339134, 0.23624829435543743, 0.16234528098713227, 0.05643156884206744, 0.27311469957526097, -0.3157595389009176, -0.03490150366567723, -0.07021057479198799, 0.06983814116648242, 0.11121791074065761, -0.28784717274940014, -0.3472330217560849, 0.09609698502473761, -0.07342771096529126, 0.052056380284254154, -0.12634563445200064, -0.1675982288923229, -0.0766681792885646, -0.03022252807649384, -0.013050317970446668, 0.06289769972122503, 0.031804521941528774, -0.13693717973568753, 0.021206018313207783, -0.12766994227341272, 0.276565534726454, -0.0404495724464645, 0.03718720145375202, 0.299656623809631, -0.25341125457600855, 0.007316299200838786, -0.04188896421229906, 0.09707887616490676, 0.09113395836912318, -0.2705427772213086, -0.3176302601317077, -0.14999022506426518, -0.040289805193929364, -0.00926694237676628, -0.11830823046887463, -0.1675982288923229, 0.02703162417202959, -0.04854973676664999, 0.0421358335733216, 0.032027345081716174, -0.07839311847126418, 0.03586367099886759], [-0.01921875397152377, -0.07618836144333954, -0.014197696319392546, -0.024612068182776396, -0.026057492708962106, -0.025459785144612817, 0.00041553275044079726, -0.05059668178984625, -0.12380130390799099, 0.025058244506741472, 0.013029811310957973, 0.13930554864505484, 0.013351884625357864, 0.11444848717258076, 0.1234365514166985, 0.07469607308111433, 0.08547292096782426, -0.09386986316418272, -0.14883989466836134, 0.06669604087352188, -0.017740697105663703, -0.12335573345836118, 0.14232975828157338, 0.1467235651461456, -0.01921875397152377, 0.21471716524762308, 0.10916412059881218, -0.02386290784150547, 0.19829600608971182, 0.2864380004873019, 0.05212740430033211, 0.23538060056090268, 0.16977187719289757, -0.13721243054075352, -0.18712687405857456, 0.022978104628002313, 0.13994248828042075, -0.13702485163344913, -0.11623265914042215, -0.08796975663783704, 0.2479807334541232, 0.219281388897484, 0.16544836149842945, 0.01120244830583191, 0.047182242986724646, 0.341944176097372, 0.007139099889485608, 0.1467235651461456, -0.6627640804665387, -0.11375980913922087, 0.46877956683185784, -0.31566088459776337, 0.4069388152585312, -0.14041305130447448], [-0.0886319496321701, 0.21649455726689829, 0.17135687568360725, 0.053995872058471665, 0.10015060215059016, 0.3493573445510839, 0.05971924640008341, -0.057999088292242776, -0.017033930605045617, 0.0149850882172794, -0.1108048563662637, 0.09069083089835908, 0.5502538376882649, -0.23641503304743058, -0.11864733480082304, -0.10076981257282043, 0.10755202730784712, 0.46539532529961214, 0.07171202887831495, 0.229377399280934, 0.1721798367544482, 0.21944550526034193, -0.048499670429344645, 0.09058505974283773, -0.0886319496321701, -0.05733537516569727, -0.048177784382374765, 0.15027319370845624, 0.1210154333629609, -0.1642102322389876, 0.029049614593322462, 0.13964231584435746, -0.10939129968682475, -0.001905668764622197, 0.09249229832896742, -0.08717726027434908, 0.2114308064497126, -0.010335030066884304, 0.04046139657641091, -0.08493919368090698, -0.06359124229827148, 0.03930979563538845, -0.17277912596489528, -0.019526594920029745, -0.08712680667478247, -0.010039275376475235, 0.12242312693425496, 0.09058505974283773, 0.6093901915235389, 0.058439962232839926, -0.4709436778051787, 0.32110379896419156, -0.02797022039199095, 0.18489830751918404], [0.21857971630010747, -0.05192666867542844, -0.05329000668909974, 0.09845675964444824, -0.002546632095764033, -0.1091794597936576, -0.030469604223146995, 0.006091499292801335, -0.10973863004614197, 0.10527505810278309, -0.0976941547627128, 0.13083056290852718, 0.05187561494177107, 0.20627216905801044, 0.07532126604831753, -0.07406245558919514, -0.06546369083646104, 0.02398332942132074, 0.16492903477805002, 0.11864620791512104, -0.006955976381876384, -0.09924743549526298, 0.13989752641801298, 0.07662782918159323, 0.21857971630010747, -0.02068162750313779, 0.009905009407532275, -6.320890151246878e-05, -0.18577193748879908, -0.06584131190360949, -0.09475355697634309, -0.18684048635790132, -0.10371228564904782, -0.06780070599907327, -0.037204937378429996, -0.17748041024365038, 0.12149311063329586, 0.07414208233296979, -0.045778465072074136, -0.023102108444943637, -0.013643346009574925, 0.15540306260964362, 0.13714549098157017, -0.047599985506579054, -0.03767310239017692, -0.0035292646319875693, 0.14057092250948866, 0.07662782918159323, -0.06681266776662018, 0.14592711031575697, -0.1366814184583375, -0.0953400043258527, -0.12118364501440523, -0.09172823601235104], [-0.05412095301337113, -0.00801538767940881, 0.038992791216164606, -0.045958296945352306, -0.023240520145722105, 0.0969513042720956, -0.15361855185681741, 0.003464995240811926, -0.057578667766080095, -0.09701920317105524, 0.09812711196473305, 0.08503626503132011, -0.06496345759034564, -0.10441149756696447, -0.21954120359120416, -0.07171114591676057, -0.2218927630257679, -0.046357069607666376, 0.0824310201608545, -0.3520093490642453, 0.0016574847747344514, -0.015375111459084026, -0.05011523255703437, -0.06094596862970035, -0.05412095301337113, -0.06779894417864009, -0.02244045508957585, 0.07940865947682894, -0.053087765703298345, -0.0057592300511876094, -0.15230778491544858, -0.044597240870333794, -0.1651322585219224, -0.05449201444000362, 0.05327755165006158, -0.22081515131574952, 0.019100918723348907, 0.0496775034498265, 0.29537060472221877, -0.03070864539799934, -0.16368718647119185, 0.04372026149165096, 0.11981959622878233, -0.02551928851974509, 0.16510388278573912, -0.29826076822439224, -0.055216548398257584, -0.06094596862970035, 0.13856979093548702, 0.020249561719191678, 0.09997515486615083, -0.09857873525503358, -0.09599586983758811, -0.06918892282110198], [0.1440527621421718, -0.022866813429953784, 0.0022722651003149177, 0.020481301428849533, 0.11677090709087394, -0.1704327643736901, -0.06026390584556655, 0.15771182304121478, -0.32093220679371937, 0.030770895387948485, -0.06588253068356219, -0.16799320879452648, -0.2716098469280805, 0.1183503811217073, -0.03949216790200735, 0.027304607853285857, -0.11003604512817, 0.023396228371319275, 0.03831293682870715, 0.17635906639047233, 0.12458442916355283, -0.19961298103228453, -0.0023084336301138334, -0.06309928124563853, 0.1440527621421718, -0.030725335231822963, -0.08708966812650032, -0.09271700573462592, -0.004442385093066497, 0.05563885992714399, -0.030386785366901644, 0.28918918607259286, -0.12545039864360694, 0.001770741837638881, -0.09716090359622703, 0.0673908418779554, 0.08005369896416827, 0.15637028886659163, -0.028096163354667296, 0.06917264016792142, -0.09023824775441985, -0.19715322528450407, 0.09356386289031939, -0.13653048900803644, -0.043429754362986586, -0.013186264044589849, 0.04703991047968784, -0.06309928124563853, 0.020395214407269987, -0.05661547942724896, 0.07686940991062723, -0.031547922082110776, -0.11052982656194939, -0.289484292802802], [0.07886453372796007, -0.05181357878525833, -0.13031413607730646, 0.0008823014854131659, -0.02960457144794509, -0.302553256301585, 0.0443823070530671, -0.1412982666074471, -0.028041077358504987, 0.062020294909866956, 0.06223724398320064, 0.050648886911797114, -0.16827563861690487, 0.013591958986696127, -0.08276329743530479, -0.04982576218633699, -0.07110084994761402, -0.027621058801797018, 0.2598475484803139, -0.2642277180653465, 0.0357643724426235, -0.14187920243236302, -0.027024889162447593, 0.007147440894443252, 0.07886453372796007, -0.06284434341729246, 0.128608319517315, -0.11871933953649347, 0.09215652379304305, -0.10316623219755508, -0.00631083033354616, -0.04770563718941234, -0.16905983910332462, 0.00851424125040175, 0.0846070584940341, -0.0979035581117109, 0.014912692721168732, 0.007179624145948707, -0.04350526400418464, -0.0529123083336203, -0.006890778999288711, -0.0461226177528738, -0.06630781717891232, -0.13413325611924704, 0.008170385958721642, -0.11911610595850453, 0.010781690381994169, 0.007147440894443252, 0.03059880516043166, 0.10477440253516504, -0.05578020996802052, -0.11870920279021405, 0.014097069368451396, -0.14206992288899548], [0.019549952111113135, -0.08305672675658778, -0.023472762485011078, 0.10407945728127505, -0.04952311684472796, -0.12590303133579944, 0.029334212429815563, 0.043205791411408304, -0.2785628065290129, -0.034878119474499036, 0.04608963057173482, -0.20555011509910137, -0.3013588725156022, 0.08639025912732934, 0.09540593135414346, -0.06344239510538853, -0.1153208850241918, -0.02298727792732419, 0.047873777131741777, -0.08996034943549189, 0.0019148659543887867, -0.17726909875296093, -0.07573364755573142, -0.009179330952579624, 0.019549952111113135, -0.04773879496993035, -0.049626523915009706, -0.06415819683418723, 0.0474150808701884, -0.025013980080524287, 0.011522782667722185, -0.09093334699751689, -0.03838046650650022, 0.06577169147953903, 0.05664492961811423, 0.0030137122687171, -0.08770276356232713, 0.01386854129289565, 0.04466801589882021, 0.03620291403526565, -0.06024102533692141, -0.3049285943781452, 0.02847368703946081, -0.12916734591178455, 0.07344785768202604, -0.17403311244815098, -0.13006831640092542, -0.009179330952579624, -0.07308695051506287, 0.03474434791327871, -0.027209847479710714, -0.0987136555878339, -0.2470809169841815, -0.13145779782367045], [0.18951789965510327, -0.05773321238537052, 0.1770178965202121, 0.09843080585435986, -0.13815121231084257, -0.05265878214825182, -0.18254840713488807, -0.1057566302568775, 0.03484333239454268, 0.09255519130076119, 0.08984977375405523, -0.004076806515548639, 0.0683102493496437, -0.009530574053112674, -0.15529736751921525, -0.034860772553121235, 0.3421718635630689, 0.22499796075359582, 0.07524263438115711, -0.02259158566308672, 0.11654475358900372, -0.2579023571295447, 0.08763851659827877, 0.1105514168007217, 0.18951789965510327, -0.06954724546690555, 0.08202004278143006, -0.012250619460070503, -0.0316436049634807, -0.2172103991671912, -0.17652689156212573, -0.19846208389152128, -0.04710575386229328, -0.0678981079518367, 0.08552525511288686, 0.06021680361714148, 0.1198686917016932, 0.016890029623661568, 0.1671385194026016, -0.04966829284164581, 0.3051990398217905, 0.09721424845172545, 0.06518095513777579, -0.00881304217358772, -0.05651942727107523, -0.060593357861855116, 0.10543699221663548, 0.1105514168007217, 0.032995494161476235, 0.014248375600351292, 0.005362979522249484, 0.051077703216772005, 0.04222701394592973, -0.07767244196266918], [0.023452381369071327, -0.042686348227278986, -0.04660156916922442, -0.14358535663303804, 0.22853648389535303, -0.170351581394525, -0.04682216889475121, 0.023725142409224215, -0.05549211727533449, -0.16995575122109796, 0.04582707580138453, 0.030614953998554156, 0.26942074279743317, 0.017753161901915223, -0.07083922604206459, -0.0844000308436669, -0.13460510937031206, 0.15756854947267754, -0.04918311997836911, -0.12574954575001546, 0.2523708066029715, -0.39756307561282056, 0.02645217390249046, 0.016116846333553817, 0.023452381369071327, -0.058176587060866584, -0.006181227888143645, 0.03992910797010359, -0.13487215790374615, -0.08996689622998834, -0.04726924353037813, -0.06866244448518614, -0.11136019104187203, 0.04904226582007521, 0.06291092578621793, -0.08452865075114228, 0.17243429797610804, 0.023653040553410755, -0.04945922070324702, -0.053954010918948324, -0.12000292195269077, 0.13663176745457012, 0.01987647054667818, -0.10478723095937893, 0.05248640702153449, -0.15974697531335055, 0.024626669835475436, 0.016116846333553817, 0.0399631846994207, 0.008033335558752935, -0.04250876745869843, -0.020203422129249866, -0.06740504712555734, 0.0164639169077116]], "intercept": [-0.32989212720707467, -0.25649779451473415, 0.15369153736774724, 0.2846527748963043, 0.27813545141736745, 0.3373712542384871, 0.20435007623350632, -0.20044885077259966, -0.4333162719598729, -1.3699772876715532, -1.070446354816162, 0.3985076254196239, 0.3378404489044811, 0.3205822273130323, 0.36404091970144153, 0.272664009981679, 0.35698845111934174, 0.35175391034899317], "classes": ["claude-fable-5", "claude-opus-4-6", "claude-opus-4-8", "claude-opus-5", "claude-sonnet-5", "gemini-3.1-pro-preview", "gemini-3.6-flash", "gemini-3.7-flash", "gemini-3.8-flash", "glm-5.3", "glm-5.3-flash", "gpt-5.5", "gpt-5.6-luna", "gpt-5.6-sol", "gpt-5.6-terra", "gpt-6-astra", "kimi-k3", "qwen3.8-max"], "C": 0.05, "W": 1.5, "entropy_ref": 0.5692304418233374, "max_text_tokens": 160, "tok_per_char": 0.19383936500423704, "train_rows": 2699, "source": "experiments/bench/ll_prefixB/L160", "prefix": "[gMASK]<sop><|system|>Reasoning Effort: Low<|user|>{user}<|assistant|><think></think>", "user_template": "Write a single paragraph of about {n_words} words, with no title or headings.", "prefix_rows": 30, "train_changed_in_sample": 0.15561319007039645}''')))
_m = _j.loads(Path('/kaggle/temp/v1/ll_model.json').read_text())
assert _m['format'] == 'ntl-v1-ll-model-1' and _m['W'] == 1.5 and _m['max_text_tokens'] == 160
assert _m['prefix_rows'] == 30 and '{user}' in _m['prefix'] and '{n_words}' in _m['user_template']
print('ll_model_prefixB_L160 staged: W', _m['W'], 'C', _m['C'], 'prefix_rows', _m['prefix_rows'], 'tok_per_char', round(_m['tok_per_char'], 4), flush=True)


In [ ]:
# Resolve `ntl-glm-prefix-ab` binary (`ntl-segrun` + CUDA 13 libs), verify 25 GGUF segments & run GPU smoke test
def _pinned_dataset(owner, slug):
    namespaced = Path('/kaggle/input/datasets') / owner / slug
    flat = Path('/kaggle/input') / slug
    if namespaced.is_dir():
        return namespaced
    assert flat.is_dir(), (namespaced, flat)
    return flat

# Locate the `ntl-glm-prefix-ab` kernel output binary supporting `--user-template`
_bin_cands = [
    Path('/kaggle/input/notebooks/estopadilla/ntl-glm-prefix-ab/ntl-bin'),
    Path('/kaggle/input/ntl-glm-prefix-ab/ntl-bin'),
]
artifacts = next(
    (p for p in _bin_cands if (p / 'ntl-segrun').is_file() and (p / 'lib' / 'libcublasLt.so.13').is_file()),
    None,
)
if artifacts is None:
    _glob_hits = [
        Path(p).parent for p in sorted(glob.glob('/kaggle/input/**/ntl-glm-prefix-ab/**/ntl-bin/ntl-segrun', recursive=True))
        if Path(p).with_name('lib').joinpath('libcublasLt.so.13').is_file()
    ]
    assert _glob_hits, f'Missing ntl-glm-prefix-ab binary under /kaggle/input: checked {_bin_cands}'
    artifacts = _glob_hits[0]

for _fn in ('ntl-segrun', 'lib/libcublas.so.13', 'lib/libcublasLt.so.13', 'lib/libcudart.so.13'):
    assert (artifacts / _fn).is_file(), artifacts / _fn

V1_BIN = Path('/kaggle/temp/v1-bin')
V1_BIN.mkdir(exist_ok=True)
shutil.copyfile(artifacts / 'ntl-segrun', V1_BIN / 'ntl-segrun')
(V1_BIN / 'ntl-segrun').chmod(0o755)
V1_ENV = dict(os.environ, LD_LIBRARY_PATH=str(artifacts / 'lib') + ':' + os.environ.get('LD_LIBRARY_PATH', ''))

# Verify exact byte sizes and create symlink directories expected by `v1_stage.py` (`*.gguf` globs)
GLM_SEGMENTS = {
    'jamesliebel/glm53-iq3xxs-segs-a-mirror': [
        ['glm53_iq3xxs-seg00-of-19-L0-6.gguf', 12753355264], ['glm53_iq3xxs-seg01-of-19-L6-10.gguf', 15651714688],
        ['glm53_iq3xxs-seg02-of-19-L10-14.gguf', 14583346912], ['glm53_iq3xxs-seg03-of-19-L14-18.gguf', 14583346912],
        ['glm53_iq3xxs-seg04-of-19-L18-22.gguf', 14583346912], ['glm53_iq3xxs-seg05-of-19-L22-26.gguf', 14583346912],
        ['glm53_iq3xxs-seg06-of-19-L26-30.gguf', 14583346912], ['glm53_iq3xxs-seg07-of-19-L30-34.gguf', 14583346912],
        ['glm53_iq3xxs-seg08-of-19-L34-38.gguf', 14583346912], ['glm53_iq3xxs-seg09-of-19-L38-42.gguf', 14583346912],
    ],
    'jamesliebel/glm53-iq3xxs-segs-b-mirror': [
        ['glm53_iq3xxs-seg10-of-19-L42-46.gguf', 14583346912], ['glm53_iq3xxs-seg11-of-19-L46-50.gguf', 14583346912],
        ['glm53_iq3xxs-seg12-of-19-L50-54.gguf', 14583346912], ['glm53_iq3xxs-seg13-of-19-L54-58.gguf', 14583346912],
        ['glm53_iq3xxs-seg14-of-19-L58-62.gguf', 14583346912], ['glm53_iq3xxs-seg15-of-19-L62-66.gguf', 14583346912],
        ['glm53_iq3xxs-seg16-of-19-L66-70.gguf', 14583346912], ['glm53_iq3xxs-seg17-of-19-L70-74.gguf', 14583346912],
        ['glm53_iq3xxs-seg18-of-19-L74-78.gguf', 16345433920],
    ],
    'jamesliebel/glm53-flash-iq3xxs-segs-mirror': [
        ['flash_iq3xxs-seg00-of-06-L0-8.gguf', 14879704352], ['flash_iq3xxs-seg01-of-06-L8-16.gguf', 22864262976],
        ['flash_iq3xxs-seg02-of-06-L16-24.gguf', 21821175680], ['flash_iq3xxs-seg03-of-06-L24-32.gguf', 21821175680],
        ['flash_iq3xxs-seg04-of-06-L32-40.gguf', 21821175680], ['flash_iq3xxs-seg05-of-06-L40-45.gguf', 14407253632],
    ],
}

V1_FLASH = Path('/kaggle/temp/v1-segs-flash')
V1_GLM53 = Path('/kaggle/temp/v1-segs-glm53')
V1_FLASH.mkdir(exist_ok=True)
V1_GLM53.mkdir(exist_ok=True)

n_flash = n_glm53 = 0
for _ref, _files in GLM_SEGMENTS.items():
    _owner, _slug = _ref.split('/')
    _root = _pinned_dataset(_owner, _slug)
    for _filename, _size in _files:
        _src = _root / _filename
        assert _src.is_file() and _src.stat().st_size == _size, _src
        _dst_dir = V1_FLASH if _filename.startswith('flash_') else V1_GLM53
        _link = _dst_dir / _filename
        if not _link.exists():
            _link.symlink_to(_src)
        if _filename.startswith('flash_'):
            n_flash += 1
        else:
            n_glm53 += 1
assert n_flash == 6 and n_glm53 == 19, (n_flash, n_glm53)

# Commit/rerun pre-flight check: tokenize 3 rows with both tokenizers + Flash GPU decode smoke test on commit
if V1_ENABLED:
    sys.path.insert(0, str(V1_DIR))
    import segio
    _t = pd.read_csv(V1_TEST_CSV, dtype={'id': str}).head(3)
    segio.write_texts('/kaggle/temp/v1_smoke_texts.bin', _t.text.fillna('').astype(str).tolist())
    _seg0_f = sorted(V1_FLASH.glob('*.gguf'))[0]
    _seg0_g = sorted(V1_GLM53.glob('*.gguf'))[0]
    subprocess.run(
        [str(V1_BIN / 'ntl-segrun'), 'tokenize', '--model', str(_seg0_g), '--texts', '/kaggle/temp/v1_smoke_texts.bin',
         '--out', '/kaggle/temp/v1_smoke_tok_g.bin', '--max-text-tokens', '160',
         '--prefix', '[gMASK]<sop><|system|>Reasoning Effort: Low<|user|><|assistant|><think></think>'],
        env=V1_ENV, check=True,
    )
    _tok_cmd = (
        f'{V1_BIN}/ntl-segrun tokenize --model {_seg0_f} --texts /kaggle/temp/v1_smoke_texts.bin '
        f'--out /kaggle/temp/v1_smoke_tok.bin --max-text-tokens 160 '
        f'--prefix "[gMASK]<sop><|system|>Reasoning Effort: Low<|user|><|assistant|><think></think>"'
    )
    if not V1_RERUN:
        _tok_cmd += (
            f' && {V1_BIN}/ntl-segrun run --model {_seg0_f} --tokens /kaggle/temp/v1_smoke_tok.bin '
            f'--out /kaggle/temp/v1_smoke_state.bin --state-dtype f32 --seq-batch 8 --split-mid'
        )
    _r = subprocess.run(_tok_cmd, shell=True, capture_output=True, text=True, env=V1_ENV)
    (V1_OUT / 'smoke.log').write_text(_r.stdout + _r.stderr)
    print('\n'.join(l for l in (_r.stdout + _r.stderr).splitlines() if l.startswith('ntl-segrun')), flush=True)
    assert _r.returncode == 0 and (V1_RERUN or 'state max |x|' in _r.stderr), 'V1 smoke test failed, see /kaggle/working/v1/smoke.log'
    print('V1 checks OK: ntl-glm-prefix-ab binary + CUDA 13 libs, 6 + 19 segments,', 'tokenizer' if V1_RERUN else 'GPU smoke test', flush=True)


In [ ]:
# Execute Stage-2 PB-115 Token-Aligned Hybrid GLM-5.3 & GLM-5.3-Flash likelihood scoring
if V1_ENABLED and not FAST_COMMIT:
    _s10_s = time.time() - NB_T0
    _deadline = time.monotonic() + (NB_T0 + V1_DEADLINE_H * 3600 - time.time())
    _cmd = [
        sys.executable, str(V1_DIR / 'v1_stage.py'), '--test-csv', V1_TEST_CSV,
        '--final-logits', FINAL_LOGITS, '--post-logits', '/kaggle/temp/prefix-votes.npz',
        '--pb115-dir', PB115_DIR, '--train-csv', os.path.join(DATA, 'train.csv'), '--hybrid-alpha', '0.50', '--max-cover', '0.40',
        '--out', OUT, '--report', str(V1_OUT / 'report.json'),
        '--bin', str(V1_BIN), '--pipeline', str(V1_DIR / 'run_pipeline.py'),
        '--flash-segs', str(V1_FLASH / '*.gguf'), '--glm53-segs', str(V1_GLM53 / '*.gguf'),
        '--model', str(V1_DIR / 'll_model.json'), '--work', '/kaggle/temp/v1-work', '--outdir', str(V1_OUT),
        '--deadline-mono', repr(_deadline), '--seq-batch', str(V1_SEQ_BATCH), '--ubatch-div', str(V1_UBATCH_DIV),
    ]
    _cmd += ['--pack'] if V1_PACK else []
    _cmd += ['--priority-order', '--shrink'] if V1_ROBUST else []
    _cmd += ['--prefetch'] if V1_PREFETCH else []
    for _k, _v in V1_COST.items():
        _cmd += ['--' + _k.replace('_', '-'), repr(_v)]
    _code = subprocess.call(_cmd, env=V1_ENV)
    _rep = json.loads((V1_OUT / 'report.json').read_text()) if (V1_OUT / 'report.json').exists() else {}
    _rep['s10_s'] = _s10_s
    (V1_OUT / 'report.json').write_text(json.dumps(_rep, indent=1))
    print(json.dumps(_rep, indent=1), flush=True)
    assert _code == 0, f'V1 stage failed (exit code {_code}), see /kaggle/working/v1/'
    _final = pd.read_parquet(OUT)
    assert _final.columns.tolist() == ['id', 'model'], _final.columns.tolist()
    assert _final.id.tolist() == pd.read_csv(V1_TEST_CSV, dtype={'id': str}).id.tolist()
if V1_ENABLED and FAST_COMMIT:
    sys.path.insert(0, str(V1_DIR))
    import v1_stage as _v1s
    _mdl = json.loads((V1_DIR / 'll_model.json').read_text())
    _rng = np.random.default_rng(7)
    _pb_tmp = Path('/kaggle/temp/v1-pb115-smoke')
    _pb_tmp.mkdir(parents=True, exist_ok=True)
    _pf = pd.read_csv(os.path.join(PB115_DIR, 'flash_features.csv'), dtype={'id': str}).head(5)
    _pg = pd.read_csv(os.path.join(PB115_DIR, 'glm53_features.csv'), dtype={'id': str}).head(5)
    _sids = _pf['id'].tolist()
    _sc = np.column_stack([
        -np.abs(_rng.normal(size=100)),
        np.abs(_rng.normal(size=100)) + 1.0,
        np.abs(_rng.normal(size=100)) + 5.0,
        _rng.integers(0, 20, size=100).astype(np.float32),
    ]).astype(np.float32)
    _off = np.array([0, 20, 40, 60, 80, 100], dtype=np.int64)
    np.savez(_pb_tmp / 'flash_scores.npz', ids=np.asarray(_sids, dtype='U'), offsets=_off, scores=_sc)
    np.savez(_pb_tmp / 'glm53_scores.npz', ids=np.asarray(_sids, dtype='U'), offsets=_off, scores=_sc * 0.95)
    _ll_pb, _m3_pb = _v1s.fit_and_predict_pb115(
        PB115_DIR, os.path.join(DATA, 'train.csv'),
        {'flash': _pf.set_index('id'), 'glm53': _pg.set_index('id')},
        _sids, _pb_tmp / 'flash_scores.npz', _pb_tmp / 'glm53_scores.npz', _mdl['classes'],
    )
    _z0 = _v1s.log_softmax(_rng.normal(size=(16, 18)), 1)
    _d0, _ci = _v1s.compute_hybrid_delta(_z0, _ll_pb, _mdl['classes'], _mdl['W'], _mdl['entropy_ref'], hybrid_alpha=0.50, ll_m3_override=_m3_pb)
    assert _d0.shape == (5, 18) and np.isfinite(_d0).all()
    print('PB115 end-to-end smoke ok:', _ll_pb.shape, _ci, flush=True)
display(pd.read_parquet(OUT).head())

## 5. Stage-3 All-GLM-Out Guard, Sibling Odds, Mass-Preserving Residual & Typographic Invariants

Combines the **PB-115 Hybrid Apex** and **Sibling Odds** SOTA pipeline:
1. **All-GLM-Out Protection Guard (`fable_return`)**: Protects all PB-115 causal likelihood transitions away from GLM (`glm-5.3`, `glm-5.3-flash`) against downstream override.
2. **Within-Family Sibling Odds (`+0.55`)**: Uses `_post - base_lp` within-family log-odds shifts for close (`top[:, -1] - top[:, -2] < 0.8`) Claude, GPT, and Gemini decisions.
3. **Mass-Preserving Sibling Residual (`pngy_residual.py`)** & **Typographic Invariants** (`' — '`, `'’'`, `'—'`).

In [ ]:
# Stage-3 All-GLM-Out Guard, Mass-Preserving Sibling Residual & Typographic Invariants
import json
import os
import numpy as np
import pandas as pd


def fable_return(base, post, labels):
    """Protect all PB-115 likelihood transitions moving away from GLM against downstream override."""
    glm = [labels.index("glm-5.3"), labels.index("glm-5.3-flash")]
    return np.isin(base.argmax(1), glm) & (~np.isin(post.argmax(1), glm))


if not FAST_COMMIT and os.path.exists(OUT):
    try:
        print('Starting Apex Grandmaster V2 Post-Processing Stage...', flush=True)
        from pngy_residual import Residual
        query_df = pd.read_csv(V1_TEST_CSV, dtype={'id': str})
        train_df = pd.read_csv(os.path.join(DATA, 'train.csv'))
        labels = sorted(train_df.model.unique())

        if os.path.exists(FINAL_LOGITS):
            z_raw = np.load(FINAL_LOGITS)
            key = 'logits' if 'logits' in z_raw else 'logp' if 'logp' in z_raw else list(z_raw.keys())[0]
            base_lp = np.asarray(z_raw[key], dtype=np.float64)
            if 'labels' in z_raw:
                z_labels = list(z_raw['labels'])
                base_lp = base_lp[:, [z_labels.index(l) for l in labels]]
        else:
            base_lp = np.log(1.0 / 18.0 * np.ones((len(query_df), 18)))

        post_model = Residual(labels).fit(train_df.text.fillna('').astype(str), train_df.model)
        refined_lp = post_model.predict_log_probability(base_lp, query_df.text.fillna('').astype(str))
        refined_preds = np.asarray(labels)[refined_lp.argmax(1)]

        current_sub = pd.read_parquet(OUT)
        current_preds = current_sub.model.values.copy()

        _v1_glm_mask = np.isin(current_preds, ['glm-5.3', 'glm-5.3-flash'])
        _protect = np.zeros(len(query_df), dtype=bool)
        _report = json.loads((V1_OUT / 'report.json').read_text()) if (V1_OUT / 'report.json').is_file() else {}
        if _report.get('status') == 'applied':
            with np.load('/kaggle/temp/prefix-votes.npz', allow_pickle=False) as _v:
                assert _v['ids'].tolist() == query_df.id.astype(str).tolist() == current_sub.id.astype(str).tolist()
                assert set(_v['labels'].tolist()) == set(labels)
                _post = _v['logits'][:, [_v['labels'].tolist().index(c) for c in labels]]
            assert np.array_equal(np.asarray(labels)[_post.argmax(1)], current_preds)
            _protect = fable_return(base_lp, _post, labels)
            delta = _post - base_lp
            adjusted = refined_lp.copy()
            for prefix in ('claude-', 'gpt-', 'gemini-'):
                group = np.flatnonzero([l.startswith(prefix) for l in labels])
                within = delta[:, group]
                adjusted[:, group] += 0.55 * np.clip(within - within.mean(axis=1, keepdims=True), -3.0, 3.0)
            win = refined_lp.argmax(1)
            other = adjusted.argmax(1)
            family = np.asarray([l.split('-')[0] for l in labels])
            top = np.sort(refined_lp, axis=1)
            keep = (family[win] == family[other]) & ((top[:, -1] - top[:, -2]) < 0.8)
            refined_preds[keep] = np.asarray(labels)[other[keep]]
        _v1_glm_mask |= _protect
        print('Additional protected decisions', int(_protect.sum()), flush=True)

        _sibling_updated = 0
        for _i in range(len(query_df)):
            if not _v1_glm_mask[_i]:
                if current_preds[_i] != refined_preds[_i]:
                    current_preds[_i] = refined_preds[_i]
                    _sibling_updated += 1
        print(f'Applied mass-preserving sibling updates to {_sibling_updated} non-GLM predictions', flush=True)

        P_final = np.exp(refined_lp - refined_lp.max(1, keepdims=True))
        P_final /= P_final.sum(1, keepdims=True)
        if _report.get('status') == 'applied':
            _pp = np.exp(_post - _post.max(1, keepdims=True))
            _pp /= _pp.sum(1, keepdims=True)
            with np.load('/kaggle/temp/prefix-votes.npz', allow_pickle=False) as _v_p:
                _v1_sel_mask = _v_p['sel_mask'] if 'sel_mask' in _v_p else ~np.isclose(_post, base_lp, atol=1e-5).all(axis=1)
            if _v1_sel_mask.any():
                P_final[_v1_sel_mask] = _pp[_v1_sel_mask]
                print(f'Aligned typographic filter probabilities with V1 posterior on {int(_v1_sel_mask.sum())} texts', flush=True)

        _typo_fixed = 0
        _SPACED_ALLOWED = {'claude-opus-4-6', 'claude-opus-5', 'glm-5.3', 'glm-5.3-flash'}
        _SPACED_FORBIDDEN = np.array([l not in _SPACED_ALLOWED for l in labels])
        _NO_CURLY = np.array([l.startswith('claude-') or l.startswith('glm-') for l in labels])
        _NO_GEMINI = np.array([l.startswith('gemini-') for l in labels])

        for _i, _t in enumerate(query_df.text.fillna('').astype(str)):
            _has_spaced = ' — ' in _t
            _has_curly = any(c in _t for c in ('’', '‘', '“', '”'))
            _has_em = '—' in _t
            _violates = (
                (_has_spaced and current_preds[_i] not in _SPACED_ALLOWED)
                or (_has_curly and (current_preds[_i].startswith('claude-') or current_preds[_i].startswith('glm-')))
                or (_has_em and current_preds[_i].startswith('gemini-'))
            )
            if not _violates:
                continue
            _masked = P_final[_i].copy()
            if _has_spaced:
                _masked[_SPACED_FORBIDDEN] = 0.0
            if _has_curly:
                _masked[_NO_CURLY] = 0.0
            if _has_em:
                _masked[_NO_GEMINI] = 0.0
            if _masked.sum() > 0:
                P_final[_i] = _masked / _masked.sum()
                current_preds[_i] = labels[P_final[_i].argmax()]
                _typo_fixed += 1
            else:
                if _has_spaced and current_preds[_i] not in _SPACED_ALLOWED:
                    P_final[_i, _SPACED_FORBIDDEN] = 0.0
                elif _has_curly and (current_preds[_i].startswith('claude-') or current_preds[_i].startswith('glm-')):
                    P_final[_i, _NO_CURLY] = 0.0
                elif _has_em and current_preds[_i].startswith('gemini-'):
                    P_final[_i, _NO_GEMINI] = 0.0
                if P_final[_i].sum() > 0:
                    P_final[_i] /= P_final[_i].sum()
                    current_preds[_i] = labels[P_final[_i].argmax()]
                    _typo_fixed += 1

        current_sub['model'] = current_preds
        assert current_sub['model'].isin(labels).all() and len(current_sub) == len(query_df)
        current_sub.to_parquet(OUT, index=False)
        print(f'V2 Post-Processing Complete: corrected {_typo_fixed} invariant violations', flush=True)
        print(current_sub.model.value_counts().to_string(), flush=True)
    except Exception as _e_v2:
        raise
else:
    from pngy_residual import Residual
    _tr = pd.read_csv(os.path.join(DATA, 'train.csv'))
    _lbs = sorted(_tr.model.unique())
    _res = Residual(_lbs).fit(_tr.text.fillna('').astype(str), _tr.model)
    _q = pd.read_csv(V1_TEST_CSV, dtype={'id': str})
    _lp = _res.predict_log_probability(np.zeros((len(_q), len(_lbs))), _q.text.fillna('').astype(str))
    assert np.isfinite(_lp).all()
    _f_test = fable_return(np.zeros((len(_q), len(_lbs))), np.zeros((len(_q), len(_lbs))), _lbs)
    assert len(_f_test) == len(_q)
    print('Residual, All-GLM-Out Guard & Typographic Invariants commit smoke ok:', len(_q))

final = pd.read_parquet(OUT)
assert final.id.astype(str).tolist() == pd.read_csv(V1_TEST_CSV, dtype={'id': str}).id.tolist()
assert final.columns.tolist() == ['id', 'model'] and final.id.is_unique
display(final.head())

if FAST_COMMIT:
    print("=" * 76)
    print("0.728 | RESIDUAL SIBLING GATING + PB115 HYBRID — SMOKE VERIFICATION SUMMARY")
    print("=" * 76)
    print(f"  [PASS] Stage 1 | 5x L24 LoRA + DeBERTa + Native Q8 + GSQ-27B (share=0.035, timeout=5400s)")
    print(f"  [PASS] Stage 2 | PB-115 Hybrid GLM-5.3 (LR + 3-seed LGB, alpha=0.35, max_cover=0.40)")
    print(f"  [PASS] Stage 3 | Residual Sibling Gating ({len(_lbs)} models, {len(_tr)} train rows) + All-GLM-Out Guard")
    print(f"  [PASS] Schema  | {OUT} -> rows={len(final)}, unique_ids={final.id.is_unique}, cols={list(final.columns)}")
    print("=" * 76)

## 6. Credits, Acknowledgements & Licensing

This notebook integrates verified community breakthroughs into a clean, 100% transparent, open-access architecture without obfuscation or encrypted blobs. All code is independently verifiable and reproducible. Full credit belongs to the competitors whose ideas, models, and findings power each component:

### 0.720+ SOTA Breakthroughs & Sibling Disambiguation
- **Ranjeet (`ranjeet258`)**: Created [Name That LLM | Sibling Odds](https://www.kaggle.com/code/ranjeet258/name-that-llm-sibling-odds), pioneering relative likelihood odds adjustment for close intra-family sibling pairs to establish the **0.720 Public SOTA**.
- **Jiaobin Liu (`jiaobinliu`)**: Analyzed direct candidate margin constraints in sibling odds ([Sibling Odds Candidate Margin](https://www.kaggle.com/code/jiaobinliu/name-that-llm-sibling-odds-candidate-margin)).
- **Mathlete (`mathleteutd`)**: Extended Guarded Returns to all models ([Sibling Odds + Guarded GLM Returns](https://www.kaggle.com/code/mathleteutd/name-that-llm-sibling-odds-guarded-glm-returns)), ensuring high-confidence flips from GLM to any model are preserved, and pioneered post-likelihood fallback probabilities in typography.
- **Dennis (`dennis6886`)**: Formulated the Guarded Fable Return Gate ([GLM User Prefix - Guarded Fable Return](https://www.kaggle.com/code/dennis6886/glm-user-prefix-guarded-fable-return)), protecting critical causal likelihood flips from GLM to `claude-fable-5`.
- **`goodpjw2008`**: Calibrated the dual-T4 GLM runtime cost clock ([User Prefix + Calibrated GLM Clock](https://www.kaggle.com/code/goodpjw2008/name-that-llm-user-prefix-calibrated-glm-clock)).
- **Ross Wade (`rosswade`)**: Authored [The GLM stage was being undone](https://www.kaggle.com/code/rosswade/name-that-llm-the-glm-stage-was-being-undone).
- **Mirza Ishran (`mzishran`)**: Formulated `reconcile_typography()` ([When GLM and Fable Disagree](https://www.kaggle.com/code/mzishran/name-that-llm-when-glm-and-fable-disagree)) resolving contradictory em-dash and curly apostrophe collisions.

### 0.716 User-Prefix GLM Breakthrough & Apex Lineage
- **José Padilla (`estopadilla`)**: Authored [0.716 | Robust GLM-5.3 Likelihood + User Prefix](https://www.kaggle.com/code/estopadilla/0-716-robust-glm-5-3-likelihood-user-prefix?scriptVersionId=370363383), built `ntl-segrun` (`llama.cpp b12247` + CUDA 13 in [ntl-glm-prefix-ab](https://www.kaggle.com/code/estopadilla/ntl-glm-prefix-ab)), designed the shrink-on-overrun segment executor, and refitted `ll_model.json` (`ll_prefixB/L160`, `W=1.5`, `+0.014` OOF).
- **`prvsiyan`**: Contributed the insight to condition GLM likelihood evaluation on a length-matched synthetic `<|user|>` turn prompt (and shared the extended dataset).
- **James Liebel (`jamesliebel`)**: Created the public GLM-5.3 and GLM-5.3-Flash `IQ3_XXS` GGUF layer-segment mirrors ([A](https://www.kaggle.com/datasets/jamesliebel/glm53-iq3xxs-segs-a-mirror), [B](https://www.kaggle.com/datasets/jamesliebel/glm53-iq3xxs-segs-b-mirror), [Flash](https://www.kaggle.com/datasets/jamesliebel/glm53-flash-iq3xxs-segs-mirror)).
- **LumiBee (`lumibee`)**: Developed the Apex v44 mass-preserving sibling residual (`pngy_residual.py`) and typographic invariant rules.
- **Alex Chilton (`alexchilton`)**: Assembled the reproducible all-public stack reference and hosted Meta Llama 3.1 8B.
- **Vladislav Lavrentev (`lightningv08`)**: Engineered deterministic `pinned_assets.py`, complementary stack isolation, native Q8 porting, and GSQ-27B calibration.
- **Areeb Ali (`areebbb`)**, **`feiyu26`**, **Ömer Faruk Merey (`omerfarukmerey`)**, **`sanpier`**, **`lbjyang`**, and **`ozertuu`**: Built and open-sourced the 0.684 LoRA + Native Q8 + GSQ-27B stack, 0.696 V1 graft, 0.706–0.709 extended token windows, and 0.716–0.718 verified pipelines.

### Pretrained Readers, Adapters & Corpora
- **CHEN Han (`concyclics`)**: Multi-depth 5-fold Qwen3-Embedding-8B expert.
- **Dhruv Tibarewal (`dhruvtibarewal`)**, **Nuri Dudak (`nuridudak`)**, **Nishant Dahal (`nishantdahal`)**, **`vnton37nguyn`**, **Joseph Adamski (`josephadamski91`)**, **Kirill Tym (`kirilltym`)**, **Zbigniew Kirchner (`zbigniewkirchner`)**, **Lucas Boesen (`lucasboesen`)**, **`pjmathematician`**, **`crtproduction`**, **`seyitkaangunes`**, **`abdullah1afify`**, **`dophuchung`**, and **`bbucxi`**.
- **Base Foundation Weights:** Z.ai (GLM-5.3), Qwen Team (Qwen2.5, Qwen3, Qwen3-Embedding), Meta Platforms (Llama 3.1 Community License), `llama.cpp / ggml` (MIT).


In [ ]:
'''Confirm that Kaggle will score the exact test IDs and a valid label on each row.'''
reference_ids = pd.read_csv(Path(DATA) / 'test.csv', dtype={'id': str}).id.tolist()
scored = pd.read_parquet(OUT)
assert scored.columns.tolist() == ['id', 'model']
assert scored.id.is_unique and scored.id.astype(str).tolist() == reference_ids
assert scored.model.isin(pd.read_csv(Path(DATA) / 'train.csv').model.unique()).all()